In [ ]:
# بسم الله الرحمن الرحیم
# Last Version
# ---> (Solar system): 1404/06/01
# ---> ( Christian calendar) 2025-08-23

In [ ]:
# Table of Contents:
# 1- Installing Libraries & Google Drive Connection for Data Retrieval
# 2- Data Feature Extraction
# 3- Device Choose = CPU or GPU
# 4- Data Utils
# 5- Model Architecture
# 6- Inference
# 7- Basic Losses
# 8- Evaluation Function:
# 9-Information Losses (Inter-Intra)
# 10-Cross Encoder Information Loss
# 11-Other information losses (SpeakerIdentity, Transcription, SpeakerAttribute)
# 12- Disentaglement Classification
# 14- HyperParameter & Configuration
# 15-Solver
# 16-Run
# 17- Plotting from Google Drive *.pkls

In [21]:
# BaseLine for HDvoicer

**-------------------------------------------------------------------------------------**

**---> 1- Installing Libraries & Google Drive Connection for Data Retrieval <----**

**-------------------------------------------------------------------------------------**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install "librosa==0.9.1"  tensorboardX pysptk pesq pystoi transformers accelerate datasets evaluate jiwer soundfile sentence-transformers


**----------------------------------------------------------------**

**--------> 2-Data Feature Extraction <-------------------**

**----------------------------------------------------------------**

In [4]:
# https://github.com/jjery2243542/adaptive_voice_conversion
# preprocess/tacotron/hyperparams.py

# -*- coding: utf-8 -*-
#/usr/bin/python2
'''
By kyubyong park. kbpark.linguist@gmail.com.
https://www.github.com/kyubyong/tacotron
'''
class Hyperparams:
    '''Hyper parameters'''

    top_db = 15

    # signal processing
    sr = 22050 # Sample rate.
    # n_fft = 2048 # fft points (samples)
    n_fft = 1024 # fft points (samples)
    # frame_shift = 0.0125 # seconds
    # frame_length = 0.05 # seconds
    # hop_length = int(sr*frame_shift) # samples.
    hop_length = 256 # samples.
    # win_length = int(sr*frame_length) # samples.
    win_length = 1024 # samples.
    n_mels = 80 # Number of Mel banks to generate
    power = 1.2 # Exponent for amplifying the predicted magnitude
    n_iter = 100 # Number of inversion iterations
    preemphasis = .97 # or None
    max_db = 100
    ref_db = 20
###
hp=Hyperparams()

################################################################################


In [ ]:
############################
## MRH: Extracting the data ....both txt and mel
############################

import os
import glob
import re
import random
import pickle
import json
import shutil
import librosa
import numpy as np
from collections import defaultdict

# --- Configuration ---
# Adjust these paths to match your VCTK dataset and desired output location
OUTPUT_DIR = "/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/"
# AUDIO_DATA_ROOT_DIR = "/content/drive/MyDrive/Dataset/VCTK_092/VCTK-Corpus-0.92/wav48_silence_trimmed/"
AUDIO_DATA_ROOT_DIR = "/content/drive/MyDrive/Dataset/VCTK-Corpus-MRH-Experimental/wav48/"
TRANSCRIPT_DATA_ROOT_DIR = "/content/drive/MyDrive/Dataset/VCTK-Corpus-MRH-Experimental/txt/"
SPEAKER_INFO_PATH = "/content/drive/MyDrive/Dataset/VCTK_092/VCTK-Corpus-0.92/speaker-info.txt"

# Dataset splitting parameters
TEST_SPEAKERS = 20
TEST_PROPORTION_IN_TRAIN_SPEAKERS = 0.1 # Proportion of data from train speakers used for in_test set

SAMPLE_RATE = 22050
# Mel Spectrogram parameters (IMPORTANT: adjust to match your model's expected input)

N_FFT = 1024       # Window size for STFT
HOP_LENGTH = 256   # Hop length for STFT (e.g., 12.5ms at 48kHz, 256 for 16kHz)
N_MELS = 80        # Number of Mel bins

# For mean/std calculation (n_utts_attr from your old script)
# This is used to sample a subset of spectrograms for calculating global mean/std.
# Set a reasonably large number (e.g., 5000 or more)
N_UTTS_FOR_MEAN_STD = 5000

print(f"--- Dataset Preparation Configuration ---")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Audio Root Dir: {AUDIO_DATA_ROOT_DIR}")
print(f"Transcript Root Dir: {TRANSCRIPT_DATA_ROOT_DIR}")
print(f"Speaker Info Path: {SPEAKER_INFO_PATH}")
print(f"Test Speakers (by ID): {TEST_SPEAKERS}")
print(f"Test Proportion (within train speakers): {TEST_PROPORTION_IN_TRAIN_SPEAKERS}")
print(f"Mel Params: SR={SAMPLE_RATE if SAMPLE_RATE else 'Original'}, N_FFT={N_FFT}, HOP_LENGTH={HOP_LENGTH}, N_MELS={N_MELS}")
print(f"Utterances for Mean/Std Calc: {N_UTTS_FOR_MEAN_STD}")
print(f"-----------------------------------------\n")


# --- Helper Functions ---

def read_speaker_info(speaker_info_path):
    """Reads VCTK speaker-info.txt and returns a list of speaker IDs (e.g., ['p225', 'p226'])."""
    speaker_ids = []
    if not os.path.exists(speaker_info_path):
        raise FileNotFoundError(f"Speaker info file not found: {speaker_info_path}")
    with open(speaker_info_path, 'r') as f:
        for i, line in enumerate(f):
            if i == 0: # Skip header line
                continue
            parts = line.strip().split()
            if parts:
                speaker_id = parts[0]
                # Ensure 'p' prefix as VCTK speaker folders typically have it
                if not speaker_id.startswith('p'):
                    speaker_id = 'p' + speaker_id
                speaker_ids.append(speaker_id)
    print(f"Debug: Found {len(speaker_ids)} speaker IDs from {speaker_info_path}.")
    return speaker_ids

def get_mel_spectrogram(wav_file_path, sr=None, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS):
    """
    Computes Mel spectrogram from a WAV/FLAC file.
    Adjust n_fft, hop_length, n_mels to match your model's specific requirements.
    """
    try:
        y, sr_loaded = librosa.load(wav_file_path, sr=sr) # sr=None will load at original sample rate
        y, _ = librosa.effects.trim(y, top_db=20) # Trim silence

        # Compute Mel-spectrogram
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr_loaded, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
        # Convert to log scale (decibels)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

        # Transpose to (time_frames, n_mels) which is common for many models
        return mel_spectrogram_db.T
    except Exception as e:
        print(f"Error processing {wav_file_path}: {e}")
        return None

def collect_vctk_data(audio_root_dir, transcript_root_dir):
    """
    Collects all available VCTK audio (mic2.flac) and their corresponding transcriptions,
    creating a unified data structure with standardized keys.
    """
    all_utterance_data = {}

    # Use glob to find all mic2.flac files, as this was your source for Mel
    # audio_glob_pattern = os.path.join(audio_root_dir, 'p*', '*_mic2.flac')
    audio_glob_pattern = os.path.join(audio_root_dir, 'p*', '*.wav')
    print(f"Debug: Scanning for audio files using pattern: {audio_glob_pattern}")
    audio_files = glob.glob(audio_glob_pattern)
    print(f"Debug: Found {len(audio_files)} mic2.flac audio files.")

    for audio_path in sorted(audio_files):
        audio_filename = os.path.basename(audio_path) # e.g., p225_001_mic2.flac
        print('audio_filename= ',audio_filename)

        # Extract speaker_id and utterance_num to form the standardized key
        # match = re.match(r'(p\d+)_(\d+)_mic2\.flac', audio_filename)
        match = re.match(r'(p\d+)_(\d+)\.wav', audio_filename)
        if not match:
            print(f"Warning: Audio filename '{audio_filename}' did not match mic2.flac pattern. Skipping.")
            continue

        speaker_id = match.groups()[0] # e.g., 'p225'
        utterance_num = match.groups()[1] # e.g., '001'
        print('mrh speaker_id= ',speaker_id)

        # Standardized key for both Mel and Transcription dictionaries
        # This will be the pXXX_YYY.wav format
        standardized_key = f"{speaker_id}_{utterance_num}.wav"

        # Path to the corresponding transcription file
        transcript_path = os.path.join(transcript_root_dir, speaker_id, f"{speaker_id}_{utterance_num}.txt")

        if os.path.exists(transcript_path):
            try:
                with open(transcript_path, 'r', encoding='utf-8') as f:
                    transcription = f.readline().strip()

                # Store collected data
                all_utterance_data[standardized_key] = {
                    'audio_path': audio_path,
                    'speaker_id': speaker_id,
                    'transcription': transcription
                }
            except Exception as e:
                print(f"Error reading transcript {transcript_path}: {e}. Skipping.")
        else:
            print(f"Warning: No transcript found for {audio_filename} at {transcript_path}. Skipping.")

    print(f"Debug: Successfully linked {len(all_utterance_data)} audio-transcript pairs.")
    return all_utterance_data


# --- Main Processing ---
if __name__ == '__main__':
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Created output directory: {OUTPUT_DIR}")

    print('\n---- MRH1')
    # 1. Read speaker information and split into train/test
    all_speaker_ids = read_speaker_info(SPEAKER_INFO_PATH)
    random.shuffle(all_speaker_ids) # Randomize speakers for consistent split across runs

    print('\n---- MRH2')
    train_speaker_ids = all_speaker_ids[:-TEST_SPEAKERS]
    test_speaker_ids = all_speaker_ids[-TEST_SPEAKERS:]
    print(f"Train Speakers: {len(train_speaker_ids)}, Test Speakers: {len(test_speaker_ids)}")

    print('\n---- MRH3')
    # 2. Collect all raw utterance data (audio paths, speaker IDs, transcriptions)
    # This dictionary uses standardized_key (pXXX_YYY.wav)
    all_utterance_data = collect_vctk_data(AUDIO_DATA_ROOT_DIR, TRANSCRIPT_DATA_ROOT_DIR)


    print('\n---- MRH4')
    # Group data by speaker for splitting
    speaker_to_utt_data = defaultdict(list)
    for key, val in all_utterance_data.items():
        speaker_to_utt_data[val['speaker_id']].append((key, val['audio_path'], val['transcription']))

    print('\n---- MRH5')
    # 3. Split the data into train, in_test, out_test based on speaker IDs and proportion
    train_set_list = []      # (standardized_key, audio_path, transcription)
    in_test_set_list = []    # (standardized_key, audio_path, transcription)
    out_test_set_list = []   # (standardized_key, audio_path, transcription)

    for speaker_id in train_speaker_ids:
        utterances_for_speaker = speaker_to_utt_data.get(speaker_id, [])
        if not utterances_for_speaker:
            print(f"Warning: No utterances found for training speaker {speaker_id}.")
            continue
        random.shuffle(utterances_for_speaker) # Shuffle for random train/in_test split within speaker

        test_size_for_speaker = int(len(utterances_for_speaker) * TEST_PROPORTION_IN_TRAIN_SPEAKERS)

        train_set_list.extend(utterances_for_speaker[:-test_size_for_speaker])
        in_test_set_list.extend(utterances_for_speaker[-test_size_for_speaker:])

    for speaker_id in test_speaker_ids:
        utterances_for_speaker = speaker_to_utt_data.get(speaker_id, [])
        if not utterances_for_speaker:
            print(f"Warning: No utterances found for test speaker {speaker_id}.")
            continue
        out_test_set_list.extend(utterances_for_speaker)

    print(f"\nDataset Split Sizes:")
    print(f"  Train: {len(train_set_list)} utterances")
    print(f"  In-Test: {len(in_test_set_list)} utterances")
    print(f"  Out-Test: {len(out_test_set_list)} utterances")

    # 4. Process each split: generate Mel spectrograms and save transcriptions
    all_train_mels_for_mean_std = [] # For global mean/std calculation

    datasets_to_process = {
        'train': train_set_list,
        'in_test': in_test_set_list,
        'out_test': out_test_set_list
    }

    for dset_name, data_list in datasets_to_process.items():
        print(f"\n--- Processing {dset_name} set ({len(data_list)} utterances) ---")
        mel_data_dict = {}       # {standardized_key: mel_spectrogram}
        transcript_data_dict = {} # {standardized_key: transcription_text}

        for i, (key, audio_path, transcription) in enumerate(data_list):
            print('i= ',i)
            if i % 1000 == 0 or i == len(data_list) - 1:
                print(f'  Processing {i}/{len(data_list)} files for {dset_name}...')

            mel_spec = get_mel_spectrogram(audio_path, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
            if mel_spec is None:
                continue # Skip if Mel generation failed

            mel_data_dict[key] = mel_spec
            transcript_data_dict[key] = transcription

            # Collect Mels for mean/std calculation from the training set
            if dset_name == 'train' and len(all_train_mels_for_mean_std) < N_UTTS_FOR_MEAN_STD:
                all_train_mels_for_mean_std.append(mel_spec)

        # Save Mel spectrogram data
        mel_output_path = os.path.join(OUTPUT_DIR, f'{dset_name}.pkl')
        with open(mel_output_path, 'wb') as f:
            pickle.dump(mel_data_dict, f)
        print(f"  Saved {len(mel_data_dict)} Mel spectrograms to {mel_output_path}")

        # Save Transcription data
        transcript_output_path = os.path.join(OUTPUT_DIR, f'{dset_name}_transcripts.pkl')
        with open(transcript_output_path, 'wb') as f:
            pickle.dump(transcript_data_dict, f)
        print(f"  Saved {len(transcript_data_dict)} transcriptions to {transcript_output_path}")

    # 5. Calculate and save mean/std for Mel spectrograms (from training set)
    print(f"\n--- Calculating Mel Spectrogram Mean/Std ---")
    if all_train_mels_for_mean_std:
        concatenated_mels = np.concatenate(all_train_mels_for_mean_std, axis=0)
        mel_mean = np.mean(concatenated_mels, axis=0)
        mel_std = np.std(concatenated_mels, axis=0)
        mel_attr = {'mean': mel_mean, 'std': mel_std}

        attr_output_path = os.path.join(OUTPUT_DIR, 'attr.pkl')
        with open(attr_output_path, 'wb') as f:
            pickle.dump(mel_attr, f)
        print(f"  Saved Mel mean/std attributes to {attr_output_path}")
        print(f"  Mel Mean shape: {mel_mean.shape}, Std shape: {mel_std.shape}")
    else:
        print("  Warning: No training Mels collected for mean/std calculation.")

    # 6. Copy speaker-info.txt to output directory for later reference
    destination_speaker_info_path = os.path.join(OUTPUT_DIR, os.path.basename(SPEAKER_INFO_PATH))
    try:
        shutil.copy(SPEAKER_INFO_PATH, destination_speaker_info_path)
        print(f"\nCopied '{SPEAKER_INFO_PATH}' to '{destination_speaker_info_path}'")
    except FileNotFoundError:
        print(f"\nError: Source speaker info file not found for copy: '{SPEAKER_INFO_PATH}'")
    except Exception as e:
        print(f"\nError copying speaker info file: {e}")

    print("\n--- Unified VCTK Dataset Preparation Complete! ---")

**-------------------------------------------------**

**------> 3- Device Choose = CPU or GPU <------**

**-------------------------------------------------**

In [5]:
# adaptive_voice_conversion /utils.py
import torch
import numpy as np
from tensorboardX import SummaryWriter
import editdistance
import torch.nn as nn
import torch.nn.init as init

def cc(net):
    # print('100')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # print('200')
    return net.to(device)

class Logger(object):
    def __init__(self, logdir='./log'):
        self.writer = SummaryWriter(logdir)

    def scalar_summary(self, tag, value, step):
        self.writer.add_scalar(tag, value, step)

    def scalars_summary(self, tag, dictionary, step):
        self.writer.add_scalars(tag, dictionary, step)

    def text_summary(self, tag, value, step):
        self.writer.add_text(tag, value, step)

    def audio_summary(self, tag, value, step, sr):
        writer.add_audio(tag, value, step, sample_rate=sr)

def infinite_iter(iterable):
    it = iter(iterable)
    while True:
        try:
            ret = next(it)
            yield ret
        except StopIteration:
            it = iter(iterable)

**-------------------------------------**

**-----------> 4-Data Utils <-----------**

**-------------------------------------**

In [19]:
import pickle
import torch
import numpy as np
import random
from torch.utils.data import Dataset # Ensure Dataset is imported if it wasn't already

#adaptive_voice_conversion/data_utils.py
import torch
from torch.utils.data import Dataset
import os
import pickle
import json
import numpy as np
import torch
from torch.utils.data import DataLoader

class PickleDataset3(Dataset):
    def __init__(self, pickle_path, segment_size, transcripts_path, speaker_info_path, attr_path=None): # Added attr_path
        """
        Initializes the Dataset.
        Args:
            pickle_path (str): Path to the .pkl file containing spectrogram data ({utt_id: full_spectrogram}).
            segment_size (int): The size of the spectrogram segments.
            transcripts_path (str): Path to the .pkl file containing transcription dictionary.
            speaker_info_path (str): Path to the text file containing speaker metadata.
            attr_path (str, optional): Path to the .pkl file containing Mel mean and std. Defaults to None.
        """
        # Load main data
        with open(pickle_path, 'rb') as f:
            self.data = pickle.load(f) # self.data is {utt_id: full_spectrogram_numpy_array}

        self.utt_ids = list(self.data.keys())
        self.segment_size = segment_size

        # Load transcriptions
        with open(transcripts_path, 'rb') as f:
            self.transcriptions = pickle.load(f)

        # Load speaker metadata
        self.speaker_metadata = self._load_speaker_metadata(speaker_info_path)

        # --- NEW: Load Mel mean and std from attr.pkl ---
        self.mel_mean = None # Initialize to None
        self.mel_std = None  # Initialize to None
        if attr_path:
            try:
                # Inside the try block when loading attr.pkl
                with open(attr_path, 'rb') as f:
                    attr_data = pickle.load(f)
                    # Convert to torch.Tensor and ensure float32 for consistency with PyTorch operations
                    self.mel_mean = torch.tensor(attr_data['mean'], dtype=torch.float32) # CHANGED TO 'mean'
                    self.mel_std = torch.tensor(attr_data['std'], dtype=torch.float32)   # CHANGED TO 'std'

                    # Add a small epsilon to std dev to prevent division by zero,
                    # in case any feature dimension has zero variance in the training set
                    # (e.g., if a Mel bin is always zero).
                    self.mel_std = torch.where(self.mel_std == 0, torch.tensor(1e-8, dtype=torch.float32), self.mel_std)

                    print(f"DEBUG: Loaded Mel mean (first val): {self.mel_mean[0].item():.4f}, std (first val): {self.mel_std[0].item():.4f} from {attr_path}")
            except FileNotFoundError:
                print(f"ERROR: attr.pkl not found at {attr_path}. Mel spectrograms will NOT be normalized.")
            except Exception as e:
                print(f"ERROR: Could not load attr.pkl from {attr_path}: {e}. Mel spectrograms will NOT be normalized.")
        else:
            print("WARNING: 'attr_path' not provided to PickleDataset2. Mel spectrograms will NOT be normalized.")

        # --- Optional: Filter out utterances that are too short ---
        # Uncomment the following block if your model strictly requires segments of exact `self.segment_size`
        # and cannot handle padded segments or prefers not to use them for training.
        # This will reduce the number of samples in your dataset.
        # Given your current padding logic, this filtering is not strictly necessary for preventing errors,
        # but it can be useful if padded samples are considered "less clean" for training.

        # original_len = len(self.utt_ids)
        # self.utt_ids = [uid for uid in self.utt_ids if self.data[uid].shape[0] >= self.segment_size]
        # if len(self.utt_ids) < original_len:
        #     print(f"INFO: Filtered out {original_len - len(self.utt_ids)} utterances shorter than segment_size ({self.segment_size}).")
        #     print(f"INFO: Remaining {len(self.utt_ids)} utterances for training/evaluation.")
        # else:
        #     print(f"INFO: All {len(self.utt_ids)} utterances are long enough for segment_size ({self.segment_size}).")


    def _load_speaker_metadata(self, speaker_info_path):
        """
        Parses the speaker-info.txt file into a dictionary.
        Expected format: ID AGE GENDER ACCENTS REGION
        """
        metadata = {}
        print(f"DEBUG: Attempting to load speaker metadata from: {speaker_info_path}")
        try:
            with open(speaker_info_path, 'r') as f:
                # Skip header line
                header = f.readline().strip().split()
                print(f"DEBUG: Header line: {header}")

                lines_processed = 0
                for line in f:
                    lines_processed += 1
                    parts = line.strip().split(maxsplit=5) # Increased maxsplit to correctly handle COMMENTS column if present
                    if not parts:
                        continue # Skip empty lines

                    speaker_id_raw = parts[0] # This could be '225' or 'p225'

                    # --- CRITICAL FIX START ---
                    # Ensure the key is consistently 'pXXX' format
                    if speaker_id_raw.startswith('p'):
                        speaker_id_key = speaker_id_raw # Already in 'pXXX' format, use as is
                    else:
                        speaker_id_key = f'p{speaker_id_raw}' # Prepend 'p'
                    # --- CRITICAL FIX END ---

                    # Ensure parts has enough elements before accessing
                    # Adjust index based on speaker-info.txt format: ID AGE GENDER ACCENTS REGION (COMMENTS)
                    if len(parts) < 3: # Need at least ID, AGE, GENDER
                        print(f"WARNING: Malformed line (too few parts) in speaker-info.txt: '{line.strip()}'")
                        continue

                    # Assuming header: ID AGE GENDER ACCENTS REGION COMMENTS
                    # parts[0]=ID, parts[1]=AGE, parts[2]=GENDER, parts[3]=ACCENTS, parts[4]=REGION, parts[5]=COMMENTS
                    age = int(parts[1]) if parts[1].isdigit() else None
                    gender = parts[2]
                    accent = parts[3]
                    region = parts[4] if len(parts) > 4 else None

                    metadata[speaker_id_key] = {
                        'age': age,
                        'gender': gender,
                        'accent': accent,
                        'region': region
                    }
                print(f"DEBUG: Finished processing {lines_processed} lines from speaker-info.txt.")
                print(f"DEBUG: Loaded {len(metadata)} speaker entries into metadata dictionary.")
                if metadata:
                    first_key = next(iter(metadata))
                    print(f"DEBUG: Example speaker metadata entry: '{first_key}': {metadata[first_key]}")
                else:
                    print("DEBUG: Speaker metadata dictionary is empty after loading.")

        except FileNotFoundError:
            print(f"ERROR: speaker-info.txt not found at {speaker_info_path}")
        except Exception as e:
            print(f"ERROR: An error occurred while parsing speaker-info.txt: {e}")
        return metadata


    def __getitem__(self, ind):
        # 1. Select a unique utterance ID based on the dataset index
        utt_id = self.utt_ids[ind]
        full_spectrogram = self.data[utt_id] # This is a NumPy array (frames, mel_bins)

        # 2. Randomly sample a segment from this full spectrogram
        if full_spectrogram.shape[0] < self.segment_size:
            # Handle utterances shorter than segment_size: pad with zeros
            segment = np.pad(full_spectrogram, ((0, self.segment_size - full_spectrogram.shape[0]), (0,0)), mode='constant')
            t = 0 # Starting frame index (for consistency, though not truly random)
        else:
            t_max = full_spectrogram.shape[0] - self.segment_size
            t = random.randint(0, t_max) # Randomly choose starting frame
            segment = full_spectrogram[t:t + self.segment_size]

        # --- NEW: Convert to Tensor and Apply Normalization ---
        # Convert segment (NumPy array) to PyTorch tensor of float32
        segment = torch.tensor(segment, dtype=torch.float32)

        # # --- DEBUG PRINTS HERE ---
        # print(f"\n--- DEBUG: Inside PickleDataset3 __getitem__ for {utt_id} ---")
        # print(f"Segment before norm (min/max): {segment.min().item():.4f} / {segment.max().item():.4f}")
        # if self.mel_mean is None:
        #     print("mel_mean is None. Normalization will not apply.")
        # else:
        #     print(f"mel_mean shape: {self.mel_mean.shape}, first val: {self.mel_mean[0].item():.4f}")
        #     print(f"mel_std shape: {self.mel_std.shape}, first val: {self.mel_std[0].item():.4f}")
        # print(f"Condition for normalization: mel_mean is not None: {self.mel_mean is not None}")
        # -----------------------------

        if self.mel_mean is not None and self.mel_std is not None:
            # Apply Z-score normalization: (value - mean) / std
            segment = (segment - self.mel_mean) / self.mel_std
            # --- DEBUG PRINT AFTER NORM ---
            # print(f"Segment AFTER norm (mean/std): {segment.mean().item():.4f} / {segment.std().item():.4f}")
            # ----------------------------------

        # Get transcription
        transcription = self.transcriptions.get(utt_id, "")

        # Get speaker metadata
        speaker_id_from_utt = utt_id.split('_')[0]
        speaker_info = self.speaker_metadata.get(speaker_id_from_utt, {})
        gender = speaker_info.get('gender', 'unknown')
        age = speaker_info.get('age', -1)
        accent = speaker_info.get('accent', 'unknown')

        # Return all components
        return segment, utt_id, transcription, gender, age, accent, t

    def __len__(self):
        return len(self.utt_ids)

In [6]:
################################################################################

# --- CollateFn needs to be updated too ---
class CollateFn(object):
    def __init__(self, frame_size):
        self.frame_size = frame_size

    def make_frames(self, tensor):
        # Your existing make_frames logic
        out = tensor.view(tensor.size(0), tensor.size(1) // self.frame_size, self.frame_size * tensor.size(2))
        out = out.transpose(1, 2)
        return out

    def __call__(self, l):
        # l will now be a list of (segment, utt_id, transcription, gender, age, accent) tuples

        numerical_segments = [item[0] for item in l]
        utt_ids = [item[1] for item in l]
        transcriptions = [item[2] for item in l]
        gender_list = [item[3] for item in l] # NEW
        age_list = [item[4] for item in l]    # NEW
        accent_list = [item[5] for item in l] # NEW
        t_list = [item[6] for item in l] # Add t_list

        data_tensor = torch.from_numpy(np.array(numerical_segments))
        segment = self.make_frames(data_tensor)

        # Return all components
        return segment, utt_ids, transcriptions, gender_list, age_list, accent_list, t_list # Add t_list

#########
# MRH: also returning utt_id
import torch # Assuming data is or will be converted to torch.Tensor

class SequenceDataset(Dataset):
    def __init__(self, data):
        self.data = data # This 'data' is likely a dict where keys are utt_ids and values are full spectrograms
        self.utt_ids = list(self.data.keys())

    def __getitem__(self, ind):
        utt_id = self.utt_ids[ind]
        ret = self.data[utt_id].transpose() # Assuming this prepares the data for your model

        # Add the utt_id to the return
        return ret, utt_id # <<< MRH: Add utt_id to return

    def __len__(self):
        return len(self.utt_ids)

#########
def get_data_loader(dataset, batch_size, frame_size, shuffle=True, num_workers=4, drop_last=False):
    _collate_fn = CollateFn(frame_size=frame_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
            num_workers=num_workers, collate_fn=_collate_fn, pin_memory=True)
    return dataloader

################################################################################


**--------------------------------------------------------**

**-----------> 5- Model Architecture <-------------------**

**--------------------------------------------------------**

In [7]:
##### Model
# adaptive_voice_conversion/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as ag
import numpy as np
from math import ceil
from functools import reduce
from torch.nn.utils import spectral_norm
# from utils import cc

class DummyEncoder(object):
    def __init__(self, encoder):
        self.encoder = encoder

    def load(self, target_network):
        self.encoder.load_state_dict(target_network.state_dict())

    def __call__(self, x):
        return self.encoder(x)

def pad_layer(inp, layer, pad_type='reflect'):
    kernel_size = layer.kernel_size[0]
    if kernel_size % 2 == 0:
        pad = (kernel_size//2, kernel_size//2 - 1)
    else:
        pad = (kernel_size//2, kernel_size//2)
    # padding
    inp = F.pad(inp,
            pad=pad,
            mode=pad_type)
    out = layer(inp)
    return out

def pad_layer_2d(inp, layer, pad_type='reflect'):
    kernel_size = layer.kernel_size
    if kernel_size[0] % 2 == 0:
        pad_lr = [kernel_size[0]//2, kernel_size[0]//2 - 1]
    else:
        pad_lr = [kernel_size[0]//2, kernel_size[0]//2]
    if kernel_size[1] % 2 == 0:
        pad_ud = [kernel_size[1]//2, kernel_size[1]//2 - 1]
    else:
        pad_ud = [kernel_size[1]//2, kernel_size[1]//2]
    pad = tuple(pad_lr + pad_ud)
    # padding
    inp = F.pad(inp,
            pad=pad,
            mode=pad_type)
    out = layer(inp)
    return out

def pixel_shuffle_1d(inp, scale_factor=2):
    batch_size, channels, in_width = inp.size()
    channels //= scale_factor
    out_width = in_width * scale_factor
    inp_view = inp.contiguous().view(batch_size, channels, scale_factor, in_width)
    shuffle_out = inp_view.permute(0, 1, 3, 2).contiguous()
    shuffle_out = shuffle_out.view(batch_size, channels, out_width)
    return shuffle_out

def upsample(x, scale_factor=2):
    x_up = F.interpolate(x, scale_factor=scale_factor, mode='nearest')
    return x_up

def flatten(x):
    out = x.contiguous().view(x.size(0), x.size(1) * x.size(2))
    return out

def concat_cond(x, cond):
    # x = [batch_size, x_channels, length]
    # cond = [batch_size, c_channels]
    cond = cond.unsqueeze(dim=2)
    cond = cond.expand(*cond.size()[:-1], x.size(-1))
    out = torch.cat([x, cond], dim=1)
    return out

def append_cond(x, cond):
    # x = [batch_size, x_channels, length]
    # cond = [batch_size, x_channels * 2]
    p = cond.size(1) // 2
    mean, std = cond[:, :p], cond[:, p:]
    out = x * std.unsqueeze(dim=2) + mean.unsqueeze(dim=2)
    return out

def conv_bank(x, module_list, act, pad_type='reflect'):
    outs = []
    for layer in module_list:
        out = act(pad_layer(x, layer, pad_type))
        outs.append(out)
    out = torch.cat(outs + [x], dim=1)
    return out

def get_act(act):
    if act == 'relu':
        return nn.ReLU()
    elif act == 'lrelu':
        return nn.LeakyReLU()
    else:
        return nn.ReLU()

class MLP(nn.Module):
    def __init__(self, c_in, c_h, n_blocks, act, sn):
        super(MLP, self).__init__()
        self.act = get_act(act)
        self.n_blocks = n_blocks
        f = spectral_norm if sn else lambda x: x
        self.in_dense_layer = f(nn.Linear(c_in, c_h))
        self.first_dense_layers = nn.ModuleList([f(nn.Linear(c_h, c_h)) for _ in range(n_blocks)])
        self.second_dense_layers = nn.ModuleList([f(nn.Linear(c_h, c_h)) for _ in range(n_blocks)])

    def forward(self, x):
        h = self.in_dense_layer(x)
        for l in range(self.n_blocks):
            y = self.first_dense_layers[l](h)
            y = self.act(y)
            y = self.second_dense_layers[l](y)
            y = self.act(y)
            h = h + y
        return h

class Prenet(nn.Module):
    def __init__(self, c_in, c_h, c_out,
            kernel_size, n_conv_blocks,
            subsample, act, dropout_rate):
        super(Prenet, self).__init__()
        self.act = get_act(act)
        self.subsample = subsample
        self.n_conv_blocks = n_conv_blocks
        self.in_conv_layer = nn.Conv2d(1, c_h, kernel_size=kernel_size)
        self.first_conv_layers = nn.ModuleList([nn.Conv2d(c_h, c_h, kernel_size=kernel_size) for _ \
                in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList([nn.Conv2d(c_h, c_h, kernel_size=kernel_size, stride=sub)
            for sub, _ in zip(subsample, range(n_conv_blocks))])
        output_size = c_in
        for l, sub in zip(range(n_conv_blocks), self.subsample):
            output_size = ceil(output_size / sub)
        self.out_conv_layer = nn.Conv1d(c_h * output_size, c_out, kernel_size=1)
        self.dropout_layer = nn.Dropout(p=dropout_rate)
        self.norm_layer = nn.InstanceNorm2d(c_h, affine=False)

    def forward(self, x):
        # reshape x to 4D
        x = x.contiguous().view(x.size(0), 1, x.size(1), x.size(2))
        out = pad_layer_2d(x, self.in_conv_layer)
        out = self.act(out)
        out = self.norm_layer(out)
        for l in range(self.n_conv_blocks):
            y = pad_layer_2d(out, self.first_conv_layers[l])
            y = self.act(y)
            y = self.norm_layer(y)
            y = self.dropout_layer(y)
            y = pad_layer_2d(y, self.second_conv_layers[l])
            y = self.act(y)
            y = self.norm_layer(y)
            y = self.dropout_layer(y)
            if self.subsample[l] > 1:
                out = F.avg_pool2d(out, kernel_size=self.subsample[l], ceil_mode=True)
            out = y + out
        out = out.contiguous().view(out.size(0), out.size(1) * out.size(2), out.size(3))
        out = pad_layer(out, self.out_conv_layer)
        out = self.act(out)
        return out

class Postnet(nn.Module):
    def __init__(self, c_in, c_h, c_out, c_cond,
            kernel_size, n_conv_blocks,
            upsample, act, sn):
        super(Postnet, self).__init__()
        self.act = get_act(act)
        self.upsample = upsample
        self.c_h = c_h
        self.n_conv_blocks = n_conv_blocks
        f = spectral_norm if sn else lambda x: x
        total_upsample = reduce(lambda x, y: x*y, upsample)
        self.in_conv_layer = f(nn.Conv1d(c_in, c_h * c_out // total_upsample, kernel_size=1))
        self.first_conv_layers = nn.ModuleList([f(nn.Conv2d(c_h, c_h, kernel_size=kernel_size)) for _ \
                in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList([f(nn.Conv2d(c_h, c_h*up*up, kernel_size=kernel_size))
            for up, _ in zip(upsample, range(n_conv_blocks))])
        self.out_conv_layer = f(nn.Conv2d(c_h, 1, kernel_size=1))
        self.conv_affine_layers = nn.ModuleList(
                [f(nn.Linear(c_cond, c_h * 2)) for _ in range(n_conv_blocks*2)])
        self.norm_layer = nn.InstanceNorm2d(c_h, affine=False)
        self.ps = nn.PixelShuffle(max(upsample))

    def forward(self, x, cond):
        out = pad_layer(x, self.in_conv_layer)
        out = out.contiguous().view(out.size(0), self.c_h, out.size(1) // self.c_h, out.size(2))
        for l in range(self.n_conv_blocks):
            y = pad_layer_2d(out, self.first_conv_layers[l])
            y = self.act(y)
            y = self.norm_layer(y)
            y = append_cond_2d(y, self.conv_affine_layers[l*2](cond))
            y = pad_layer_2d(y, self.second_conv_layers[l])
            y = self.act(y)
            if self.upsample[l] > 1:
                y = self.ps(y)
                y = self.norm_layer(y)
                y = append_cond_2d(y, self.conv_affine_layers[l*2+1](cond))
                out = y + upsample(out, scale_factor=(self.upsample[l], self.upsample[l]))
            else:
                y = self.norm_layer(y)
                y = append_cond(y, self.conv_affine_layers[l*2+1](cond))
                out = y + out
        out = self.out_conv_layer(out)
        out = out.squeeze(dim=1)
        return out

################################################################################

class SpeakerEncoder1(nn.Module):
    def __init__(self, c_in, c_h, c_out, kernel_size,
            bank_size, bank_scale, c_bank,
            n_conv_blocks, n_dense_blocks,
            subsample, act, dropout_rate,
            use_hierarchy=False, n_levels=1):
        super(SpeakerEncoder1, self).__init__()
        self.c_in = c_in
        self.c_h = c_h
        self.c_out = c_out # Dimension for each individual latent level (z1, z2, etc.)
        self.kernel_size = kernel_size
        self.n_conv_blocks = n_conv_blocks
        self.n_dense_blocks = n_dense_blocks
        self.subsample = subsample
        self.act = get_act(act)
        self.use_hierarchy = use_hierarchy
        self.n_levels = n_levels

        self.conv_bank = nn.ModuleList(
                [nn.Conv1d(c_in, c_bank, kernel_size=k) for k in range(bank_scale, bank_size + 1, bank_scale)])
        in_channels = c_bank * (bank_size // bank_scale) + c_in
        self.in_conv_layer = nn.Conv1d(in_channels, c_h, kernel_size=1)

        # Convolutional blocks
        self.first_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size) for _ in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size, stride=sub) for sub in subsample])

        self.pooling_layer = nn.AdaptiveAvgPool1d(1)

        # Dense blocks
        self.first_dense_layers = nn.ModuleList([nn.Linear(c_h, c_h) for _ in range(n_dense_blocks)])
        self.second_dense_layers = nn.ModuleList([nn.Linear(c_h, c_h) for _ in range(n_dense_blocks)])

        if self.use_hierarchy:
            # For each level (z1, z2, ..., zn), we need posterior mu and log_sigma layers
            # All levels have c_out dimensions
            self.mu_layers = nn.ModuleList([nn.Linear(c_h, c_out) for _ in range(n_levels)])
            self.log_sigma_layers = nn.ModuleList([nn.Linear(c_h, c_out) for _ in range(n_levels)])

            # Prior predictor layers: `prior_predictor_layers[k]` maps from z_{k+1} to prior params for z_k
            # e.g., prior_predictor_layers[0] for z1_prior from z2.
            #       prior_predictor_layers[1] for z2_prior from z3.
            self.prior_predictor_layers = nn.ModuleList()
            for _ in range(n_levels - 1): # (n_levels - 1) prior networks needed
                self.prior_predictor_layers.append(nn.Sequential(
                    nn.Linear(c_out, c_h),
                    self.act,
                    nn.Linear(c_h, c_out * 2) # For mu_prior and log_sigma_prior
                ))
        else:
            self.output_layer = nn.Linear(c_h, c_out)

        self.dropout_layer = nn.Dropout(p=dropout_rate)

    def conv_blocks(self, inp):
        out = inp
        for l in range(self.n_conv_blocks):
            y = pad_layer(out, self.first_conv_layers[l])
            y = self.act(y)
            y = self.dropout_layer(y)
            y = pad_layer(y, self.second_conv_layers[l])
            y = self.act(y)
            y = self.dropout_layer(y)
            if self.subsample[l] > 1:
                out = F.avg_pool1d(out, kernel_size=self.subsample[l], ceil_mode=True)
            out = y + out # Residual connection
        return out

    def dense_blocks(self, inp):
        out = inp
        for l in range(self.n_dense_blocks):
            y = self.first_dense_layers[l](out)
            y = self.act(y)
            y = self.dropout_layer(y)
            y = self.second_dense_layers[l](y)
            y = self.act(y)
            y = self.dropout_layer(y)
            out = y + out # Residual connection
        return out

    def forward(self, x):
        out = conv_bank(x, self.conv_bank, act=self.act)
        out = pad_layer(out, self.in_conv_layer)
        out = self.act(out)
        out = self.conv_blocks(out)
        out = self.pooling_layer(out).squeeze(2) # Shape: (batch_size, c_h)
        pooled_features = self.dense_blocks(out) # pooled_features is (batch_size, c_h)

        if self.use_hierarchy:
            mus = []
            log_sigmas = []
            mu_priors = [] # [None (for z1), prior for z2 (from z1), prior for z3 (from z2)]
            log_sigma_priors = [] # Same
            sampled_latents = []

            # --- For z1 (Highest Level - standard normal prior) ---
            # Posterior for z1 from pooled_features
            mu_post_z1 = self.mu_layers[0](pooled_features)
            log_sigma_post_z1 = self.log_sigma_layers[0](pooled_features)

            # Sample z1
            eps_z1 = torch.randn_like(mu_post_z1)
            z1 = mu_post_z1 + torch.exp(log_sigma_post_z1 * 0.5) * eps_z1

            mus.append(mu_post_z1)
            log_sigmas.append(log_sigma_post_z1)
            sampled_latents.append(z1)

            # Prior for z1 is standard normal (N(0,I))
            mu_priors.append(None)
            log_sigma_priors.append(None)

            # --- For subsequent levels (z2, z3, etc.) ---
            # Loop from the second level (index 1) up to n_levels-1
            for i in range(1, self.n_levels):
                # Posterior for z_{i+1} from pooled_features
                mu_post_zi = self.mu_layers[i](pooled_features)
                log_sigma_post_zi = self.log_sigma_layers[i](pooled_features)

                # Sample z_{i+1}
                eps_zi = torch.randn_like(mu_post_zi)
                zi = mu_post_zi + torch.exp(log_sigma_post_zi * 0.5) * eps_zi

                mus.append(mu_post_zi)
                log_sigmas.append(log_sigma_post_zi)
                sampled_latents.append(zi) # sampled_latents[i] is z_{i+1}

                # Compute prior for z_{i+1} from z_i (sampled_latents[i-1])
                prior_params = self.prior_predictor_layers[i-1](sampled_latents[i-1])
                mu_prior_zi, log_sigma_prior_zi = prior_params.chunk(2, dim=-1)

                mu_priors.append(mu_prior_zi)
                log_sigma_priors.append(log_sigma_prior_zi)

            # The decoder input for speaker is the concatenation of all sampled speaker latents
            decoder_input_emb = torch.cat(sampled_latents, dim=-1)

            return {
                'mus': mus, # List of posterior means [mu_z1, mu_z2, mu_z3]
                'log_sigmas': log_sigmas, # List of posterior log_sigmas [log_sigma_z1, log_sigma_z2, log_sigma_z3]
                'mu_priors': mu_priors, # List of priors [None (for z1), prior_z2_from_z1, prior_z3_from_z2]
                'log_sigma_priors': log_sigma_priors, # Same
                'decoder_input': decoder_input_emb # Concatenated speaker embedding (z1 || z2 || z3 ...)
            }

        else:
            # Original non-hierarchical output remains the same
            emb = self.output_layer(pooled_features)
            return {
                'mus': [emb],
                'log_sigmas': [torch.zeros_like(emb)],
                'mu_priors': [None],
                'log_sigma_priors': [None],
                'decoder_input': emb
            }


################################################################################

class ContentEncoder1(nn.Module):
    def __init__(self, c_in, c_h, c_out, kernel_size,
            bank_size, bank_scale, c_bank,
            n_conv_blocks, subsample,
            act, dropout_rate,
            use_hierarchy=False, n_levels=1):
        super(ContentEncoder1, self).__init__()
        self.n_conv_blocks = n_conv_blocks
        self.subsample = subsample
        self.act = get_act(act)
        self.use_hierarchy = use_hierarchy
        self.n_levels = n_levels
        self.c_out = c_out # Dimension for each individual latent level (z1, z2, etc.)

        self.conv_bank = nn.ModuleList(
                [nn.Conv1d(c_in, c_bank, kernel_size=k) for k in range(bank_scale, bank_size + 1, bank_scale)])
        in_channels = c_bank * (bank_size // bank_scale) + c_in
        self.in_conv_layer = nn.Conv1d(in_channels, c_h, kernel_size=1)
        self.first_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size) for _ in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size, stride=sub) for sub in subsample])
        self.norm_layer = nn.InstanceNorm1d(c_h, affine=False)
        self.dropout_layer = nn.Dropout(p=dropout_rate)

        if self.use_hierarchy:
            # For each level (z1, z2, ..., zn), we need posterior mu and log_sigma layers
            # All levels have c_out dimensions
            self.mu_layers = nn.ModuleList([nn.Conv1d(c_h, c_out, kernel_size=1) for _ in range(n_levels)])
            self.log_sigma_layers = nn.ModuleList([nn.Conv1d(c_h, c_out, kernel_size=1) for _ in range(n_levels)])

            # Prior predictor layers: `prior_predictor_layers[k]` maps from z_{k+1} to prior params for z_k
            self.prior_predictor_layers = nn.ModuleList()
            for _ in range(n_levels - 1):
                self.prior_predictor_layers.append(nn.Sequential(
                    nn.Conv1d(c_out, c_h, kernel_size=1),
                    self.act,
                    nn.Conv1d(c_h, c_out * 2, kernel_size=1)
                ))
        else:
            self.mean_layer = nn.Conv1d(c_h, c_out, kernel_size=1)
            self.std_layer = nn.Conv1d(c_h, c_out, kernel_size=1)

    def forward(self, x):
        out = conv_bank(x, self.conv_bank, act=self.act)
        out = pad_layer(out, self.in_conv_layer) # This is correct for in_conv_layer
        out = self.norm_layer(out)
        out = self.act(out)
        out = self.dropout_layer(out)
        # convolution blocks
        for l in range(self.n_conv_blocks):
            y = pad_layer(out, self.first_conv_layers[l]) # Correct
            y = self.norm_layer(y)
            y = self.act(y)
            y = self.dropout_layer(y)
            y = pad_layer(y, self.second_conv_layers[l]) # Correct
            y = self.norm_layer(y)
            y = self.act(y)
            y = self.dropout_layer(y)
            if self.subsample[l] > 1:
                out = F.avg_pool1d(out, kernel_size=self.subsample[l], ceil_mode=True)
            out = y + out # Residual connection

        processed_features = out # (batch_size, c_h, seq_len)

        if self.use_hierarchy:
            mus = []
            log_sigmas = []
            mu_priors = [] # [None (for z1), prior for z2 (from z1), prior for z3 (from z2)]
            log_sigma_priors = [] # Same
            sampled_latents = []

            # --- For z1 (Highest Level - standard normal prior) ---
            # Posterior for z1 from processed_features
            # These are 1x1 convs, no need for external pad_layer.
            mu_post_z1 = self.mu_layers[0](processed_features)
            log_sigma_post_z1 = self.log_sigma_layers[0](processed_features)

            # Sample z1
            eps_z1 = torch.randn_like(mu_post_z1)
            z1 = mu_post_z1 + torch.exp(log_sigma_post_z1 * 0.5) * eps_z1

            mus.append(mu_post_z1)
            log_sigmas.append(log_sigma_post_z1)
            sampled_latents.append(z1)

            # Prior for z1 is standard normal (N(0,I))
            mu_priors.append(None)
            log_sigma_priors.append(None)

            # --- For subsequent levels (z2, z3, etc.) ---
            # Loop from the second level (index 1) up to n_levels-1
            for i in range(1, self.n_levels):
                # Posterior for z_{i+1} from processed_features
                # These are 1x1 convs, no need for external pad_layer.
                mu_post_zi = self.mu_layers[i](processed_features)
                log_sigma_post_zi = self.log_sigma_layers[i](processed_features)

                # Sample z_{i+1}
                eps_zi = torch.randn_like(mu_post_zi)
                zi = mu_post_zi + torch.exp(log_sigma_post_zi * 0.5) * eps_zi

                mus.append(mu_post_zi)
                log_sigmas.append(log_sigma_post_zi)
                sampled_latents.append(zi) # sampled_latents[i] is z_{i+1}

                # Compute prior for z_{i+1} from z_i (sampled_latents[i-1])
                # prior_predictor_layers[0] for z2's prior from z1
                # prior_predictor_layers[1] for z3's prior from z2
                # So, prior_predictor_layers[i-1] takes sampled_latents[i-1] (z_i)
                # FIX: Remove pad_layer here. Just call the Sequential module directly.
                prior_params = self.prior_predictor_layers[i-1](sampled_latents[i-1])
                mu_prior_zi, log_sigma_prior_zi = prior_params.chunk(2, dim=1)

                mu_priors.append(mu_prior_zi)
                log_sigma_priors.append(log_sigma_prior_zi)

            # The decoder input is always z1 (the highest level latent)
            decoder_input_content = sampled_latents[0]

            return {
                'mus': mus, # List of posterior means [mu_z1, mu_z2, mu_z3]
                'log_sigmas': log_sigmas, # List of posterior log_sigmas [log_sigma_z1, log_sigma_z2, log_sigma_z3]
                'mu_priors': mu_priors, # List of priors [None (for z1), prior_z2_from_z1, prior_z3_from_z2]
                'log_sigma_priors': log_sigma_priors, # Same
                'decoder_input': decoder_input_content # z1 only
            }

        else:
            # Original non-hierarchical output remains the same
            # These are 1x1 convs, no need for external pad_layer.
            mu = self.mean_layer(processed_features)
            log_sigma = self.std_layer(processed_features)

            eps = torch.randn_like(mu)
            z = mu + torch.exp(log_sigma * 0.5) * eps

            return {
                'mus': [mu],
                'log_sigmas': [log_sigma],
                'mu_priors': [None],
                'log_sigma_priors': [None],
                'decoder_input': z
            }

################################################################################

class Decoder(nn.Module):
    def __init__(self,
            c_in, c_cond, c_h, c_out,
            kernel_size,
            n_conv_blocks, upsample, act, sn, dropout_rate):
        super(Decoder, self).__init__()
        self.n_conv_blocks = n_conv_blocks
        self.upsample = upsample
        self.act = get_act(act)
        f = spectral_norm if sn else lambda x: x
        self.in_conv_layer = f(nn.Conv1d(c_in, c_h, kernel_size=1))
        self.first_conv_layers = nn.ModuleList([f(nn.Conv1d(c_h, c_h, kernel_size=kernel_size)) for _ \
                in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList(\
                [f(nn.Conv1d(c_h, c_h * up, kernel_size=kernel_size)) \
                for _, up in zip(range(n_conv_blocks), self.upsample)])
        self.norm_layer = nn.InstanceNorm1d(c_h, affine=False)
        self.conv_affine_layers = nn.ModuleList(
                [f(nn.Linear(c_cond, c_h * 2)) for _ in range(n_conv_blocks*2)])
        self.out_conv_layer = f(nn.Conv1d(c_h, c_out, kernel_size=1))
        self.dropout_layer = nn.Dropout(p=dropout_rate)

    def forward(self, z, cond):
        out = pad_layer(z, self.in_conv_layer)
        out = self.norm_layer(out)
        out = self.act(out)
        out = self.dropout_layer(out)
        # convolution blocks
        for l in range(self.n_conv_blocks):
            y = pad_layer(out, self.first_conv_layers[l])
            y = self.norm_layer(y)
            y = append_cond(y, self.conv_affine_layers[l*2](cond))
            y = self.act(y)
            y = self.dropout_layer(y)
            y = pad_layer(y, self.second_conv_layers[l])
            if self.upsample[l] > 1:
                y = pixel_shuffle_1d(y, scale_factor=self.upsample[l])
            y = self.norm_layer(y)
            y = append_cond(y, self.conv_affine_layers[l*2+1](cond))
            y = self.act(y)
            y = self.dropout_layer(y)
            if self.upsample[l] > 1:
                out = y + upsample(out, scale_factor=self.upsample[l])
            else:
                out = y + out
        out = pad_layer(out, self.out_conv_layer)
        return out

################################################################################

class AE1(nn.Module):
    def __init__(self,  args):
        super(AE1, self).__init__()
        self.use_content_hierarchy = args.use_content_hierarchy
        self.n_content_levels = args.n_content_levels if args.use_content_hierarchy else 1
        self.use_speaker_hierarchy = args.use_speaker_hierarchy
        self.n_speaker_levels = args.n_speaker_levels if args.use_speaker_hierarchy else 1

        # Configure SpeakerEncoder
        speaker_encoder_config = args.config['SpeakerEncoder'].copy()
        speaker_encoder_config['use_hierarchy'] = self.use_speaker_hierarchy
        speaker_encoder_config['n_levels'] = self.n_speaker_levels
        self.speaker_encoder = SpeakerEncoder1(**speaker_encoder_config)

        # Configure ContentEncoder
        content_encoder_config = args.config['ContentEncoder'].copy()
        content_encoder_config['use_hierarchy'] = self.use_content_hierarchy
        content_encoder_config['n_levels'] = self.n_content_levels
        self.content_encoder = ContentEncoder1(**content_encoder_config)

        # Configure Decoder
        decoder_config = args.config['Decoder'].copy()

        # Dynamically set c_in for decoder (input from ContentEncoder)
        decoder_config['c_in'] = content_encoder_config['c_out'] # z1 of content is always c_out

        # Dynamically set c_cond for decoder (input from SpeakerEncoder)
        if self.use_speaker_hierarchy:
            # Speaker decoder input is the concatenation of all speaker latents (z1 || z2 || z3 ...)
            decoder_config['c_cond'] = speaker_encoder_config['c_out'] * self.n_speaker_levels
        else:
            decoder_config['c_cond'] = speaker_encoder_config['c_out']

        self.decoder = Decoder(**decoder_config)

    def forward(self, x):
        speaker_latents_info = self.speaker_encoder(x)
        content_latents_info = self.content_encoder(x)

        # Decoder receives the specific 'decoder_input' from the encoder.
        # For speaker, it's the concatenated embedding.
        # For content, it's z1.
        speaker_decoder_input = speaker_latents_info['decoder_input']
        content_decoder_input = content_latents_info['decoder_input']

        dec = self.decoder(content_decoder_input, speaker_decoder_input)

        return {
            'dec': dec,
            'speaker_latents': speaker_latents_info,
            'content_latents': content_latents_info,
        }

    def inference(self, x, x_cond):
        speaker_latents_info = self.speaker_encoder(x_cond)
        content_latents_info = self.content_encoder(x)

        # For inference, use the posterior mean (mu) of z1 for content, as z1 is reconstructed.
        content_decoder_input = content_latents_info['mus'][0]

        # For speaker, use the `decoder_input` which is the concatenated embedding.
        speaker_decoder_input = speaker_latents_info['decoder_input']

        dec = self.decoder(content_decoder_input, speaker_decoder_input)
        return dec

    def get_speaker_embeddings(self, x):
        # This function should return the final speaker embedding used by the decoder.
        speaker_latents_info = self.speaker_encoder(x)
        return speaker_latents_info['decoder_input']


In [16]:
speaker_encoder_config = {
    "c_in": 80,
    "c_h": 128,
    "c_out": 128,
    "kernel_size": 5,
    "bank_size": 8,
    "bank_scale": 1,
    "c_bank": 128,
    "n_conv_blocks": 6,
    "n_dense_blocks": 6,
    "subsample": [1, 2, 1, 2, 1, 2],
    "act": 'relu',
    "dropout_rate": 0,
    "use_hierarchy": True,
    "n_levels": 3
}

# Instantiate SpeakerEncoder
speaker_encoder = SpeakerEncoder1(**speaker_encoder_config)

# Generate fake input data
batch_size = 128
mel_channels = 80
segment_length = 128 # e.g., 128 frames

rand_input = torch.randn(batch_size, mel_channels, segment_length)
print(f"Random input shape: {rand_input.shape}")

# Pass through the encoder
output = speaker_encoder(rand_input)

# Check the output shapes
print("\n--- Output Shapes ---")
print(f"Decoder input shape: {output['decoder_input'].shape}")

print("\n--- Individual Latent Level Shapes ---")
for i, mu in enumerate(output['mus']):
    print(f"mu_z{i+1} shape: {mu.shape}")

Random input shape: torch.Size([128, 80, 128])

--- Output Shapes ---
Decoder input shape: torch.Size([128, 384])

--- Individual Latent Level Shapes ---
mu_z1 shape: torch.Size([128, 128])
mu_z2 shape: torch.Size([128, 128])
mu_z3 shape: torch.Size([128, 128])


In [20]:

# --- Constants for clamping log_sigma ---
MIN_LOG_SIGMA = -5.0 # From -10.0
MAX_LOG_SIGMA = 1.0  # From 2.0
# ----------------------------------------


class ContentEncoder2(nn.Module):
    def __init__(self, c_in, c_h, c_out, kernel_size,
            bank_size, bank_scale, c_bank,
            n_conv_blocks, subsample,
            act, dropout_rate,
            use_hierarchy=False, n_levels=1):
        super(ContentEncoder2, self).__init__()
        self.n_conv_blocks = n_conv_blocks
        self.subsample = subsample # Defines downsampling in conv blocks (e.g., [2,2,2])
        self.act = get_act(act)
        self.use_hierarchy = use_hierarchy
        self.n_levels = n_levels
        self.c_out = c_out # Dimension for each individual latent level (z1, z2, etc.)

        self.conv_bank = nn.ModuleList(
                [nn.Conv1d(c_in, c_bank, kernel_size=k) for k in range(bank_scale, bank_size + 1, bank_scale)])
        in_channels = c_bank * (bank_size // bank_scale) + c_in
        self.in_conv_layer = nn.Conv1d(in_channels, c_h, kernel_size=1)
        self.first_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size) for _ in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size, stride=sub) for sub in subsample])
        self.norm_layer = nn.InstanceNorm1d(c_h, affine=False)
        self.dropout_layer = nn.Dropout(p=dropout_rate)

        # Global pooling layer for content encoder, same as in SpeakerEncoder
        self.pooling_layer = nn.AdaptiveAvgPool1d(1)


        if self.use_hierarchy:
            # Latent layers for mu/log_sigma and prior predictors are Linear,
            # as they operate on pooled (2D) features.
            self.mu_layers = nn.ModuleList([nn.Linear(c_h, c_out) for _ in range(n_levels)])
            self.log_sigma_layers = nn.ModuleList([nn.Linear(c_h, c_out) for _ in range(n_levels)])

            self.prior_predictor_layers = nn.ModuleList()
            for _ in range(n_levels - 1):
                self.prior_predictor_layers.append(nn.Sequential(
                    nn.Linear(c_out, c_h),
                    self.act,
                    nn.Linear(c_h, c_out * 2)
                ))
        else:
            # For non-hierarchical, also use Linear layers after pooling
            self.mean_layer = nn.Linear(c_h, c_out)
            self.std_layer = nn.Linear(c_h, c_out)

    def forward(self, x):
        out = conv_bank(x, self.conv_bank, act=self.act)
        out = pad_layer(out, self.in_conv_layer)
        out = self.norm_layer(out)
        out = self.act(out)
        out = self.dropout_layer(out)
        # convolution blocks
        for l in range(self.n_conv_blocks):
            residual_out = out
            y = pad_layer(out, self.first_conv_layers[l])
            y = self.norm_layer(y)
            y = self.act(y)
            y = self.dropout_layer(y)
            y = pad_layer(y, self.second_conv_layers[l]) # This layer applies stride
            y = self.norm_layer(y)
            y = self.act(y)
            y = self.dropout_layer(y)

            # Apply subsampling to the residual connection to match y's downsampled shape
            if self.subsample[l] > 1:
                residual_out = F.avg_pool1d(residual_out, kernel_size=self.subsample[l], ceil_mode=True)
            out = y + residual_out

        # Pool the sequential features to get a fixed-size representation
        # `out` is (batch_size, c_h, downsampled_seq_len) after conv blocks
        pooled_features = self.pooling_layer(out).squeeze(2) # Converts to (batch_size, c_h)

        if self.use_hierarchy:
            mus = []
            log_sigmas = []
            mu_priors = []
            log_sigma_priors = []
            sampled_latents = []

            # --- For z1 (Highest Level - standard normal prior) ---
            mu_post_z1 = self.mu_layers[0](pooled_features) # (batch_size, c_out)
            log_sigma_post_z1 = self.log_sigma_layers[0](pooled_features) # (batch_size, c_out)
            log_sigma_post_z1 = torch.clamp(log_sigma_post_z1, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

            eps_z1 = torch.randn_like(mu_post_z1)
            z1 = mu_post_z1 + torch.exp(log_sigma_post_z1 * 0.5) * eps_z1

            mus.append(mu_post_z1)
            log_sigmas.append(log_sigma_post_z1)
            sampled_latents.append(z1)

            mu_priors.append(None)
            log_sigma_priors.append(None)

            # --- For subsequent levels (z2, z3, etc.) ---
            for i in range(1, self.n_levels):
                mu_post_zi = self.mu_layers[i](pooled_features) # (batch_size, c_out)
                log_sigma_post_zi = self.log_sigma_layers[i](pooled_features) # (batch_size, c_out)
                log_sigma_post_zi = torch.clamp(log_sigma_post_zi, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

                eps_zi = torch.randn_like(mu_post_zi)
                zi = mu_post_zi + torch.exp(log_sigma_post_zi * 0.5) * eps_zi

                mus.append(mu_post_zi)
                log_sigmas.append(log_sigma_post_zi)
                sampled_latents.append(zi)

                prior_params = self.prior_predictor_layers[i-1](sampled_latents[i-1])
                mu_prior_zi, log_sigma_prior_zi = prior_params.chunk(2, dim=-1)
                log_sigma_prior_zi = torch.clamp(log_sigma_prior_zi, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

                mu_priors.append(mu_prior_zi)
                log_sigma_priors.append(log_sigma_prior_zi)

            # DECODER INPUT: The sampled z1 (highest level latent)
            decoder_input_content = sampled_latents[0] # This is z1: (batch_size, c_out)

            return {
                'mus': mus, # List of 2D tensors: [(128, 64), (128, 64)]
                'log_sigmas': log_sigmas, # List of 2D tensors: [(128, 64), (128, 64)]
                'mu_priors': mu_priors, # List of 2D tensors: [None, (128, 64)]
                'log_sigma_priors': log_sigma_priors, # Same
                'decoder_input': decoder_input_content # (batch_size, c_out)
            }

        else:
            # Non-hierarchical case: uses mean_layer and std_layer (Linear) on pooled_features
            mu = self.mean_layer(pooled_features) # (batch_size, c_out)
            log_sigma = self.std_layer(pooled_features) # (batch_size, c_out)
            log_sigma = torch.clamp(log_sigma, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

            eps = torch.randn_like(mu)
            z = mu + torch.exp(log_sigma * 0.5) * eps

            return {
                'mus': [mu], # (batch_size, c_out)
                'log_sigmas': [log_sigma], # (batch_size, c_out)
                'mu_priors': [None],
                'log_sigma_priors': [None],
                'decoder_input': z # (batch_size, c_out)
            }


################################################################################


# --- Constants for clamping log_sigma ---
# (Define these globally or ensure they are accessible)
MIN_LOG_SIGMA = -5.0 # From -10.0
MAX_LOG_SIGMA = 1.0  # From 2.0
# ----------------------------------------


class SpeakerEncoder1(nn.Module):
    def __init__(self, c_in, c_h, c_out, kernel_size,
            bank_size, bank_scale, c_bank,
            n_conv_blocks, n_dense_blocks,
            subsample, act, dropout_rate,
            use_hierarchy=False, n_levels=1):
        super(SpeakerEncoder1, self).__init__()
        self.c_in = c_in
        self.c_h = c_h
        self.c_out = c_out # Dimension for each individual latent level (z1, z2, etc.)
        self.kernel_size = kernel_size
        self.n_conv_blocks = n_conv_blocks
        self.n_dense_blocks = n_dense_blocks
        self.subsample = subsample
        self.act = get_act(act)
        self.use_hierarchy = use_hierarchy
        self.n_levels = n_levels

        self.conv_bank = nn.ModuleList(
                [nn.Conv1d(c_in, c_bank, kernel_size=k) for k in range(bank_scale, bank_size + 1, bank_scale)])
        in_channels = c_bank * (bank_size // bank_scale) + c_in
        self.in_conv_layer = nn.Conv1d(in_channels, c_h, kernel_size=1)

        # Convolutional blocks
        self.first_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size) for _ in range(n_conv_blocks)])
        self.second_conv_layers = nn.ModuleList([nn.Conv1d(c_h, c_h, kernel_size=kernel_size, stride=sub) for sub in subsample])

        self.pooling_layer = nn.AdaptiveAvgPool1d(1)

        # Dense blocks
        self.first_dense_layers = nn.ModuleList([nn.Linear(c_h, c_h) for _ in range(n_dense_blocks)])
        self.second_dense_layers = nn.ModuleList([nn.Linear(c_h, c_h) for _ in range(n_dense_blocks)])

        if self.use_hierarchy:
            # For each level (z1, z2, ..., zn), we need posterior mu and log_sigma layers
            # All levels have c_out dimensions
            self.mu_layers = nn.ModuleList([nn.Linear(c_h, c_out) for _ in range(n_levels)])
            self.log_sigma_layers = nn.ModuleList([nn.Linear(c_h, c_out) for _ in range(n_levels)])

            # Prior predictor layers: `prior_predictor_layers[k]` maps from z_{k+1} to prior params for z_k
            # e.g., prior_predictor_layers[0] for z1_prior from z2.
            #       prior_predictor_layers[1] for z2_prior from z3.
            self.prior_predictor_layers = nn.ModuleList()
            for _ in range(n_levels - 1): # (n_levels - 1) prior networks needed
                self.prior_predictor_layers.append(nn.Sequential(
                    nn.Linear(c_out, c_h),
                    self.act,
                    nn.Linear(c_h, c_out * 2) # For mu_prior and log_sigma_prior
                ))
        else:
            self.output_layer = nn.Linear(c_h, c_out)

        self.dropout_layer = nn.Dropout(p=dropout_rate)

    def conv_blocks(self, inp):
        out = inp
        for l in range(self.n_conv_blocks):
            y = pad_layer(out, self.first_conv_layers[l])
            y = self.act(y)
            y = self.dropout_layer(y)
            y = pad_layer(y, self.second_conv_layers[l])
            y = self.act(y)
            y = self.dropout_layer(y)
            if self.subsample[l] > 1:
                out = F.avg_pool1d(out, kernel_size=self.subsample[l], ceil_mode=True)
            out = y + out # Residual connection
        return out

    def dense_blocks(self, inp):
        out = inp
        for l in range(self.n_dense_blocks):
            y = self.first_dense_layers[l](out)
            y = self.act(y)
            y = self.dropout_layer(y)
            y = self.second_dense_layers[l](y)
            y = self.act(y)
            y = self.dropout_layer(y)
            out = y + out # Residual connection
        return out

    def forward(self, x):
        out = conv_bank(x, self.conv_bank, act=self.act)
        out = pad_layer(out, self.in_conv_layer)
        out = self.act(out)
        out = self.conv_blocks(out)
        out = self.pooling_layer(out).squeeze(2) # Shape: (batch_size, c_h)
        pooled_features = self.dense_blocks(out) # pooled_features is (batch_size, c_h)

        if self.use_hierarchy:
            mus = []
            log_sigmas = []
            mu_priors = [] # [None (for z1), prior for z2 (from z1), prior for z3 (from z2)]
            log_sigma_priors = [] # Same
            sampled_latents = []

            # --- For z1 (Highest Level - standard normal prior) ---
            # Posterior for z1 from pooled_features
            mu_post_z1 = self.mu_layers[0](pooled_features)
            log_sigma_post_z1 = self.log_sigma_layers[0](pooled_features)
            log_sigma_post_z1 = torch.clamp(log_sigma_post_z1, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

            # Sample z1
            eps_z1 = torch.randn_like(mu_post_z1)
            z1 = mu_post_z1 + torch.exp(log_sigma_post_z1 * 0.5) * eps_z1

            mus.append(mu_post_z1)
            log_sigmas.append(log_sigma_post_z1)
            sampled_latents.append(z1)

            # Prior for z1 is standard normal (N(0,I))
            mu_priors.append(None)
            log_sigma_priors.append(None)

            # --- For subsequent levels (z2, z3, etc.) ---
            # Loop from the second level (index 1) up to n_levels-1
            for i in range(1, self.n_levels):
                # Posterior for z_{i+1} from pooled_features
                mu_post_zi = self.mu_layers[i](pooled_features)
                log_sigma_post_zi = self.log_sigma_layers[i](pooled_features)
                log_sigma_post_zi = torch.clamp(log_sigma_post_zi, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

                # Sample z_{i+1}
                eps_zi = torch.randn_like(mu_post_zi)
                zi = mu_post_zi + torch.exp(log_sigma_post_zi * 0.5) * eps_zi

                mus.append(mu_post_zi)
                log_sigmas.append(log_sigma_post_zi)
                sampled_latents.append(zi) # sampled_latents[i] is z_{i+1}

                # Compute prior for z_{i+1} from z_i (sampled_latents[i-1])
                prior_params = self.prior_predictor_layers[i-1](sampled_latents[i-1])
                mu_prior_zi, log_sigma_prior_zi = prior_params.chunk(2, dim=-1)
                log_sigma_prior_zi = torch.clamp(log_sigma_prior_zi, MIN_LOG_SIGMA, MAX_LOG_SIGMA) # <--- CLAMPING

                mu_priors.append(mu_prior_zi)
                log_sigma_priors.append(log_sigma_prior_zi)

            # The decoder input for speaker is the concatenation of all sampled speaker latents
            decoder_input_emb = torch.cat(sampled_latents, dim=-1)

            return {
                'mus': mus, # List of posterior means [mu_z1, mu_z2, mu_z3]
                'log_sigmas': log_sigmas, # List of posterior log_sigmas [log_sigma_z1, log_sigma_z2, log_sigma_z3]
                'mu_priors': mu_priors, # List of priors [None (for z1), prior_z2_from_z1, prior_z3_from_z2]
                'log_sigma_priors': log_sigma_priors, # Same
                'decoder_input': decoder_input_emb # Concatenated speaker embedding (z1 || z2 || z3 ...)
            }

        else:
            # Original non-hierarchical output remains the same (log_sigma is always 0 here)
            emb = self.output_layer(pooled_features)
            return {
                'mus': [emb],
                'log_sigmas': [torch.zeros_like(emb)], # This is always 0, no clamping needed here.
                'mu_priors': [None],
                'log_sigma_priors': [None],
                'decoder_input': emb
            }

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Assuming get_act, spectral_norm, append_cond, pixel_shuffle_1d, pad_layer are defined.
# You MUST ensure your 'upsample' helper function is defined and correctly handles
# upsampling for residual connections (e.g., F.interpolate or ConvTranspose1d).

class Decoder2(nn.Module): # Renamed back to Decoder for consistency
    def __init__(self,
            c_in,           # Dimension of the flat content latent z (e.g., 64 or 128)
            c_cond,         # Dimension of the conditioning vector (speaker embedding)
            c_h,            # Hidden channel dimension for conv blocks
            c_out,          # Output mel channel dimension
            kernel_size,
            n_conv_blocks,
            upsample,       # List of upsampling factors (e.g., [2, 1, 2, 1, 2, 1])
            act, sn, dropout_rate,
            target_output_seq_len=128 # Default to 128 frames for convenience
            ):
        super(Decoder2, self).__init__()

        self.n_conv_blocks = n_conv_blocks
        self.upsample = upsample # This is your list [2, 1, 2, 1, 2, 1]
        self.act = get_act(act)
        f = spectral_norm if sn else lambda x: x

        self.c_h = c_h # Store c_h for view operation

        # Calculate total upsampling factor from the provided 'upsample' list
        total_upsample_factor = 1
        for factor in self.upsample:
            total_upsample_factor *= factor

        # Infer the initial sequence length required for the first linear layer
        if total_upsample_factor == 0:
            raise ValueError("Total upsample factor cannot be zero. Check your upsample config.")

        self.initial_seq_len = target_output_seq_len // total_upsample_factor
        if target_output_seq_len % total_upsample_factor != 0:
            print(f"Warning: target_output_seq_len ({target_output_seq_len}) is not perfectly divisible "
                  f"by total_upsample_factor ({total_upsample_factor}). Output length might be rounded "
                  f"or an error might occur depending on padding/upsampling specifics.")

        # The first layer that handles the 2D input 'z'
        # Input: (batch_size, c_in)
        # Output: (batch_size, c_h * self.initial_seq_len)
        self.initial_linear = f(nn.Linear(c_in, c_h * self.initial_seq_len))

        # Remaining layers are as per your original structure
        self.first_conv_layers = nn.ModuleList([f(nn.Conv1d(c_h, c_h, kernel_size=kernel_size)) for _ \
                in range(n_conv_blocks)])

        self.second_conv_layers = nn.ModuleList(\
                [f(nn.Conv1d(c_h, c_h * up_factor, kernel_size=kernel_size)) \
                for up_factor in self.upsample]) # Use self.upsample directly here

        self.norm_layer = nn.InstanceNorm1d(c_h, affine=False)

        self.conv_affine_layers = nn.ModuleList(
                [f(nn.Linear(c_cond, c_h * 2)) for _ in range(n_conv_blocks*2)])

        self.out_conv_layer = f(nn.Conv1d(c_h, c_out, kernel_size=1))
        # Dropout layer initialized here with dropout_rate
        self.dropout_layer = nn.Dropout(p=dropout_rate)

    def forward(self, z, cond):
        # 1. Transform the 2D content latent 'z' into a 3D feature map
        # z: (batch_size, c_in)
        out = self.initial_linear(z)

        # Reshape to (batch_size, c_h, initial_seq_len)
        out = out.view(out.size(0), self.c_h, self.initial_seq_len)

        out = self.norm_layer(out)
        out = self.act(out)
        out = self.dropout_layer(out) # Corrected: simply call the module with the input tensor

        # convolution blocks
        for l in range(self.n_conv_blocks):
            y = pad_layer(out, self.first_conv_layers[l])
            y = self.norm_layer(y)
            y = append_cond(y, self.conv_affine_layers[l*2](cond))
            y = self.act(y)
            y = self.dropout_layer(y) # Corrected

            y = pad_layer(y, self.second_conv_layers[l])

            # Apply pixel shuffle for upsampling
            up_factor = self.upsample[l]
            if up_factor > 1:
                y = pixel_shuffle_1d(y, scale_factor=up_factor)

            y = self.norm_layer(y)
            y = append_cond(y, self.conv_affine_layers[l*2+1](cond))
            y = self.act(y)
            y = self.dropout_layer(y) # Corrected

            # Residual connection: upsample 'out' to match 'y's new sequence length
            if up_factor > 1:
                out = y + upsample(out, scale_factor=up_factor)
            else:
                out = y + out

        out = pad_layer(out, self.out_conv_layer)
        return out



################################################

class AE2(nn.Module): ############# MRH: change AE1 > AE2
    def __init__(self,  args):
        super(AE2, self).__init__()
        self.use_content_hierarchy = args.use_content_hierarchy
        self.n_content_levels = args.n_content_levels if args.use_content_hierarchy else 1
        self.use_speaker_hierarchy = args.use_speaker_hierarchy
        self.n_speaker_levels = args.n_speaker_levels if args.use_speaker_hierarchy else 1

        # Configure SpeakerEncoder
        speaker_encoder_config = args.config['SpeakerEncoder'].copy()
        speaker_encoder_config['use_hierarchy'] = self.use_speaker_hierarchy
        speaker_encoder_config['n_levels'] = self.n_speaker_levels
        self.speaker_encoder = SpeakerEncoder1(**speaker_encoder_config)

        # Configure ContentEncoder
        content_encoder_config = args.config['ContentEncoder'].copy()
        content_encoder_config['use_hierarchy'] = self.use_content_hierarchy
        content_encoder_config['n_levels'] = self.n_content_levels
        self.content_encoder = ContentEncoder2(**content_encoder_config) ############ MRH: change ContentEncoder1 to ContentEncoder2

        # Configure Decoder
        decoder_config = args.config['Decoder'].copy()

        # Dynamically set c_in for decoder (input from ContentEncoder)
        decoder_config['c_in'] = content_encoder_config['c_out'] # z1 of content is always c_out

        # Dynamically set c_cond for decoder (input from SpeakerEncoder)
        if self.use_speaker_hierarchy:
            # Speaker decoder input is the concatenation of all speaker latents (z1 || z2 || z3 ...)
            decoder_config['c_cond'] = speaker_encoder_config['c_out'] * self.n_speaker_levels
        else:
            decoder_config['c_cond'] = speaker_encoder_config['c_out']

        self.decoder = Decoder2(**decoder_config)

    def forward(self, x):
        speaker_latents_info = self.speaker_encoder(x)
        content_latents_info = self.content_encoder(x)

        # Decoder receives the specific 'decoder_input' from the encoder.
        # For speaker, it's the concatenated embedding.
        # For content, it's z1.
        speaker_decoder_input = speaker_latents_info['decoder_input']
        content_decoder_input = content_latents_info['decoder_input']

        # print('speaker_decoder_input= ',speaker_decoder_input.shape)
        # print('content_decoder_input= ',content_decoder_input.shape)

        dec = self.decoder(content_decoder_input, speaker_decoder_input)

        return {
            'dec': dec,
            'speaker_latents': speaker_latents_info,
            'content_latents': content_latents_info,
        }

    def inference(self, x, x_cond):
        speaker_latents_info = self.speaker_encoder(x_cond)
        content_latents_info = self.content_encoder(x)

        # For inference, use the posterior mean (mu) of z1 for content, as z1 is reconstructed.
        content_decoder_input = content_latents_info['mus'][0]

        # For speaker, use the `decoder_input` which is the concatenated embedding.
        speaker_decoder_input = speaker_latents_info['decoder_input']

        dec = self.decoder(content_decoder_input, speaker_decoder_input)
        return dec

    def get_speaker_embeddings(self, x):
        # This function should return the final speaker embedding used by the decoder.
        speaker_latents_info = self.speaker_encoder(x)
        return speaker_latents_info['decoder_input']


**-----------------------------------------**

**----------> 6- Inference <----------------**

**-----------------------------------------**

In [10]:
####################
# preprocess/tacotron/utils.py

# -*- coding: utf-8 -*-
# /usr/bin/python2
'''
By kyubyong park. kbpark.linguist@gmail.com.
https://www.github.com/kyubyong/dc_tts
'''
from __future__ import print_function, division

# MRH:
# from .hyperparams import Hyperparams as hp

import numpy as np
import tensorflow as tf
import librosa
import copy
#import matplotlib
#matplotlib.use('pdf')
#import matplotlib.pyplot as plt
from scipy import signal
import os

def _mel_to_linear_matrix(sr, n_fft, n_mels):
    m = librosa.filters.mel(sr, n_fft, n_mels)
    m_t = np.transpose(m)
    p = np.matmul(m, m_t)
    d = [1.0 / x if np.abs(x) > 1.0e-8 else x for x in np.sum(p, axis=0)]
    return np.matmul(m_t, np.diag(d))

def get_spectrograms(fpath):
    '''Returns normalized log(melspectrogram) and log(magnitude) from `sound_file`.
    Args:
      sound_file: A string. The full path of a sound file.

    Returns:
      mel: A 2d array of shape (T, n_mels) <- Transposed
      mag: A 2d array of shape (T, 1+n_fft/2) <- Transposed
    '''
    # num = np.random.randn()
    # if num < .2:
    #     y, sr = librosa.load(fpath, sr=hp.sr)
    # else:
    #     if num < .4:
    #         tempo = 1.1
    #     elif num < .6:
    #         tempo = 1.2
    #     elif num < .8:
    #         tempo = 0.9
    #     else:
    #         tempo = 0.8
    #     cmd = "ffmpeg -i {} -y ar {} -hide_banner -loglevel panic -ac 1 -filter:a atempo={} -vn temp.wav".format(fpath, hp.sr, tempo)
    #     os.system(cmd)
    #     y, sr = librosa.load('temp.wav', sr=hp.sr)

    # Loading sound file
    y, sr = librosa.load(fpath, sr=hp.sr)


    # Trimming
    y, _ = librosa.effects.trim(y, top_db=hp.top_db)

    # Preemphasis
    y = np.append(y[0], y[1:] - hp.preemphasis * y[:-1])

    # stft
    linear = librosa.stft(y=y,
                          n_fft=hp.n_fft,
                          hop_length=hp.hop_length,
                          win_length=hp.win_length)

    # magnitude spectrogram
    mag = np.abs(linear)  # (1+n_fft//2, T)

    # mel spectrogram
    mel_basis = librosa.filters.mel(hp.sr, hp.n_fft, hp.n_mels)  # (n_mels, 1+n_fft//2)
    mel = np.dot(mel_basis, mag)  # (n_mels, t)

    # to decibel
    mel = 20 * np.log10(np.maximum(1e-5, mel))
    mag = 20 * np.log10(np.maximum(1e-5, mag))

    # normalize
    mel = np.clip((mel - hp.ref_db + hp.max_db) / hp.max_db, 1e-8, 1)
    mag = np.clip((mag - hp.ref_db + hp.max_db) / hp.max_db, 1e-8, 1)

    # Transpose
    mel = mel.T.astype(np.float32)  # (T, n_mels)
    mag = mag.T.astype(np.float32)  # (T, 1+n_fft//2)

    return mel, mag

def melspectrogram2wav(mel):
    '''# Generate wave file from spectrogram'''
    # transpose
    mel = mel.T

    # de-noramlize
    mel = (np.clip(mel, 0, 1) * hp.max_db) - hp.max_db + hp.ref_db

    # to amplitude
    mel = np.power(10.0, mel * 0.05)
    m = _mel_to_linear_matrix(hp.sr, hp.n_fft, hp.n_mels)
    mag = np.dot(m, mel)

    # wav reconstruction
    wav = griffin_lim(mag)

    # de-preemphasis
    wav = signal.lfilter([1], [1, -hp.preemphasis], wav)

    # trim
    wav, _ = librosa.effects.trim(wav)

    return wav.astype(np.float32)

def spectrogram2wav(mag):
    '''# Generate wave file from spectrogram'''
    # transpose
    mag = mag.T

    # de-noramlize
    mag = (np.clip(mag, 0, 1) * hp.max_db) - hp.max_db + hp.ref_db

    # to amplitude
    mag = np.power(10.0, mag * 0.05)

    # wav reconstruction
    wav = griffin_lim(mag)

    # de-preemphasis
    wav = signal.lfilter([1], [1, -hp.preemphasis], wav)

    # trim
    wav, _ = librosa.effects.trim(wav)

    return wav.astype(np.float32)


def griffin_lim(spectrogram):
    '''Applies Griffin-Lim's raw.
    '''
    X_best = copy.deepcopy(spectrogram)
    for i in range(hp.n_iter):
        X_t = invert_spectrogram(X_best)
        est = librosa.stft(X_t, hp.n_fft, hp.hop_length, win_length=hp.win_length)
        phase = est / np.maximum(1e-8, np.abs(est))
        X_best = spectrogram * phase
    X_t = invert_spectrogram(X_best)
    y = np.real(X_t)

    return y


def invert_spectrogram(spectrogram):
    '''
    spectrogram: [f, t]
    '''
    return librosa.istft(spectrogram, hp.hop_length, win_length=hp.win_length, window="hann")


def plot_alignment(alignment, gs):
    """Plots the alignment
    alignments: A list of (numpy) matrix of shape (encoder_steps, decoder_steps)
    gs : (int) global step
    """
    fig, ax = plt.subplots()
    im = ax.imshow(alignment)

    # cbar_ax = fig.add_axes([0.85, 0.15, 0.05, 0.7])
    fig.colorbar(im)
    plt.title('{} Steps'.format(gs))
    plt.savefig('{}/alignment_{}k.png'.format(hp.logdir, gs//1000), format='png')

def learning_rate_decay(init_lr, global_step, warmup_steps=4000.):
    '''Noam scheme from tensor2tensor'''
    step = tf.cast(global_step + 1, dtype=tf.float32)
    return init_lr * warmup_steps ** 0.5 * tf.minimum(step * warmup_steps ** -1.5, step ** -0.5)

def load_spectrograms(fpath):
    fname = os.path.basename(fpath)
    mel, mag = get_spectrograms(fpath)
    t = mel.shape[0]
    num_paddings = hp.r - (t % hp.r) if t % hp.r != 0 else 0 # for reduction
    mel = np.pad(mel, [[0, num_paddings], [0, 0]], mode="constant")
    mag = np.pad(mag, [[0, num_paddings], [0, 0]], mode="constant")
    return fname, mel.reshape((-1, hp.n_mels*hp.r)), mag

In [11]:

####################
import numpy as np
if not hasattr(np, 'float'):
    np.float = float


#adaptive_voice_conversion/inference.py
import torch
# import numpy as np
import sys
import os
import torch.nn as nn
import torch.nn.functional as F
import yaml
import pickle
# from model import AE
# from utils import *
from functools import reduce
import json
from collections import defaultdict
from torch.utils.data import Dataset
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
from argparse import ArgumentParser, Namespace
from scipy.io.wavfile import write
import random
# from preprocess.tacotron.utils import melspectrogram2wav
# from preprocess.tacotron.utils import get_spectrograms
import librosa

class Inferencer(object):
    def __init__(self, config, args):
        # config store the value of hyperparameters, turn to attr by AttrDict
        self.config = config
        # print(config)
        # args store other information
        self.args = args
        # print(self.args)

        # init the model with config
        self.build_model()

        # load model
        # self.load_model() #### MRH: i just commented but since i use online model... but for offline we need this

        with open(self.args.attr, 'rb') as f:
            self.attr = pickle.load(f)

    def load_model(self):
        print(f'Load model from {self.args.model}')
        self.model.load_state_dict(torch.load(f'{self.args.model}'))
        return

    def build_model(self):
        # create model, discriminator, optimizers
        # self.model = cc(AE(self.config))
        self.model = cc(AE1(self.args))
        # print(self.model)
        self.model.eval()
        return

    def utt_make_frames(self, x):
        frame_size = self.config['data_loader']['frame_size']
        remains = x.size(0) % frame_size
        if remains != 0:
            x = F.pad(x, (0, remains))
        out = x.view(1, x.size(0) // frame_size, frame_size * x.size(1)).transpose(1, 2)
        return out

    def inference_one_utterance(self, x, x_cond):
        x = self.utt_make_frames(x)
        x_cond = self.utt_make_frames(x_cond)
        dec = self.model.inference(x, x_cond)
        # print('dec shape= ',dec.shape)
        dec = dec.transpose(1, 2).squeeze(0)
        # print('dec shape= ',dec.shape)
        dec = dec.detach().cpu().numpy()
        # print('dec shape= ',dec.shape)
        dec = self.denormalize(dec)
        # print('dec shape= ',dec.shape)
        wav_data = melspectrogram2wav(dec)
        return wav_data, dec

    def denormalize(self, x):
        m, s = self.attr['mean'], self.attr['std']
        ret = x * s + m
        return ret

    def normalize(self, x):
        m, s = self.attr['mean'], self.attr['std']
        ret = (x - m) / s
        return ret

    def write_wav_to_file(self, wav_data, output_path):
        write(output_path, rate=self.args.sample_rate, data=wav_data)
        return

    def inference_from_path(self):
        src_mel, _ = get_spectrograms(self.args.source)
        tar_mel, _ = get_spectrograms(self.args.target)
        src_mel = torch.from_numpy(self.normalize(src_mel)).cuda()
        tar_mel = torch.from_numpy(self.normalize(tar_mel)).cuda()
        conv_wav, conv_mel = self.inference_one_utterance(src_mel, tar_mel)
        self.write_wav_to_file(conv_wav, self.args.output)
        return

    #########################################
    def mrh_inference_from_mel_spectograms(self,src_mel,tar_mel):
        # src_mel, _ = get_spectrograms(self.args.source)
        # tar_mel, _ = get_spectrograms(self.args.target)
        sample_source_mel = src_mel[0:1, :, :] # Take the first sample, keep batch dimension
        sample_target_mel = tar_mel[0:1, :, :] # Take the first sample, keep batch dimension
        # src_mel = self.normalize(example_source_mel).cuda() MRH: we do not need normalization since it is already normalized
        # tar_mel = self.normalize(example_target_mel).cuda()

        sample_source_mel = sample_source_mel.transpose(1, 2).squeeze(0)
        sample_target_mel = sample_target_mel.transpose(1, 2).squeeze(0)
        #
        sample_source_mel = sample_source_mel.detach().cpu().numpy()
        sample_target_mel = sample_target_mel.detach().cpu().numpy()
        #
        sample_source_mel = self.denormalize(sample_source_mel)
        sample_target_mel = self.denormalize(sample_target_mel)
        #
        sample_source_wav_data = melspectrogram2wav(sample_source_mel)
        sample_target_wav_data = melspectrogram2wav(sample_target_mel)
        #
        return sample_source_wav_data, sample_target_wav_data

        conv_wav, conv_mel = self.inference_one_utterance(src_mel, tar_mel)
        # self.write_wav_to_file(conv_wav, self.args.output)
        return conv_wav, conv_mel


**------------------------------------**

**--------> 7- Losses <----------------**

**------------------------------------**

In [22]:
class AELoss2:
    def __init__(self, n_speaker_levels: int, n_content_levels: int):
        """
        Initializes the AELoss class.

        Args:
            n_speaker_levels: Number of speaker hierarchy levels.
            n_content_levels: Number of content hierarchy levels.
        """
        self.n_speaker_levels = n_speaker_levels
        self.n_content_levels = n_content_levels

        print(' mrh inside AELoss: n_speaker_levels, n_content_levels = ',self.n_speaker_levels, self.n_content_levels )


    def calculate_kld(self, mu_post, log_sigma_post, mu_prior=None, log_sigma_prior=None):
        # ... (clamp if not already done by encoder or for safety)

        if mu_prior is None or log_sigma_prior is None:
            # Standard normal prior case (mu_prior=0, log_sigma_prior=0 implies variance=1)
            # This is D_KL(N(mu_post, exp(log_sigma_post)) || N(0, 1))
            kld_element = 0.5 * (
                mu_post.pow(2) +
                torch.exp(log_sigma_post) - # This is sigma_post^2 (variance)
                1 -
                log_sigma_post              # This is log(sigma_post^2)
            )
            kld = torch.sum(kld_element)

            # --- DEBUG PRINTS FOR STANDARD PRIOR ---
            # print("\n--- DEBUG KLD (Standard Prior) ---")
            # print(f"  mu_post stats (min/max/mean/std): {mu_post.min().item():.4f}/{mu_post.max().item():.4f}/{mu_post.mean().item():.4f}/{mu_post.std().item():.4f}")
            # print(f"  log_sigma_post stats (min/max/mean/std): {log_sigma_post.min().item():.4f}/{log_sigma_post.max().item():.4f}/{log_sigma_post.mean().item():.4f}/{log_sigma_post.std().item():.4f}")
            # print(f"  sigma_post^2 (variance) stats (min/max/mean/std): {torch.exp(log_sigma_post).min().item():.4f}/{torch.exp(log_sigma_post).max().item():.4f}/{torch.exp(log_sigma_post).mean().item():.4f}/{torch.exp(log_sigma_post).std().item():.4f}")
            # print(f"  KLD Element stats (min/max/mean/std): {kld_element.min().item():.4f}/{kld_element.max().item():.4f}/{kld_element.mean().item():.4f}/{kld_element.std().item():.4f}")
            # print(f"  Summed KLD: {kld.item():.4f}")
            # --- END DEBUG PRINTS ---

        else: # Gaussian prior (N(mu_prior, exp(log_sigma_prior)))
            # D_KL(N(mu_post, exp(log_sigma_post)) || N(mu_prior, exp(log_sigma_prior)))
            kld_element = 0.5 * (
                ((mu_prior - mu_post)**2) * torch.exp(-log_sigma_prior) + # Term 1: (mu_diff)^2 / sigma_prior^2
                torch.exp(log_sigma_post - log_sigma_prior) -             # Term 2: sigma_post^2 / sigma_prior^2
                1 +                                                       # Term 3: -1
                (log_sigma_prior - log_sigma_post)                        # Term 4: log(sigma_prior^2 / sigma_post^2)
            )
            kld = torch.sum(kld_element)

            # --- DEBUG PRINTS FOR GAUSSIAN PRIOR ---
            # print("\n--- DEBUG KLD (Gaussian Prior) ---")
            # print(f"  mu_post stats (min/max/mean/std): {mu_post.min().item():.4f}/{mu_post.max().item():.4f}/{mu_post.mean().item():.4f}/{mu_post.std().item():.4f}")
            # print(f"  log_sigma_post stats (min/max/mean/std): {log_sigma_post.min().item():.4f}/{log_sigma_post.max().item():.4f}/{log_sigma_post.mean().item():.4f}/{log_sigma_post.std().item():.4f}")
            # print(f"  mu_prior stats (min/max/mean/std): {mu_prior.min().item():.4f}/{mu_prior.max().item():.4f}/{mu_prior.mean().item():.4f}/{mu_prior.std().item():.4f}")
            # print(f"  log_sigma_prior stats (min/max/mean/std): {log_sigma_prior.min().item():.4f}/{log_sigma_prior.max().item():.4f}/{log_sigma_prior.mean().item():.4f}/{log_sigma_prior.std().item():.4f}")

            term1 = ((mu_prior - mu_post)**2) * torch.exp(-log_sigma_prior)
            term2 = torch.exp(log_sigma_post - log_sigma_prior)
            term4 = (log_sigma_prior - log_sigma_post)

            # print(f"  Term 1 (mu_diff^2/prior_var) stats (min/max/mean/std): {term1.min().item():.4f}/{term1.max().item():.4f}/{term1.mean().item():.4f}/{term1.std().item():.4f}")
            # print(f"  Term 2 (post_var/prior_var) stats (min/max/mean/std): {term2.min().item():.4f}/{term2.max().item():.4f}/{term2.mean().item():.4f}/{term2.std().item():.4f}")
            # print(f"  Term 4 (log(prior_var/post_var)) stats (min/max/mean/std): {term4.min().item():.4f}/{term4.max().item():.4f}/{term4.mean().item():.4f}/{term4.std().item():.4f}")
            # print(f"  KLD Element stats (min/max/mean/std): {kld_element.min().item():.4f}/{kld_element.max().item():.4f}/{kld_element.mean().item():.4f}/{kld_element.std().item():.4f}")
            # print(f"  Summed KLD: {kld.item():.4f}")
            # # --- END DEBUG PRINTS ---

        return kld


    def speaker_kld_loss(self, speaker_latents):
        """
        Calculates the KLD loss for the speaker hierarchy.
        """
        speaker_kld_loss = 0
        for i in range(self.n_speaker_levels):
            mu_post = speaker_latents['mus'][i]
            log_sigma_post = speaker_latents['log_sigmas'][i]
            mu_prior = speaker_latents['mu_priors'][i]
            log_sigma_prior = speaker_latents['log_sigma_priors'][i]
            speaker_kld_loss += self.calculate_kld(mu_post, log_sigma_post, mu_prior, log_sigma_prior)
        return speaker_kld_loss

    def content_kld_loss(self, content_latents):
        """
        Calculates the KLD loss for the content hierarchy.
        """
        content_kld_loss = 0
        for i in range(self.n_content_levels):
            mu_post = content_latents['mus'][i]
            log_sigma_post = content_latents['log_sigmas'][i]
            mu_prior = content_latents['mu_priors'][i]
            log_sigma_prior = content_latents['log_sigma_priors'][i]
            content_kld_loss += self.calculate_kld(mu_post, log_sigma_post, mu_prior, log_sigma_prior)
        return content_kld_loss

    def loss_calculate(self, x, dec, speaker_latents, content_latents, lambda_kl: float = 1.0):
        """
        Calculates the total loss for the Autoencoder.

        Args:
            x: Original input tensor.
            dec: Reconstructed output tensor from the decoder.
            speaker_latents: Dictionary containing speaker latent distributions.
            content_latents: Dictionary containing content latent distributions.
            lambda_kl: Weight for the KLD loss components.

        Returns:
            A dictionary containing individual and total loss components.
        """
        # Reconstruction Loss (e.g., L1)
        # print('---- mrh ')
        recon_loss = F.l1_loss(dec, x)

        speaker_kld_loss = self.speaker_kld_loss(speaker_latents)
        content_kld_loss = self.content_kld_loss(content_latents)

        # Total Loss with KLD weighting
        total_loss = recon_loss + lambda_kl * (speaker_kld_loss + content_kld_loss)
        # total_loss = recon_loss +  (speaker_kld_loss + content_kld_loss)

        return {
            'total_loss': total_loss,
            'recon_loss': recon_loss,
            'speaker_kld_loss': speaker_kld_loss,
            'content_kld_loss': content_kld_loss
        }

**---------------------------------------------------------------**

**-------------> 8- Evaluation Function: <------------**

class SpeakerVerificationEvaluator

class MCDMetric

class VoiceQualityEvaluator

class ASRWhisperEvaluator

class ASRWav2Vec2Evaluator

**---------------------------------------------------------------**

In [14]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_curve, auc
from tqdm.auto import tqdm
import itertools # For generating pairs
################################################################################
class SpeakerVerificationEvaluator:
    def __init__(self, device='cpu'):
        """
        Initializes the SpeakerVerificationEvaluator.

        Args:
            device (str): The device to perform calculations on ('cuda' or 'cpu').
        """
        self.device = torch.device(device)

    def _extract_speaker_id_from_filename(self, utt_id_filename):
        """
        Extracts the speaker ID from a filename like 'p229_018.wav'.
        Assumes the speaker ID is the first part before the first underscore.
        """
        if not isinstance(utt_id_filename, str):
            raise TypeError(f"Expected utt_id_filename to be a string, but got {type(utt_id_filename)}")
        return utt_id_filename.split('_')[0]

    def _calculate_cosine_similarities(self, embeddings, speaker_ids_or_filenames):
        """
        Calculates cosine similarities for all possible pairs of embeddings.

        Args:
            embeddings (list of torch.Tensor): List of individual speaker embedding tensors.
                                               Each tensor should be 1-D, e.g., (embedding_dim,).
            speaker_ids_or_filenames (list): List of corresponding speaker IDs (strings)
                                             OR filenames (strings) from which IDs can be extracted.

        Returns:
            tuple: (similarities (np.array), labels (np.array))
                   similarities: cosine similarity scores for each pair.
                   labels: 1 for same speaker pair, 0 for different speaker pair.
        """
        if not embeddings or len(embeddings) != len(speaker_ids_or_filenames):
            raise ValueError("Embeddings and speaker_ids_or_filenames lists must be non-empty and of same length.")

        # Determine if we need to extract speaker IDs
        # We assume if the first element is a string containing '.wav', it's a filename.
        # Otherwise, assume it's already a speaker ID.
        if speaker_ids_or_filenames and isinstance(speaker_ids_or_filenames[0], str) and '.wav' in speaker_ids_or_filenames[0]:
            actual_speaker_ids = [self._extract_speaker_id_from_filename(f) for f in speaker_ids_or_filenames]
            print("Note: Speaker IDs extracted from filenames for evaluation.")
        else:
            actual_speaker_ids = speaker_ids_or_filenames
            print("Note: Using provided speaker IDs directly for evaluation.")


        n_embeddings = len(embeddings)
        print(f"Calculating similarities for {n_embeddings} embeddings. This involves "
              f"{n_embeddings * (n_embeddings - 1) // 2} pairs and can take a while for large datasets.")

        # Stack embeddings into a single tensor for potentially faster batch processing on GPU
        # Ensure embeddings are on the correct device
        embeddings_tensor = torch.stack(embeddings).to(self.device)

        similarities = []
        labels = []

        # Iterate through all unique pairs (i, j) where i < j
        # tqdm provides a progress bar
        for i, j in tqdm(itertools.combinations(range(n_embeddings), 2),
                         total=n_embeddings * (n_embeddings - 1) // 2,
                         desc="Calculating embedding pairs"):
            # Cosine similarity expects 2D tensors (batch_size, embedding_dim)
            # Unsqueeze adds a batch dimension of 1
            sim = F.cosine_similarity(embeddings_tensor[i].unsqueeze(0), embeddings_tensor[j].unsqueeze(0)).item()
            similarities.append(sim)
            labels.append(1 if actual_speaker_ids[i] == actual_speaker_ids[j] else 0)

        return np.array(similarities), np.array(labels)

    def _calculate_eer(self, fpr, tpr, thresholds):
        """
        Calculates the Equal Error Rate (EER) and the corresponding threshold.
        EER is the point where False Positive Rate (FPR) equals False Negative Rate (FNR).
        FNR = 1 - True Positive Rate (TPR).
        """
        fnr = 1 - tpr
        # Find the index where FPR and FNR are closest
        eer_threshold_idx = np.argmin(np.abs(fnr - fpr))
        eer = (fpr[eer_threshold_idx] + fnr[eer_threshold_idx]) / 2 # Average for precision
        eer_threshold = thresholds[eer_threshold_idx]
        return eer, eer_threshold

    def evaluate(self, embeddings, speaker_ids_or_filenames):
        """
        Performs speaker verification evaluation using EER and ROC AUC.

        Args:
            embeddings (list of torch.Tensor): A list of speaker embedding tensors.
                                               Each tensor should be 1-D (embedding_dim,).
                                               It's crucial to pass embeddings from a diverse set of
                                               speakers and multiple utterances per speaker for
                                               meaningful EER/ROC AUC results.
            speaker_ids_or_filenames (list): A list of corresponding speaker IDs (strings)
                                             OR filenames (strings) for each embedding.
                                             If filenames are provided (e.g., 'p229_018.wav'),
                                             the speaker ID will be extracted from them.

        Returns:
            dict: A dictionary containing:
                  'eer': Equal Error Rate.
                  'eer_threshold': The threshold at which EER occurs.
                  'roc_auc': Area Under the Receiver Operating Characteristic Curve.
                  'fpr': False Positive Rates for the ROC curve.
                  'tpr': True Positive Rates for the ROC curve.
                  'thresholds': Thresholds used for FPR/TPR calculation.
        """
        # Basic input validation
        if len(set(speaker_ids_or_filenames)) < 2: # Check unique values in the raw input list
            raise ValueError("Need embeddings from at least two different speakers for meaningful EER/ROC AUC calculation.")
        if len(embeddings) < 2:
            raise ValueError("Need at least two embeddings to form pairs for evaluation.")

        # Calculate all pairwise cosine similarities and their true labels
        similarities, labels = self._calculate_cosine_similarities(embeddings, speaker_ids_or_filenames)

        if len(set(labels)) < 2:
            # This happens if all pairs are "same speaker" or all are "different speaker"
            raise ValueError("Not enough 'same speaker' AND 'different speaker' pairs for ROC analysis. "
                             "Ensure your input data contains both types of pairs (i.e., multiple utterances "
                             "from at least one speaker, and utterances from at least two speakers).")

        # Calculate ROC curve components
        fpr, tpr, thresholds = roc_curve(labels, similarities)

        # Calculate Area Under Curve (AUC)
        roc_auc = auc(fpr, tpr)

        # Calculate EER and its threshold
        eer, eer_threshold = self._calculate_eer(fpr, tpr, thresholds)

        results = {
            'eer': eer,
            'eer_threshold': eer_threshold,
            'roc_auc': roc_auc,
            'fpr': fpr,
            'tpr': tpr,
            'thresholds': thresholds
        }
        return results


################################################################################
import numpy as np
import pysptk
from scipy.spatial.distance import euclidean
import librosa # Import librosa for DTW

class MCDMetric:
    """
    Calculates Mel-Cepstral Distortion (MCD) and Log-Spectral Distance (LSD)
    between two batches of mel-spectrograms, including DTW-based MCD.
    Lower MCD/LSD indicates better spectral similarity.
    """
    def __init__(self, n_mfcc: int = 20):
        """
        Initializes the MCDMetric.

        Args:
            n_mfcc (int): Number of Mel-Frequency Cepstral Coefficients (MFCCs)
                          to compute. Typically 12 to 24. Default is 20.
        """
        self.n_mfcc = n_mfcc
        print(f"MCDMetric initialized with n_mfcc={self.n_mfcc}")

    def _mel_to_mcc(self, mel_spec: np.ndarray) -> np.ndarray:
        """
        Converts a single mel-spectrogram to Mel-Cepstral Coefficients (MCCs).

        Args:
            mel_spec (np.ndarray): Input mel-spectrogram of shape (n_mels, n_frames).
                                   Assumed to be in dB scale (log-power).

        Returns:
            np.ndarray: MCCs of shape (n_mfcc, n_frames).
        """
        # Ensure mel_spec is 2D (n_mels, n_frames)
        # This handles cases where a single frame or a single mel-bin is passed
        if mel_spec.ndim == 1:
            mel_spec = mel_spec.reshape(-1, 1) # Treat as single frame if 1D (n_mels,)
        elif mel_spec.ndim > 2:
            # This shouldn't happen if inputs are correctly batch-indexed, but good for robustness
            raise ValueError(f"mel_spec must be 1D or 2D, but got {mel_spec.ndim}D for _mel_to_mcc.")

        # Handle empty mel_spec to prevent errors in pysptk
        if mel_spec.shape[1] == 0: # Check for n_frames == 0
            # Return an empty MCC array with correct feature dimension
            return np.empty((self.n_mfcc, 0), dtype=mel_spec.dtype)

        # Convert from dB to linear power spectrum
        # Adding a small epsilon (1e-10) to avoid issues with log(0)
        # if the mel_spec has very low (near -inf dB) or zero values.
        linear_power_spec = 10**(mel_spec / 10.0) + 1e-10

        # Transpose to (n_frames, n_mels) for pysptk, which expects rows as frames
        # and columns as spectral bins.
        # .copy(order='C') ensures it's C-contiguous, which pysptk prefers.
        # pysptk.sptk.mfcc expects a power spectrum and returns MFCCs.
        # n_mfcc-1 because the 0th coefficient (energy) is usually included,
        # so if you want 20 coefficients, the order is 19.
        mccs = pysptk.sptk.mfcc(linear_power_spec.T.copy(order='C'), order=self.n_mfcc - 1)

        # Transpose back to (n_mfcc, n_frames) for consistency
        return mccs.T

    def calculate_mcd(self, mel_spec_batch1: np.ndarray, mel_spec_batch2: np.ndarray) -> float:
        """
        Calculates the average frame-by-frame MCD between two batches of mel-spectrograms.

        Args:
            mel_spec_batch1 (np.ndarray): First batch of mel-spectrograms (e.g., ground truth/target),
                                          shape (batch_size, n_mels, n_frames).
            mel_spec_batch2 (np.ndarray): Second batch of mel-spectrograms (e.g., model output),
                                          shape (batch_size, n_mels, n_frames).

        Returns:
            float: The average Mel-Cepstral Distortion (MCD) value for the batch.
                   Returns 0.0 if no valid MCDs could be calculated or if inputs are invalid.
        """
        # --- Robustness Checks ---
        if mel_spec_batch1.shape != mel_spec_batch2.shape:
            print(f"Warning (MCD): Input mel-spectrogram batches have different shapes. "
                  f"Batch1: {mel_spec_batch1.shape}, Batch2: {mel_spec_batch2.shape}. Cannot calculate MCD.")
            return 0.0
        if mel_spec_batch1.ndim != 3 or mel_spec_batch2.ndim != 3:
             print(f"Warning (MCD): Input mel-spectrogram batches must be 3D (batch, mels, frames). "
                  f"Batch1: {mel_spec_batch1.ndim}D, Batch2: {mel_spec_batch2.ndim}D. Cannot calculate MCD.")
             return 0.0

        batch_size = mel_spec_batch1.shape[0]
        mcd_values = []

        for b in range(batch_size):
            mel_spec1 = mel_spec_batch1[b]
            mel_spec2 = mel_spec_batch2[b]

            try:
                mcc1 = self._mel_to_mcc(mel_spec1)
                mcc2 = self._mel_to_mcc(mel_spec2)
            except Exception as e:
                print(f"Error converting mel-spec to MCC for sample {b}: {e}. Skipping MCD for this sample.")
                continue

            min_frames = min(mcc1.shape[1], mcc2.shape[1])
            if min_frames == 0:
                print(f"Warning (MCD): No frames to compare for sample {b}. Skipping.")
                continue

            # Truncate to min_frames. Ensures both MCCs have same length for element-wise comparison.
            mcc1 = mcc1[:, :min_frames]
            mcc2 = mcc2[:, :min_frames]

            distances = []
            for i in range(min_frames):
                distances.append(euclidean(mcc1[:, i], mcc2[:, i]))

            if distances: # Check if list is not empty before taking mean
                mcd_values.append(np.mean(distances))

        if not mcd_values:
            # This can happen if all samples in batch failed or had 0 frames
            return 0.0

        return np.mean(mcd_values)

    def calculate_mcd_dtw(self, mel_spec_batch1: np.ndarray, mel_spec_batch2: np.ndarray) -> float:
        """
        Calculates the average DTW-based MCD between two batches of mel-spectrograms.

        Args:
            mel_spec_batch1 (np.ndarray): First batch of mel-spectrograms (e.g., ground truth/target),
                                          shape (batch_size, n_mels, n_frames).
            mel_spec_batch2 (np.ndarray): Second batch of mel-spectrograms (e.g., model output),
                                          shape (batch_size, n_mels, n_frames).

        Returns:
            float: The average DTW-MCD value for the batch.
                   Returns 0.0 if no valid MCDs could be calculated or if inputs are invalid.
        """
        # --- Robustness Checks ---
        if mel_spec_batch1.shape[0] != mel_spec_batch2.shape[0]:
            print(f"Warning (MCD-DTW): Input mel-spectrogram batches have different batch sizes. "
                  f"Batch1: {mel_spec_batch1.shape}, Batch2: {mel_spec_batch2.shape}. Cannot calculate MCD-DTW.")
            return 0.0
        if mel_spec_batch1.ndim != 3 or mel_spec_batch2.ndim != 3:
             print(f"Warning (MCD-DTW): Input mel-spectrogram batches must be 3D (batch, mels, frames). "
                  f"Batch1: {mel_spec_batch1.ndim}D, Batch2: {mel_spec_batch2.ndim}D. Cannot calculate MCD-DTW.")
             return 0.0

        batch_size = mel_spec_batch1.shape[0]
        mcd_dtw_values = []

        for b in range(batch_size):
            mel_spec1 = mel_spec_batch1[b]
            mel_spec2 = mel_spec_batch2[b]

            try:
                mcc1 = self._mel_to_mcc(mel_spec1)
                mcc2 = self._mel_to_mcc(mel_spec2)
            except Exception as e:
                print(f"Error converting mel-spec to MCC for sample {b}: {e}. Skipping MCD-DTW for this sample.")
                continue

            # Ensure there are enough frames for meaningful DTW
            # librosa.sequence.dtw will raise an error if input sequences are too short (e.g., 0 or 1 frame)
            # A common heuristic is to require at least 2 frames for DTW.
            if mcc1.shape[1] < 2 or mcc2.shape[1] < 2:
                print(f"Warning (MCD-DTW): Not enough frames for DTW for sample {b} (MCC1: {mcc1.shape[1]}, MCC2: {mcc2.shape[1]}). Skipping.")
                continue

            # Ensure MCCs are C-contiguous for librosa.dtw, it expects (features, frames)
            # .ascontiguousarray() creates a copy if not already contiguous.
            mcc1_dtw = np.ascontiguousarray(mcc1)
            mcc2_dtw = np.ascontiguousarray(mcc2)

            try:
                # D is the accumulated cost matrix, wp is the warping path
                # metric='euclidean' is the default for librosa.sequence.dtw and appropriate for MCCs.
                D, wp = librosa.sequence.dtw(X=mcc1_dtw, Y=mcc2_dtw, metric='euclidean')

                # The `wp` (warping path) can sometimes be empty if `librosa.sequence.dtw`
                # encounters very problematic inputs (e.g., extremely short, or if max_inst/max_hop
                # constraints make it impossible to find a path, though default should be fine).
                # Check if wp is not empty to avoid division by zero or errors from indexing it.
                if wp.shape[0] == 0:
                    print(f"Warning (MCD-DTW): DTW warping path is empty for sample {b}. Skipping.")
                    continue

                # Calculate the average distance along the optimal warping path.
                # Loop through the aligned frame pairs (r, c) from the warping path `wp`.
                # For each pair, calculate the Euclidean distance between the corresponding MCC vectors.
                path_distances = []
                for r, c in wp: # wp contains (row_idx_X, col_idx_Y) pairs for aligned points
                    path_distances.append(euclidean(mcc1_dtw[:, r], mcc2_dtw[:, c]))

                if path_distances: # Ensure there are distances before taking mean
                    mcd_dtw_values.append(np.mean(path_distances))
                else:
                    # This case should ideally be caught by wp.shape[0] == 0 check, but a fallback.
                    print(f"Warning (MCD-DTW): Calculated path_distances list is empty for sample {b}. Skipping.")

            except Exception as e:
                print(f"Error calculating DTW for sample {b}: {e}. Skipping.")
                continue

        if not mcd_dtw_values:
            # This can happen if all samples in batch failed or had insufficient frames
            print("Warning: No valid MCD-DTW values calculated for the entire batch. Returning 0.0.")
            return 0.0

        return np.mean(mcd_dtw_values)

    def calculate_lsd(self, mel_spec_batch1: np.ndarray, mel_spec_batch2: np.ndarray) -> float:
        """
        Calculates the average Log-Spectral Distance (LSD) between two batches of mel-spectrograms.

        Args:
            mel_spec_batch1 (np.ndarray): First batch of mel-spectrograms (e.g., ground truth/target),
                                          shape (batch_size, n_mels, n_frames).
            mel_spec_batch2 (np.ndarray): Second batch of mel-spectrograms (e.g., model output),
                                          shape (batch_size, n_mels, n_frames).

        Returns:
            float: The average Log-Spectral Distance (LSD) value for the batch.
                   Returns 0.0 if no valid LSDs could be calculated or if inputs are invalid.
        """
        # --- Robustness Checks ---
        if mel_spec_batch1.shape != mel_spec_batch2.shape:
            print(f"Warning (LSD): Input mel-spectrogram batches have different shapes. "
                  f"Batch1: {mel_spec_batch1.shape}, Batch2: {mel_spec_batch2.shape}. Cannot calculate LSD.")
            return 0.0
        if mel_spec_batch1.ndim != 3 or mel_spec_batch2.ndim != 3:
             print(f"Warning (LSD): Input mel-spectrogram batches must be 3D (batch, mels, frames). "
                  f"Batch1: {mel_spec_batch1.ndim}D, Batch2: {mel_spec_batch2.ndim}D. Cannot calculate LSD.")
             return 0.0

        batch_size = mel_spec_batch1.shape[0]
        lsd_values = []

        for b in range(batch_size):
            mel_spec1 = mel_spec_batch1[b]
            mel_spec2 = mel_spec_batch2[b]

            # Convert to (n_mels, n_frames) if needed (already handled by batch indexing)
            # Ensure inputs are treated as log-power spectra (dB scale).
            # If your mel_spec is already in dB, no explicit conversion needed here.
            # Assuming mel_spec_batch1 and mel_spec_batch2 are already in dB scale.
            log_mel_spec1 = mel_spec1
            log_mel_spec2 = mel_spec2

            # Handle empty mel_spec to prevent errors
            if log_mel_spec1.shape[1] == 0 or log_mel_spec2.shape[1] == 0:
                print(f"Warning (LSD): No frames to compare for sample {b}. Skipping.")
                continue

            try:
                # Ensure both log-mel-spectrograms have the same number of frames
                min_frames = min(log_mel_spec1.shape[1], log_mel_spec2.shape[1])

                # If there are no common frames, skip this sample
                if min_frames == 0:
                    print(f"Warning (LSD): No common frames for sample {b}. Skipping.")
                    continue

                truncated_log_mel_spec1 = log_mel_spec1[:, :min_frames]
                truncated_log_mel_spec2 = log_mel_spec2[:, :min_frames]

                # Calculate the difference between the two log-mel-spectrograms
                diff = truncated_log_mel_spec1 - truncated_log_mel_spec2

                # Square the differences
                squared_diff = np.square(diff)

                # Sum over frequency bins (axis=0 for n_mels) for each frame
                sum_squared_diff_per_frame = np.sum(squared_diff, axis=0)

                # Take the square root of the sum_squared_diff_per_frame (this is the Euclidean distance per frame)
                # Then average over frames to get the LSD for this sample
                sample_lsd = np.mean(np.sqrt(sum_squared_diff_per_frame))
                lsd_values.append(sample_lsd)

            except Exception as e:
                print(f"Error calculating LSD for sample {b}: {e}. Skipping.")
                continue

        if not lsd_values:
            print("Warning: No valid LSD values calculated for the entire batch. Returning 0.0.")
            return 0.0

        return np.mean(lsd_values)

################################################################################
# new version
from numpy import random
import numpy as np
import scipy.signal # Import scipy for resampling
# Assuming 'pesq' and 'stoi' are imported from their respective libraries
from pesq import pesq
from pystoi.stoi import stoi

class VoiceQualityEvaluator:
    """
    A class to evaluate speech quality using PESQ and STOI metrics.
    It directly accepts NumPy arrays for audio data and handles resampling
    for PESQ if the input sample rate is different from the target.
    """

    def __init__(self, pesq_sample_rate: int = 16000):
        """
        Initializes the VoiceQualityEvaluator.

        Args:
            pesq_sample_rate (int): The sample rate that PESQ will use (typically 8kHz or 16kHz).
                                      Input audio will be resampled to this rate if necessary for PESQ.
        """
        self.pesq_sample_rate = pesq_sample_rate

    def _resample_audio(self, audio_array: np.ndarray, original_sr: int) -> np.ndarray:
        """
        Helper method to resample an audio array to the pesq_sample_rate.

        Args:
            audio_array (np.ndarray): The input audio array.
            original_sr (int): The original sample rate of the audio_array.

        Returns:
            np.ndarray: The resampled audio array.
        """
        if original_sr == self.pesq_sample_rate:
            return audio_array

        # Calculate the number of samples in the resampled audio
        num_samples_resampled = int(len(audio_array) * (self.pesq_sample_rate / original_sr))

        # Perform resampling using scipy.signal.resample
        resampled_audio = scipy.signal.resample(audio_array, num_samples_resampled)
        return resampled_audio

    def calculate_pesq(self, reference_audio: np.ndarray, degraded_audio: np.ndarray, original_sr: int) -> float:
        """
        Calculates the Perceptual Evaluation of Speech Quality (PESQ) score.
        Resamples audio to self.pesq_sample_rate if original_sr is different.

        Args:
            reference_audio (np.ndarray): The clean, original reference audio as a 1D NumPy array.
            degraded_audio (np.ndarray): The converted/degraded audio as a 1D NumPy array.
            original_sr (int): The original sample rate of both reference_audio and degraded_audio.

        Returns:
            float: The PESQ score. Returns NaN if calculation fails.
        """
        try:
            # Resample if needed for PESQ
            if original_sr != self.pesq_sample_rate:
                resampled_ref = self._resample_audio(reference_audio, original_sr)
                resampled_deg = self._resample_audio(degraded_audio, original_sr)
            else:
                resampled_ref = reference_audio
                resampled_deg = degraded_audio

            # Ensure both arrays have the same length for PESQ calculation
            min_len = min(len(resampled_ref), len(resampled_deg))
            ref_audio_clipped = resampled_ref[:min_len]
            deg_audio_clipped = resampled_deg[:min_len]

            # PESQ mode based on the target sample rate
            mode = 'wb' if self.pesq_sample_rate == 16000 else 'nb'
            score = pesq(self.pesq_sample_rate, ref_audio_clipped, deg_audio_clipped, mode)
            return score
        except Exception as e:
            # print(f"PESQ calculation failed: {e}") # Uncomment for debugging
            return np.nan

    def calculate_stoi(self, reference_audio: np.ndarray, degraded_audio: np.ndarray, original_sr: int) -> float:
        """
        Calculates the Short-Time Objective Intelligibility (STOI) score.
        STOI typically accepts various sample rates, so no internal resampling for STOI.

        Args:
            reference_audio (np.ndarray): The clean, original reference audio as a 1D NumPy array.
            degraded_audio (np.ndarray): The converted/degraded audio as a 1D NumPy array.
            original_sr (int): The original sample rate of both reference_audio and degraded_audio.

        Returns:
            float: The STOI score. Returns NaN if calculation fails.
        """
        try:
            # Ensure both arrays have the same length for STOI calculation
            min_len = min(len(reference_audio), len(degraded_audio))
            ref_audio_clipped = reference_audio[:min_len]
            deg_audio_clipped = degraded_audio[:min_len]

            # STOI accepts the original_sr directly
            score = stoi(ref_audio_clipped, deg_audio_clipped, original_sr, extended=False)
            return score
        except Exception as e:
            # print(f"STOI calculation failed: {e}") # Uncomment for debugging
            return np.nan

    def calculate_all_quality_metrics(self, reference_audio: np.ndarray, degraded_audio: np.ndarray, original_sr: int) -> tuple[float, float]:
        """
        Calculates PESQ and STOI scores for given reference and degraded audio arrays.
        Handles resampling for PESQ if original_sr differs from the pesq_sample_rate.

        Args:
            reference_audio (np.ndarray): The clean, original reference audio as a 1D NumPy array.
            degraded_audio (np.ndarray): The converted/degraded audio as a 1D NumPy array.
            original_sr (int): The original sample rate of both input audio arrays.

        Returns:
            tuple[float, float]: A tuple containing (pesq_score, stoi_score).
                                 Values will be np.nan if calculation fails.
        """
        # Pass original_sr to individual calculation methods
        pesq_score = self.calculate_pesq(reference_audio, degraded_audio, original_sr)
        stoi_score = self.calculate_stoi(reference_audio, degraded_audio, original_sr)

        return pesq_score, stoi_score



#################################################################################
import torch
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate # For WER and CER metrics
from typing import List, Union, Tuple, Optional

import torch
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate # For WER and CER metrics
from typing import List, Union, Tuple, Optional

class ASRWhisperEvaluator2:
    """
    A class to perform Automatic Speech Recognition (ASR) inference using
    Hugging Face's Whisper model and evaluate its performance (WER, CER).

    It can take either mel spectrograms or raw audio waveforms as input.
    """
    def __init__(self, model_name: str = "openai/whisper-tiny", device: Optional[torch.device] = None):
        """
        Initializes the ASRWhisperEvaluator.

        Args:
            model_name (str): The name of the Whisper model to load from Hugging Face.
                              Defaults to "openai/whisper-tiny".
            device (Optional[torch.device]): The device to run the model on (e.g., 'cuda', 'cpu').
                                             If None, it will automatically detect CUDA if available, else CPU.
        """
        self.device = device if device else (torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        print(f"Initializing ASRWhisperEvaluator on device: {self.device}")

        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name).to(self.device)
        self.model.eval() # Set model to evaluation mode

        self.wer_metric = evaluate.load("wer")
        self.cer_metric = evaluate.load("cer")

        # Fixed parameters for Whisper's mel spectrograms
        self.whisper_mel_bins: int = self._get_whisper_mel_bins()
        self.whisper_target_frames: int = 3000 # Corresponds to 30 seconds of audio

        print(f"Whisper model '{model_name}' loaded successfully.")
        print(f"Expected mel bins for Whisper: {self.whisper_mel_bins}")
        print(f"Expected target frames for Whisper: {self.whisper_target_frames}")


    def _get_whisper_mel_bins(self) -> int:
        """Helper to robustly get the number of mel bins expected by Whisper."""
        if hasattr(self.processor.feature_extractor, 'n_mels'):
            return self.processor.feature_extractor.n_mels
        elif hasattr(self.processor.feature_extractor, 'n_mel_filters'):
            return self.processor.feature_extractor.n_mel_filters
        else:
            # Fallback to default for Whisper if attributes are not found
            print("Warning: Could not determine n_mels from feature_extractor. Defaulting to 80.")
            return 80

    def _prepare_mel_input(self, mel_tensor: torch.Tensor) -> torch.Tensor:
        """
        Prepares a single mel spectrogram tensor for Whisper inference.
        Pads or truncates to Whisper's expected target_frames (3000).

        Args:
            mel_tensor (torch.Tensor): A mel spectrogram tensor of shape (n_mel_bins, n_frames).

        Returns:
            torch.Tensor: The prepared mel spectrogram tensor, padded to (whisper_mel_bins, whisper_target_frames).
        """
        if mel_tensor.shape[0] != self.whisper_mel_bins:
            raise ValueError(
                f"Input mel spectrogram has {mel_tensor.shape[0]} mel bins, "
                f"but Whisper expects {self.whisper_mel_bins}. "
                "Please ensure your MelDecoder outputs the correct number of mel bins."
            )

        # Create a padded tensor with the target length
        padded_mel = torch.zeros(
            self.whisper_mel_bins,
            self.whisper_target_frames,
            dtype=mel_tensor.dtype
        )

        # Place the input frames into the padded tensor
        num_frames_to_copy = min(mel_tensor.shape[1], self.whisper_target_frames)
        padded_mel[:, :num_frames_to_copy] = mel_tensor[:, :num_frames_to_copy]

        return padded_mel.float()

    def transcribe(self,
                   inputs: Union[torch.Tensor, List[torch.Tensor], List[np.ndarray]],
                   input_type: str = "mel",
                   sampling_rate: Optional[int] = 16000,
                   language: Optional[str] = None) -> List[str]:
        """
        Performs ASR transcription on a batch of inputs.

        Args:
            inputs (Union[torch.Tensor, List[torch.Tensor], List[np.ndarray]]):
                Input data.
                If `input_type` is "mel":
                    - A single torch.Tensor of shape (batch_size, n_mel_bins, n_frames)
                      OR (n_mel_bins, n_frames) if batch_size is 1.
                    - A list of torch.Tensor, where each item is of shape (n_mel_bins, n_frames).
                If `input_type` is "wave", a list of numpy.ndarray
                representing raw audio waveforms (mono, float32).
            input_type (str): Specifies the type of input. Must be "mel" or "wave".
                              Defaults to "mel".
            sampling_rate (Optional[int]): Required if `input_type` is "wave".
                                           The sampling rate of the audio waveforms.
                                           Defaults to 16000 Hz for Whisper.
            language (Optional[str]): The language to force the transcription to. E.g., "en" for English.
                                      If None, Whisper will attempt to detect the language.
                                      Use ISO 639-1 language codes (e.g., "en", "fr", "de").

        Returns:
            List[str]: A list of transcribed text strings.
        """
        if inputs is None or (isinstance(inputs, list) and not inputs):
            return []

        if input_type not in ["mel", "wave"]:
            raise ValueError("input_type must be 'mel' or 'wave'.")

        if input_type == "mel":
            mels_to_process: List[torch.Tensor] = []
            if isinstance(inputs, torch.Tensor):
                # Handle single mel (2D) or batched mels (3D)
                if inputs.dim() == 2: # Single mel (n_mel_bins, n_frames)
                    mels_to_process.append(inputs)
                elif inputs.dim() == 3: # Batched mels (batch_size, n_mel_bins, n_frames)
                    # Iterate through the batch and append each individual mel
                    for i in range(inputs.shape[0]):
                        mels_to_process.append(inputs[i])
                else:
                    raise ValueError(
                        f"If input_type is 'mel' and inputs is a single tensor, "
                        f"it must have 2 or 3 dimensions (n_mel_bins, n_frames) or "
                        f"(batch_size, n_mel_bins, n_frames), "
                        f"but got {inputs.dim()} dimensions."
                    )
            elif isinstance(inputs, list):
                for mel in inputs:
                    if not isinstance(mel, torch.Tensor):
                        raise TypeError(f"Expected torch.Tensor for mel input in list, got {type(mel)}")
                    if mel.dim() != 2: # Ensure elements in list are 2D mels
                         raise ValueError(f"Each mel in the input list must be a 2D tensor (n_mel_bins, n_frames), but got {mel.dim()} dimensions.")
                    mels_to_process.append(mel)
            else:
                raise TypeError(
                    f"For input_type 'mel', inputs must be a torch.Tensor or List[torch.Tensor], "
                    f"but got {type(inputs)}."
                )

            # Apply _prepare_mel_input to each mel and stack them
            processed_inputs = [self._prepare_mel_input(mel) for mel in mels_to_process]
            input_features_batch = torch.stack(processed_inputs).to(self.device)
        else: # input_type == "wave"
            if sampling_rate is None:
                raise ValueError("sampling_rate must be provided for 'wave' input_type.")
            if not isinstance(inputs, list) or not all(isinstance(wave, np.ndarray) for wave in inputs):
                 raise TypeError(f"Expected List[np.ndarray] for wave input, got {type(inputs)}")

            # Processor handles the feature extraction for raw audio
            # Note: For batching raw audio, `inputs` should be a list of np.ndarray
            # The processor pads them to the longest sample in the batch.
            input_features_batch = self.processor(
                inputs,
                sampling_rate=sampling_rate,
                return_tensors="pt"
            ).input_features.to(self.device)

        # Prepare generation arguments
        generate_kwargs = {}
        if language:
            # The correct way to get forced_decoder_ids for a specific language
            # This method generates the required token IDs for forcing language and task.
            forced_decoder_ids = self.processor.get_decoder_prompt_ids(
                language=language,
                task="transcribe", # Specify the task as transcription
                no_timestamps=True # Typically desired for clean text output
            )
            generate_kwargs["forced_decoder_ids"] = forced_decoder_ids

        with torch.no_grad():
            predicted_ids = self.model.generate(input_features_batch, **generate_kwargs)

        transcriptions = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)
        return transcriptions

    def evaluate(self,
                 predictions: List[str],
                 references: List[str]) -> Tuple[float, float]:
        """
        Calculates Word Error Rate (WER) and Character Error Rate (CER).

        Args:
            predictions (List[str]): A list of transcribed text strings.
            references (List[str]): A list of ground truth text strings.

        Returns:
            Tuple[float, float]: A tuple containing (WER, CER).
        """
        if len(predictions) != len(references):
            raise ValueError("The number of predictions and references must be the same.")

        wer_score = self.wer_metric.compute(predictions=predictions, references=references)
        cer_score = self.cer_metric.compute(predictions=predictions, references=references)

        return wer_score, cer_score


################################################################################
import torch
import numpy as np
from transformers import AutoProcessor, AutoModelForCTC # More generic imports
import evaluate
from typing import List, Union, Tuple, Optional

class ASRWav2Vec2Evaluator:
    """
    A class to perform Automatic Speech Recognition (ASR) inference using
    Hugging Face's Wav2Vec2 model and evaluate its performance (WER, CER).

    It takes raw audio waveforms as input.
    """
    def __init__(self, model_name: str = "facebook/wav2vec2-base-960h", device: Optional[torch.device] = None):
        """
        Initializes the ASRWav2Vec2Evaluator.

        Args:
            model_name (str): The name of the Wav2Vec2 model to load from Hugging Face.
                              Defaults to "facebook/wav2vec2-base-960h" (a common pre-trained English model).
            device (Optional[torch.device]): The device to run the model on (e.g., 'cuda', 'cpu').
                                             If None, it will automatically detect CUDA if available, else CPU.
        """
        self.device = device if device else (torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        print(f"Initializing ASRWav2Vec2Evaluator on device: {self.device}")

        self.processor = AutoProcessor.from_pretrained(model_name)
        # Wav2Vec2 models typically use AutoModelForCTC for ASR
        self.model = AutoModelForCTC.from_pretrained(model_name).to(self.device)
        self.model.eval() # Set model to evaluation mode

        self.wer_metric = evaluate.load("wer")
        self.cer_metric = evaluate.load("cer")

        # Get expected sampling rate from the feature extractor
        self.expected_sampling_rate = self.processor.feature_extractor.sampling_rate

        print(f"Wav2Vec2 model '{model_name}' loaded successfully.")
        print(f"Expected sampling rate for Wav2Vec2: {self.expected_sampling_rate} Hz")


    def transcribe(self,
                   audio_inputs: List[np.ndarray],
                   sampling_rate: Optional[int] = None) -> List[str]:
        """
        Performs ASR transcription on a batch of raw audio waveforms.

        Args:
            audio_inputs (List[np.ndarray]): A list of numpy.ndarray, where each array
                                             represents a raw audio waveform (mono, float32).
            sampling_rate (Optional[int]): The sampling rate of the audio waveforms.
                                           If None, defaults to the model's expected sampling rate.
                                           It's crucial this matches the audio.

        Returns:
            List[str]: A list of transcribed text strings.
        """
        if not audio_inputs:
            return []

        # Ensure all inputs are numpy arrays
        if not all(isinstance(wave, np.ndarray) for wave in audio_inputs):
             raise TypeError(f"Expected numpy.ndarray for audio_inputs, got {type(audio_inputs[0])}")

        # Use the model's expected sampling rate if not provided
        if sampling_rate is None:
            sampling_rate = self.expected_sampling_rate
            print(f"Using default sampling rate: {sampling_rate} Hz")
        elif sampling_rate != self.expected_sampling_rate:
            print(f"Warning: Input sampling rate ({sampling_rate} Hz) does not match model's expected "
                  f"sampling rate ({self.expected_sampling_rate} Hz). Resampling will occur if needed "
                  f"by the processor, but it's best to match directly.")

        # Processor handles all the necessary feature extraction for raw audio
        # It will internally convert to features (like log-mel for Whisper, or learned features for Wav2Vec2)
        input_features_batch = self.processor(
            audio_inputs,
            sampling_rate=sampling_rate,
            return_tensors="pt",
            padding=True # Important for batching variable-length audio
        ).input_values.to(self.device) # Wav2Vec2 uses 'input_values' instead of 'input_features'

        with torch.no_grad():
            # For CTC models, `logits` are typically returned, then greedy decoded.
            # However, `generate` method also works if the model has a text generation head.
            # For AutoModelForCTC, it's often more direct to get logits and then decode:
            logits = self.model(input_features_batch).logits
            predicted_ids = torch.argmax(logits, dim=-1)

        # Decode the predicted IDs to text using the processor's tokenizer
        transcriptions = self.processor.batch_decode(predicted_ids)
        return transcriptions


    def evaluate(self,
                 predictions: List[str],
                 references: List[str]) -> Tuple[float, float]:
        """
        Calculates Word Error Rate (WER) and Character Error Rate (CER).
        (Same as in the Whisper evaluator)
        """
        if len(predictions) != len(references):
            raise ValueError("The number of predictions and references must be the same.")

        wer_score = self.wer_metric.compute(predictions=predictions, references=references)
        cer_score = self.cer_metric.compute(predictions=predictions, references=references)

        return wer_score, cer_score


**------------------------------------------------**

**------> 9-Information Losses (Inter-Intra) <-------**

**------------------------------------------------**

In [23]:
import torch
import torch.nn.functional as F

class DisentanglementLossCalculator:
    """
    A class to compute losses that encourage disentanglement in hierarchical latent variables.
    - Inter-level disentanglement (between z1, z2, etc.) using Mutual Information.
    - Intra-level disentanglement (within the dimensions of z1) using Distance Correlation.
    """
    def __init__(self, n_levels, mi_weight=1.0, dcor_weight=1.0):
        """
        Args:
            n_levels (int): The number of hierarchical latent levels (e.g., 3 for z1, z2, z3).
            mi_weight (float): Weight for the mutual information loss (inter-level).
            dcor_weight (float): Weight for the distance correlation loss (intra-level).
        """
        if not isinstance(n_levels, int) or n_levels < 1:
            raise ValueError(f"n_levels must be an integer >= 1, but got {n_levels}")

        self.n_levels = n_levels
        self.mi_weight = mi_weight
        self.dcor_weight = dcor_weight

        # Pre-calculate the number of pairs for MI loss normalization
        if self.n_levels > 1:
            self.num_mi_pairs = self.n_levels * (self.n_levels - 1) // 2
        else:
            self.num_mi_pairs = 0

        print("DisentanglementLossCalculator initialized.")
        print(f"  - Configured for {self.n_levels} hierarchical levels.")
        print(f"  - Mutual Information (inter-level) weight: {self.mi_weight}")
        print(f"  - Distance Correlation (intra-level) weight: {self.dcor_weight}")


    def _log_sum_exp(self, x, dim=0):
        """Numerically stable log-sum-exp."""
        max_x = torch.max(x, dim=dim, keepdim=True)[0]
        return max_x + torch.log(torch.sum(torch.exp(x - max_x), dim=dim, keepdim=True))

    def estimate_mutual_information(self, z_i, z_j):
        """
        Estimates the mutual information I(z_i; z_j) using a Jensen-Shannon Divergence estimator.
        """
        z_j_shuffled = z_j.detach()[torch.randperm(z_j.size(0))]

        joint_samples = torch.cat([z_i, z_j], dim=-1)
        marginal_samples = torch.cat([z_i, z_j_shuffled], dim=-1)

        # Simple linear discriminator
        t = torch.randn(joint_samples.size(1), 1, device=z_i.device)
        joint_score = joint_samples @ t
        marginal_score = marginal_samples @ t

        # JS-based estimator
        mi_estimate = torch.mean(F.softplus(joint_score)) + torch.mean(F.softplus(-marginal_score))
        return mi_estimate

    def inter_level_disentanglement_loss(self, speaker_latents):
        """
        Calculates the total mutual information loss between all pairs of latent levels.
        """
        if self.n_levels < 2:
            return 0.0 # No inter-level loss is possible

        mus = speaker_latents['mus']
        log_sigmas = speaker_latents['log_sigmas']

        # --- Assertion for safety ---
        # This ensures the model output matches the configured number of levels.
        if len(mus) != self.n_levels:
            raise ValueError(
                f"Mismatch: DisentanglementLossCalculator initialized with n_levels={self.n_levels}, "
                f"but received {len(mus)} levels from the speaker encoder."
            )

        # Sample from the posteriors to get concrete latent vectors
        sampled_latents = []
        for mu, log_sigma in zip(mus, log_sigmas):
            eps = torch.randn_like(mu)
            z = mu + torch.exp(log_sigma * 0.5) * eps
            sampled_latents.append(z)

        total_mi_loss = 0.0
        for i in range(self.n_levels):
            for j in range(i + 1, self.n_levels):
                mi = self.estimate_mutual_information(sampled_latents[i], sampled_latents[j])
                total_mi_loss += mi

        # Average the loss over the number of pairs for stability
        return self.mi_weight * (total_mi_loss / self.num_mi_pairs)

    def _distance_covariance(self, x, y):
        """
        Computes the distance covariance.
        This version is robust against floating point errors that could make
        the result slightly negative (which would cause `sqrt` to return NaN).
        """
        n = x.size(0)
        # Handle the edge case where the batch size is 1, as cdist will fail.
        if n < 2:
            return torch.tensor(0.0, device=x.device)

        a = torch.cdist(x, x)
        b = torch.cdist(y, y)

        # Double centering of the distance matrices
        A = a - a.mean(dim=0, keepdim=True) - a.mean(dim=1, keepdim=True) + a.mean()
        B = b - b.mean(dim=0, keepdim=True) - b.mean(dim=1, keepdim=True) + b.mean()

        d_cov_sq_unclamped = (A * B).sum() / (n * n)

        # --- THE DEFINITIVE FIX IS HERE ---
        # Clamp the value to be non-negative before the square root.
        # This prevents NaN from sqrt due to floating-point inaccuracies
        # where a value that should be 0.0 becomes -1e-9.
        d_cov_sq = torch.clamp(d_cov_sq_unclamped, min=0.0)

        # We still add a small epsilon for stability in case the result is exactly 0.
        return torch.sqrt(d_cov_sq + 1e-10)

    def _distance_correlation(self, x, y):
        """
        Computes the distance correlation between two tensors.
        This version includes an additional epsilon in the denominator for
        enhanced numerical stability to prevent NaN results.
        """
        d_cov_xy = self._distance_covariance(x, y)
        d_var_x = torch.sqrt(self._distance_covariance(x, x))
        d_var_y = torch.sqrt(self._distance_covariance(y, y))

        # --- THE FIX IS HERE ---
        # Add epsilon directly to the denominator before division.
        # This is a very common pattern for numerical stability.
        epsilon = 1e-8
        denominator = d_var_x * d_var_y + epsilon

        return d_cov_xy / denominator

    def intra_level_disentanglement_loss(self, speaker_latents, target_level=0):
        """
        (Bonus) Encourages dimensions within a single latent vector to be uncorrelated.
        """
        if target_level >= self.n_levels:
            raise ValueError(
                f"target_level={target_level} is out of bounds. "
                f"Model has {self.n_levels} levels (indexed 0 to {self.n_levels-1})."
            )

        mu = speaker_latents['mus'][target_level]
        log_sigma = speaker_latents['log_sigmas'][target_level]

        eps = torch.randn_like(mu)
        z = mu + torch.exp(log_sigma * 0.5) * eps

        independent_noise = torch.randn_like(z)
        dcor = self._distance_correlation(z.unsqueeze(-1), independent_noise.unsqueeze(-1))

        return self.dcor_weight * dcor

**------------------------------------------------------------------------**

**--------------- 10-Cross Encoder Information Loss ---------------**

**------------------------------------------------------------------------**

In [ ]:
import torch
import torch.nn.functional as F

class CrossEncoderMutualInformationLoss:
    """
    Calculates and penalizes mutual information between speaker and content latent variables
    across all specified hierarchical levels. This encourages strong disentanglement
    between the two encoder pathways (cross-encoder disentanglement).
    """
    def __init__(self,
                 speaker_n_levels: int,
                 content_n_levels: int,
                 mi_weight: float = 1.0,
                 use_post_sample: bool = True):
        """
        Initializes the CrossEncoderMutualInformationLoss.

        Args:
            speaker_n_levels (int): The number of hierarchical levels in the speaker encoder.
                                    This should correspond to `len(speaker_latents['mus'])`.
            content_n_levels (int): The number of hierarchical levels in the content encoder.
                                    This should correspond to `len(content_latents['mus'])`.
            mi_weight (float): Weight for this mutual information loss. Defaults to 1.0.
            use_post_sample (bool): If True, samples from the posterior distributions (mu + sigma * epsilon)
                                    for MI estimation. If False, it uses only the mean (mu).
                                    Defaults to True (recommended for VAE-like models).
        """
        if not isinstance(speaker_n_levels, int) or speaker_n_levels < 1:
            raise ValueError(f"speaker_n_levels must be an integer >= 1, but got {speaker_n_levels}")
        if not isinstance(content_n_levels, int) or content_n_levels < 1:
            raise ValueError(f"content_n_levels must be an integer >= 1, but got {content_n_levels}")

        self.speaker_n_levels = speaker_n_levels
        self.content_n_levels = content_n_levels
        self.mi_weight = mi_weight
        self.use_post_sample = use_post_sample

        # Calculate the total number of unique pairs across which MI will be estimated.
        self.num_mi_pairs = self.speaker_n_levels * self.content_n_levels
        if self.num_mi_pairs == 0:
             # This should not be reachable if levels >= 1, but as a safeguard.
            self.num_mi_pairs = 1 # Avoid division by zero in case of unexpected 0 levels.

        print("CrossEncoderMutualInformationLoss initialized.")
        print(f"  - Configured for {self.speaker_n_levels} speaker levels and {self.content_n_levels} content levels.")
        print(f"  - Mutual Information (cross-encoder) weight: {self.mi_weight}")
        print(f"  - Will estimate MI across {self.num_mi_pairs} total pairs and average the loss.")


    def estimate_mutual_information(self, z_a: torch.Tensor, z_b: torch.Tensor) -> torch.Tensor:
        """
        Estimates the mutual information between two batches of latent vectors (z_a and z_b)
        using a random projection and a Jensen-Shannon divergence (JS-divergence) based estimator.
        This is the same method as in your `DisentanglementLossCalculator`.

        Args:
            z_a (torch.Tensor): A batch of latent vectors from the first source. Shape: (batch_size, dim_a).
            z_b (torch.Tensor): A batch of latent vectors from the second source. Shape: (batch_size, dim_b).

        Returns:
            torch.Tensor: A scalar tensor representing the estimated mutual information.
                          This value is positive and should be minimized during training.
        """
        # Ensure z_b is detached to prevent gradients flowing through the shuffled batch,
        # which is crucial for the InfoNCE/JS-divergence style estimator.
        z_b_shuffled = z_b.detach()[torch.randperm(z_b.size(0))]

        # Concatenate z_a and z_b to form "joint" samples.
        joint_samples = torch.cat([z_a, z_b], dim=-1) # Shape: (batch_size, dim_a + dim_b)

        # Concatenate z_a with *shuffled* z_b to form "marginal" samples.
        marginal_samples = torch.cat([z_a, z_b_shuffled], dim=-1) # Shape: (batch_size, dim_a + dim_b)

        # Create a random projection vector. This projects the high-dimensional
        # samples onto a 1D line for simplified MI estimation.
        t = torch.randn(joint_samples.size(1), 1, device=z_a.device)

        # Project the joint and marginal samples.
        joint_score = joint_samples @ t   # Shape: (batch_size, 1)
        marginal_score = marginal_samples @ t # Shape: (batch_size, 1)

        # Calculate the JS-divergence based MI estimate.
        # F.softplus(x) = log(1 + exp(x)).
        mi_estimate = torch.mean(F.softplus(joint_score)) + torch.mean(F.softplus(-marginal_score))
        return mi_estimate

    def __call__(self, speaker_latents: dict, content_latents: dict) -> torch.Tensor:
        """
        Calculates the total weighted mutual information loss across all possible pairs
        of speaker and content latent levels.

        Args:
            speaker_latents (dict): A dictionary containing speaker latent information.
                                    Expected to have 'mus' and 'log_sigmas' keys,
                                    each holding a list/tuple of tensors for each level.
            content_latents (dict): A dictionary containing content latent information,
                                    structured identically to speaker_latents.

        Returns:
            torch.Tensor: A scalar tensor representing the total weighted and averaged
                          mutual information loss across all cross-encoder level pairs.
                          This value should be added to your total training loss.
        """
        # Runtime assertions to ensure the actual data matches initialization config.
        if len(speaker_latents['mus']) != self.speaker_n_levels:
            raise ValueError(
                f"Mismatch: CrossEncoderMutualInformationLoss initialized with speaker_n_levels={self.speaker_n_levels}, "
                f"but received {len(speaker_latents['mus'])} levels in `speaker_latents['mus']`."
            )
        if len(content_latents['mus']) != self.content_n_levels:
            raise ValueError(
                f"Mismatch: CrossEncoderMutualInformationLoss initialized with content_n_levels={self.content_n_levels}, "
                f"but received {len(content_latents['mus'])} levels in `content_latents['mus']`."
            )

        total_mi_loss = 0.0

        # Iterate through all speaker levels (i) and all content levels (j)
        for i in range(self.speaker_n_levels):
            for j in range(self.content_n_levels):
                # Extract the means and log-sigmas for the current (speaker_level, content_level) pair
                mu_spk_i = speaker_latents['mus'][i]
                log_sigma_spk_i = speaker_latents['log_sigmas'][i]
                mu_con_j = content_latents['mus'][j]
                log_sigma_con_j = content_latents['log_sigmas'][j]

                # Obtain the latent vectors (either sampled from posterior or just the mean)
                if self.use_post_sample:
                    z_spk_i = mu_spk_i + torch.exp(log_sigma_spk_i * 0.5) * torch.randn_like(mu_spk_i)
                    z_con_j = mu_con_j + torch.exp(log_sigma_con_j * 0.5) * torch.randn_like(mu_con_j)
                else:
                    z_spk_i = mu_spk_i
                    z_con_j = mu_con_j

                # Estimate MI for this specific pair and add to the total.
                mi_pair = self.estimate_mutual_information(z_spk_i, z_con_j)
                total_mi_loss += mi_pair

        # Return the weighted average MI loss across all pairs.
        return self.mi_weight * (total_mi_loss / self.num_mi_pairs)

**---------------------------------------------------------------------------------------**

11-Other information losses (SpeakerIdentity, Transcription, SpeakerAttribute)

**---------------------------------------------------------------------------------------**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def _calculate_info_nce(query: torch.Tensor,
                        key: torch.Tensor,
                        temperature: float = 0.2) -> torch.Tensor:
    """
    Calculates the InfoNCE loss between queries and keys.
    Assumes query and key batches are aligned (query[i] corresponds to key[i]).
    Negative samples are all other keys in the batch.

    Args:
        query (torch.Tensor): Tensor of query embeddings (batch_size, embedding_dim).
        key (torch.Tensor): Tensor of key embeddings (batch_size, embedding_dim).
        temperature (float): Temperature scalar for scaling logits. A smaller temperature
                             makes the model more sensitive to small differences and focuses
                             on harder negatives. Typical values are 0.05 to 0.2.

    Returns:
        torch.Tensor: Scalar InfoNCE loss. A lower loss indicates higher mutual information.
                      This loss should be minimized during training.
    """
    # Normalize embeddings for cosine similarity
    query = F.normalize(query, dim=-1)
    key = F.normalize(key, dim=-1)

    # Compute dot product similarities across all query-key pairs in the batch.
    # The result is a (batch_size, batch_size) matrix, where the diagonal elements
    # correspond to positive pairs (query_i, key_i) and off-diagonal elements
    # correspond to negative pairs (query_i, key_j for j != i).
    logits = torch.matmul(query, key.transpose(0, 1)) / temperature

    # Create labels for the positive samples. For each query_i, the positive key
    # is key_i, which corresponds to the i-th column in the logits matrix.
    labels = torch.arange(logits.size(0), device=query.device)

    # Compute cross-entropy loss. InfoNCE is essentially a form of cross-entropy
    # where the goal is to correctly classify the positive pair among all
    # negative pairs in the batch. Minimizing this negative log-likelihood
    # maximizes the mutual information.
    loss = F.cross_entropy(logits, labels)
    return loss

In [ ]:
class SpeakerIdentityInformationLoss(nn.Module):
    """
    Increases mutual information between speaker latent variables and speaker identity.
    Uses InfoNCE loss. This encourages the speaker latents to explicitly encode
    speaker-specific information.
    """
    def __init__(self,
                 num_unique_speakers: int, # Renamed for clarity: reflects unique speaker IDs (e.g., 'p329')
                 latent_dim: int, # Dimension of the speaker latent vectors (e.g., z_spk.size(-1))
                 mi_weight: float = 1.0,
                 temperature: float = 0.2,
                 target_speaker_level: int = 0):
        """
        Args:
            num_unique_speakers (int): Total number of unique speaker identities in your dataset
                                       (e.g., if you have 100 unique 'pXXX' IDs, this is 100).
                                       This is needed to initialize the embedding layer.
            latent_dim (int): Dimensionality of the speaker latent vectors.
                              The speaker ID embeddings will be projected to this dimension.
            mi_weight (float): Weight for this MI loss.
            temperature (float): Temperature for InfoNCE.
            target_speaker_level (int): Which hierarchical level of speaker_latents
                                        (e.g., speaker_latents['mus'][level]) to use for this loss.
                                        Defaults to the first level (0).
        """
        super().__init__()
        self.num_unique_speakers = num_unique_speakers
        self.latent_dim = latent_dim
        self.mi_weight = mi_weight
        self.temperature = temperature
        self.target_speaker_level = target_speaker_level

        # Learnable embedding for each speaker ID. Each ID will map to a vector
        # of `latent_dim`, making it directly comparable to the speaker latents.
        self.speaker_embedding_layer = nn.Embedding(num_unique_speakers, latent_dim)

        print("\n--- SpeakerIdentityInformationLoss Initialized ---")
        print(f"  - Configured for {num_unique_speakers} unique speakers, latent_dim={latent_dim}.")
        print(f"  - Target speaker latent level: {target_speaker_level}.")
        print(f"  - Loss Weight: {mi_weight}, InfoNCE Temperature: {temperature}.")

    def forward(self, speaker_latents: dict, speaker_ids_int_tensor: torch.LongTensor) -> torch.Tensor:
        """
        Calculates the InfoNCE loss between speaker latents and speaker ID embeddings.

        Args:
            speaker_latents (dict): Dictionary containing 'mus' and 'log_sigmas'
                                    from the speaker encoder, for all hierarchical levels.
            speaker_ids_int_tensor (torch.LongTensor): Batch of integer speaker IDs (e.g., 0, 1, 2...).
                                                    Shape: (batch_size,).
                                                    NOTE: This tensor should contain the mapped integer IDs
                                                    for the *speaker*, not the full utterance ID.
                                                    (e.g., if utt_id is 'p329_012.wav', you should convert
                                                    'p329' to its integer representation for this input).

        Returns:
            torch.Tensor: Weighted InfoNCE loss. This loss should be minimized.
        """
        if 'mus' not in speaker_latents or 'log_sigmas' not in speaker_latents:
             raise ValueError("`speaker_latents` dict must contain 'mus' and 'log_sigmas' keys.")
        if self.target_speaker_level >= len(speaker_latents['mus']):
            raise IndexError(
                f"target_speaker_level={self.target_speaker_level} is out of bounds. "
                f"Speaker encoder only has {len(speaker_latents['mus'])} levels."
            )

        # Get the target speaker latent vectors by sampling from the posterior.
        # This is crucial for VAEs as it works with the stochastic latent space.
        mu_spk = speaker_latents['mus'][self.target_speaker_level]
        log_sigma_spk = speaker_latents['log_sigmas'][self.target_speaker_level]
        z_spk = mu_spk + torch.exp(log_sigma_spk * 0.5) * torch.randn_like(mu_spk)

        # Get speaker ID embeddings from the learned embedding layer.
        speaker_id_embeddings = self.speaker_embedding_layer(speaker_ids_int_tensor)

        # Ensure latent dim and embedding dim match for InfoNCE
        if z_spk.size(-1) != speaker_id_embeddings.size(-1):
            raise ValueError(
                f"Dimension mismatch between speaker latent ({z_spk.size(-1)}) "
                f"and speaker ID embedding ({speaker_id_embeddings.size(-1)}). "
                f"Please ensure `latent_dim` in `__init__` matches your encoder's output "
                f"and the dimension of the embeddings."
            )

        # Calculate InfoNCE loss using the helper function.
        loss = _calculate_info_nce(query=z_spk, key=speaker_id_embeddings, temperature=self.temperature)

        return self.mi_weight * loss

In [ ]:
class TranscriptionInformationLoss(nn.Module):
    """
    Increases mutual information between content latent variables and transcription embeddings.
    Uses InfoNCE loss. This encourages the content latents to explicitly encode
    linguistic/transcription-related information.

    IMPORTANT: This class expects 'transcription_embeddings_tensor' to be
    PRE-COMPUTED fixed-size embeddings (e.g., from a text encoder like BERT).
    It does NOT handle raw text tokenization/embedding internally.
    """
    def __init__(self,
                 transcription_embedding_dim: int, # Dimension of your pre-computed transcription embeddings
                 mi_weight: float = 1.0,
                 temperature: float = 0.2,
                 target_content_level: int = 0):
        """
        Args:
            transcription_embedding_dim (int): Dimensionality of the pre-computed
                                               transcription embeddings that will be passed in.
                                               This must match the dimension of your
                                               content latent vectors.
            mi_weight (float): Weight for this MI loss.
            temperature (float): Temperature for InfoNCE.
            target_content_level (int): Which hierarchical level of content_latents
                                        to use for this loss. Defaults to the first level (0).
        """
        super().__init__()
        self.transcription_embedding_dim = transcription_embedding_dim
        self.mi_weight = mi_weight
        self.temperature = temperature
        self.target_content_level = target_content_level

        print("\n--- TranscriptionInformationLoss Initialized ---")
        print(f"  - Assumed transcription_embedding_dim={transcription_embedding_dim}.")
        print(f"  - Target content latent level: {target_content_level}.")
        print(f"  - Loss Weight: {mi_weight}, InfoNCE Temperature: {temperature}.")
        print("  - NOTE: This class requires `transcription_embeddings_tensor` to be pre-computed.")

    def forward(self, content_latents: dict, transcription_embeddings_tensor: torch.Tensor) -> torch.Tensor:
        """
        Calculates the InfoNCE loss between content latents and transcription embeddings.

        Args:
            content_latents (dict): Dictionary containing 'mus' and 'log_sigmas'
                                    from the content encoder, for all hierarchical levels.
            transcription_embeddings_tensor (torch.Tensor): Batch of pre-computed
                                                            transcription embeddings.
                                                            Shape: (batch_size, embedding_dim).

        Returns:
            torch.Tensor: Weighted InfoNCE loss. This loss should be minimized.
        """
        if 'mus' not in content_latents or 'log_sigmas' not in content_latents:
             raise ValueError("`content_latents` dict must contain 'mus' and 'log_sigmas' keys.")
        if self.target_content_level >= len(content_latents['mus']):
            raise IndexError(
                f"target_content_level={self.target_content_level} is out of bounds. "
                f"Content encoder only has {len(content_latents['mus'])} levels."
            )

        # Get the target content latent vectors by sampling from the posterior.
        mu_con = content_latents['mus'][self.target_content_level]
        log_sigma_con = content_latents['log_sigmas'][self.target_content_level]
        z_con = mu_con + torch.exp(log_sigma_con * 0.5) * torch.randn_like(mu_con)

        # Ensure latent dim matches transcription embedding dim for InfoNCE.
        if z_con.size(-1) != transcription_embeddings_tensor.size(-1):
            raise ValueError(
                f"Dimension mismatch: Content latent dim {z_con.size(-1)} "
                f"does not match transcription embedding dim {transcription_embeddings_tensor.size(-1)}. "
                f"Please ensure `transcription_embedding_dim` in `__init__` matches your "
                f"text embeddings and your content latent dimension."
            )

        # Calculate InfoNCE loss using the helper function.
        loss = _calculate_info_nce(query=z_con, key=transcription_embeddings_tensor, temperature=self.temperature)

        return self.mi_weight * loss

In [ ]:
class SpeakerAttributeInformationLoss(nn.Module):
    """
    Increases mutual information between speaker latent variables and various
    speaker attributes (gender, age, accent). Uses InfoNCE loss.
    This encourages specific speaker attributes to be encoded in the speaker latents.
    """
    def __init__(self,
                 gender_types_count: int,
                 age_types_count: int,
                 accent_types_count: int,
                 latent_dim: int, # Dimension of the speaker latent vectors
                 mi_weight: float = 1.0,
                 temperature: float = 0.2,
                 target_speaker_level: int = 0):
        """
        Args:
            gender_types_count (int): Number of unique gender categories (e.g., 2 for 'M','F').
            age_types_count (int): Number of unique age categories/bins or max_age_value + 1
                                   if treating each age as a distinct category (e.g., 73 for ages 18-90).
            accent_types_count (int): Number of unique accent categories.
            latent_dim (int): Dimensionality of the speaker latent vectors.
                              Each attribute's embedding will be projected to this dimension.
            mi_weight (float): Weight for this MI loss.
            temperature (float): Temperature for InfoNCE.
            target_speaker_level (int): Which hierarchical level of speaker_latents
                                        to use for this loss. Defaults to the first level (0).
        """
        super().__init__()
        self.mi_weight = mi_weight
        self.temperature = temperature
        self.target_speaker_level = target_speaker_level

        # Learnable embeddings for each attribute type.
        # Each embedding layer maps an integer ID to a vector of `latent_dim`.
        self.gender_embedding_layer = nn.Embedding(gender_types_count, latent_dim)
        self.age_embedding_layer = nn.Embedding(age_types_count, latent_dim)
        self.accent_embedding_layer = nn.Embedding(accent_types_count, latent_dim)

        print("\n--- SpeakerAttributeInformationLoss Initialized ---")
        print(f"  - Configured for gender_types={gender_types_count}, age_types={age_types_count}, accent_types={accent_types_count}.")
        print(f"  - Latent_dim={latent_dim}.")
        print(f"  - Target speaker latent level: {target_speaker_level}.")
        print(f"  - Loss Weight: {mi_weight}, InfoNCE Temperature: {temperature}.")

    def forward(self,
                speaker_latents: dict,
                gender_int_tensor: torch.LongTensor,
                age_int_tensor: torch.LongTensor,
                accent_int_tensor: torch.LongTensor) -> torch.Tensor:
        """
        Calculates the combined InfoNCE loss for gender, age, and accent against speaker latents.
        The losses for individual attributes are summed up.

        Args:
            speaker_latents (dict): Dictionary containing 'mus' and 'log_sigmas'
                                    from the speaker encoder, for all hierarchical levels.
            gender_int_tensor (torch.LongTensor): Batch of integer gender IDs (e.g., 0, 1). Shape: (batch_size,).
            age_int_tensor (torch.LongTensor): Batch of integer age IDs (e.g., 0, 1, 2...). Shape: (batch_size,).
            accent_int_tensor (torch.LongTensor): Batch of integer accent IDs (e.g., 0, 1, 2...). Shape: (batch_size,).
            NOTE: All integer tensors should be 0-indexed and correspond to the counts
                  provided in the `__init__` method.

        Returns:
            torch.Tensor: Weighted combined InfoNCE loss for attributes. This loss should be minimized.
        """
        if 'mus' not in speaker_latents or 'log_sigmas' not in speaker_latents:
             raise ValueError("`speaker_latents` dict must contain 'mus' and 'log_sigmas' keys.")
        if self.target_speaker_level >= len(speaker_latents['mus']):
            raise IndexError(
                f"target_speaker_level={self.target_speaker_level} is out of bounds. "
                f"Speaker encoder only has {len(speaker_latents['mus'])} levels."
            )

        # Get the target speaker latent vectors by sampling from the posterior.
        mu_spk = speaker_latents['mus'][self.target_speaker_level]
        log_sigma_spk = speaker_latents['log_sigmas'][self.target_speaker_level]
        z_spk = mu_spk + torch.exp(log_sigma_spk * 0.5) * torch.randn_like(mu_spk)

        # Get embeddings for each attribute.
        gender_embeddings = self.gender_embedding_layer(gender_int_tensor)
        age_embeddings = self.age_embedding_layer(age_int_tensor)
        accent_embeddings = self.accent_embedding_layer(accent_int_tensor)

        # Calculate separate InfoNCE losses for each attribute.
        # This allows each attribute to directly influence the latent representation.
        loss_gender = _calculate_info_nce(query=z_spk, key=gender_embeddings, temperature=self.temperature)
        loss_age = _calculate_info_nce(query=z_spk, key=age_embeddings, temperature=self.temperature)
        loss_accent = _calculate_info_nce(query=z_spk, key=accent_embeddings, temperature=self.temperature)

        # Sum the individual attribute losses. You could apply different weights here
        # if you want some attributes to be more strongly encoded than others.
        total_attribute_loss = loss_gender + loss_age + loss_accent

        return self.mi_weight * total_attribute_loss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math # For KL annealing

# --- 1. SpeakerLossCalculator Class (from previous response) ---
#    (Assume this is defined in a prior cell, included here for completeness)
class SpeakerLossCalculator:
    def __init__(self, n_speaker_levels, alpha_latent_reg=0.001, latent_reg_type='l2'):
        self.n_speaker_levels = n_speaker_levels
        self.alpha_latent_reg = alpha_latent_reg
        self.latent_reg_type = latent_reg_type.lower()

        if self.latent_reg_type not in ['l1', 'l2']:
            raise ValueError("latent_reg_type must be 'l1' or 'l2'")

    def calculate_kld(self, mu_post, log_sigma_post, mu_prior=None, log_sigma_prior=None):
        if mu_prior is None and log_sigma_prior is None:
            kld = 0.5 * torch.sum(
                (2 * log_sigma_post).exp() + mu_post.pow(2) - 1 - (2 * log_sigma_post)
            )
        else:
            sigma_post_sq = torch.exp(2 * log_sigma_post)
            sigma_prior_sq = torch.exp(2 * log_sigma_prior)

            kld = 0.5 * torch.sum(
                (2 * log_sigma_prior - 2 * log_sigma_post) +
                (sigma_post_sq + (mu_post - mu_prior).pow(2)) / sigma_prior_sq - 1
            )
        return kld

    def speaker_kld_loss(self, speaker_latents):
        total_speaker_kld_loss = 0
        for i in range(self.n_speaker_levels):
            mu_post = speaker_latents['mus'][i]
            log_sigma_post = speaker_latents['log_sigmas'][i]
            mu_prior = speaker_latents['mu_priors'][i]
            log_sigma_prior = speaker_latents['log_sigma_priors'][i]

            # Ensure that prior is handled correctly if it's None (for z1)
            if mu_prior is None and log_sigma_prior is None and i == 0:
                # KLD for the first level (z1) with standard normal prior
                total_speaker_kld_loss += self.calculate_kld(
                    mu_post, log_sigma_post, None, None
                )
            elif mu_prior is not None and log_sigma_prior is not None:
                # KLD for subsequent levels with predicted prior
                total_speaker_kld_loss += self.calculate_kld(
                    mu_post, log_sigma_post, mu_prior, log_sigma_prior
                )
            # else: (Optional: add a warning/error if a non-first level has None priors unexpectedly)
        return total_speaker_kld_loss

    def latent_mean_regularization_loss(self, speaker_latents):
        reg_loss = 0
        for mu_post in speaker_latents['mus']:
            if self.latent_reg_type == 'l1':
                reg_loss += torch.mean(torch.abs(mu_post))
            elif self.latent_reg_type == 'l2':
                reg_loss += torch.mean(mu_post.pow(2))
        return self.alpha_latent_reg * reg_loss

    def calculate_total_speaker_loss(self, speaker_latents, lambda_kl):
        kld_loss = self.speaker_kld_loss(speaker_latents)
        latent_reg_loss = self.latent_mean_regularization_loss(speaker_latents)

        weighted_kld_loss = kld_loss * lambda_kl

        total_speaker_loss = weighted_kld_loss + latent_reg_loss
        return total_speaker_loss, weighted_kld_loss, latent_reg_loss

**-------------------------------------------------------------------**

**------------- 12- Disentaglement Classification-------------**

**-------------------------------------------------------------------**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, r2_score
from sklearn.preprocessing import StandardScaler # Good practice for probe inputs
from sklearn.model_selection import train_test_split # Crucial for train/test split of probes
import warnings # To manage ConvergenceWarning

# Assuming 'cc' function is available (e.g., self.cc(tensor))
# And self.device, self.model, self._to_int mappings, and self.text_encoder/projection are initialized.

# Helper to get the latent tensor from the dictionary output of your model
def _get_first_level_latent(latent_dict):
    # Assumes latent_dict['mus'] is a list of tensors, we take the first one
    return latent_dict['mus'][0]

def evaluate_disentanglement_metrics(self, eval_dataloader, latent_dim=128):
    """
    Evaluates disentanglement metrics by training and testing probes on extracted latent spaces.
    The probes are trained on a subset of the collected latents and evaluated on another subset
    to provide a more realistic assessment of disentanglement.

    Args:
        self: The main model/trainer instance, expected to have attributes like
              self.model, self.device, self.speaker_id_to_int, etc.
        eval_dataloader: DataLoader providing evaluation data (audio segments, metadata).
                         Crucially, this should ideally be a *separate* validation/test set
                         distinct from the training data.
        latent_dim: The dimensionality of the latent spaces (used for probe hidden layers).

    Returns:
        A dictionary of disentanglement metrics.
    """
    self.model.eval() # Set main model to evaluation mode

    all_speaker_latents = []
    all_content_latents = []

    all_speaker_ids_gt = [] # Ground Truth
    all_genders_gt = []
    all_ages_gt = []
    all_accents_gt = []
    all_transcription_embeddings_gt = []

    # Suppress ConvergenceWarning during probe training, as they might not fully converge
    # but still provide useful insight into separability.
    warnings.filterwarnings("ignore", category=ConvergenceWarning)

    # Collect latents and ground truth labels from a portion of the evaluation dataset
    print("Collecting latent representations for disentanglement evaluation...")
    with torch.no_grad(): # No gradient calculation for evaluation
        for batch_idx, batch_data in enumerate(eval_dataloader):
            # Unpack batch data (adjust according to your DataLoader's output structure)
            # Ensure the order matches your DataLoader's __getitem__
            segment, utt_id, transcription, gender, age, accent = batch_data

            # Move segment to device (assuming cc handles this correctly)
            segment = self.cc(segment, self.device) # Using self.cc instead of global cc

            # --- Preprocess auxiliary data (similar to ae_step) ---
            speaker_ids_for_batch = [uid.split('_')[0] for uid in utt_id]
            speaker_ids_int = torch.tensor(
                [self.speaker_id_to_int[speaker_uid] for speaker_uid in speaker_ids_for_batch],
                dtype=torch.long, device=self.device
            )
            gender_int = torch.tensor(
                [self.gender_to_int[g] for g in gender],
                dtype=torch.long, device=self.device
            )
            age_int = torch.tensor(
                [self.age_to_int[a] for a in age],
                dtype=torch.long, device=self.device
            )
            accent_int = torch.tensor(
                [self.accent_to_int[a] for a in accent],
                dtype=torch.long, device=self.device
            )

            # Ensure text_encoder and text_encoder_projection are available and correctly used
            raw_text_embeddings = self.text_encoder.encode(
                transcription, convert_to_tensor=True, show_progress_bar=False
            )
            transcription_embeddings = self.text_encoder_projection(raw_text_embeddings).to(self.device)

            # Forward pass through AE model
            output = self.model(segment)
            speaker_latents_batch = _get_first_level_latent(output['speaker_latents'])
            content_latents_batch = _get_first_level_latent(output['content_latents'])

            # Store on CPU for scikit-learn processing (important for performance and compatibility)
            all_speaker_latents.append(speaker_latents_batch.cpu())
            all_content_latents.append(content_latents_batch.cpu())

            all_speaker_ids_gt.append(speaker_ids_int.cpu())
            all_genders_gt.append(gender_int.cpu())
            all_ages_gt.append(age_int.cpu())
            all_accents_gt.append(accent_int.cpu())
            all_transcription_embeddings_gt.append(transcription_embeddings.cpu())

            # Optional: Limit evaluation data size for faster checks during development.
            # Using too few batches can lead to unstable probe results. Aim for 100-200 batches if possible.
            if batch_idx >= 100: # Evaluate on 100 batches to get a reasonable sample size
                break
    print(f"Collected data from {batch_idx + 1} batches.")

    # Concatenate all collected data and convert to numpy for scikit-learn
    X_s = torch.cat(all_speaker_latents).numpy()
    X_c = torch.cat(all_content_latents).numpy()

    y_speaker_id = torch.cat(all_speaker_ids_gt).numpy()
    y_gender = torch.cat(all_genders_gt).numpy()
    y_age = torch.cat(all_ages_gt).numpy()
    y_accent = torch.cat(all_accents_gt).numpy()
    y_transcription_emb = torch.cat(all_transcription_embeddings_gt).numpy()

    # Handle cases where collected data might be empty (e.g., if dataloader is exhausted too soon)
    if X_s.shape[0] == 0 or X_c.shape[0] == 0:
        print("Warning: No data collected for disentanglement metrics. Returning empty metrics.")
        self.model.train()
        return {}

    # Apply StandardScaler to the full collected data *before* splitting for probes
    scaler_s = StandardScaler()
    X_s_scaled = scaler_s.fit_transform(X_s)
    scaler_c = StandardScaler()
    X_c_scaled = scaler_c.fit_transform(X_c)

    metrics = {}
    PROBE_TEST_SIZE = 0.3 # Use 30% of collected data for probe evaluation
    PROBE_RANDOM_STATE = 42 # For reproducibility of probe training/testing splits

    # --- 1. Speaker Latent Space Evaluation (s_latent) ---
    # Expected: High accuracy for speaker-related attributes, low R2 for content.

    print("\n--- Evaluating Speaker Latent Space ---")

    # 1.1 Speaker ID Classification (Desired: HIGH accuracy)
    # Split the scaled speaker latents and ground truth speaker IDs for probe training/testing
    X_s_train_id, X_s_test_id, y_speaker_id_train, y_speaker_id_test = train_test_split(
        X_s_scaled, y_speaker_id, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE,
        stratify=y_speaker_id # Stratify to maintain class distribution in splits
    )
    probe_s_speaker_id = MLPClassifier(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
    probe_s_speaker_id.fit(X_s_train_id, y_speaker_id_train) # Train probe
    metrics['speaker_latent_speaker_id_accuracy'] = probe_s_speaker_id.score(X_s_test_id, y_speaker_id_test) # Evaluate probe

    # 1.2 Gender Classification (Desired: HIGH accuracy)
    X_s_train_gender, X_s_test_gender, y_gender_train, y_gender_test = train_test_split(
        X_s_scaled, y_gender, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_gender
    )
    probe_s_gender = MLPClassifier(hidden_layer_sizes=(latent_dim // 2,), max_iter=300, random_state=PROBE_RANDOM_STATE)
    probe_s_gender.fit(X_s_train_gender, y_gender_train)
    metrics['speaker_latent_gender_accuracy'] = probe_s_gender.score(X_s_test_gender, y_gender_test)

    # 1.3 Age Classification (Desired: HIGH accuracy)
    X_s_train_age, X_s_test_age, y_age_train, y_age_test = train_test_split(
        X_s_scaled, y_age, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_age
    )
    probe_s_age = MLPClassifier(hidden_layer_sizes=(latent_dim // 2,), max_iter=300, random_state=PROBE_RANDOM_STATE)
    probe_s_age.fit(X_s_train_age, y_age_train)
    metrics['speaker_latent_age_accuracy'] = probe_s_age.score(X_s_test_age, y_age_test)

    # 1.4 Accent Classification (Desired: HIGH accuracy)
    X_s_train_accent, X_s_test_accent, y_accent_train, y_accent_test = train_test_split(
        X_s_scaled, y_accent, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_accent
    )
    probe_s_accent = MLPClassifier(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
    probe_s_accent.fit(X_s_train_accent, y_accent_train)
    metrics['speaker_latent_accent_accuracy'] = probe_s_accent.score(X_s_test_accent, y_accent_test)

    # 1.5 Content Prediction (Transcription Embedding) (Desired: LOW/NEGATIVE R2)
    # R2 near 0 or negative indicates the latent space does not encode this information well.
    X_s_train_content_reg, X_s_test_content_reg, y_transcription_emb_train_s, y_transcription_emb_test_s = train_test_split(
        X_s_scaled, y_transcription_emb, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE
    )
    probe_s_content_reg = LinearRegression() # Simpler model to check for linear dependency
    probe_s_content_reg.fit(X_s_train_content_reg, y_transcription_emb_train_s)
    metrics['speaker_latent_content_R2'] = probe_s_content_reg.score(X_s_test_content_reg, y_transcription_emb_test_s)


    # --- 2. Content Latent Space Evaluation (c_latent) ---
    # Expected: Low accuracy for speaker-related attributes, high R2 for content.

    print("\n--- Evaluating Content Latent Space ---")

    # 2.1 Speaker ID Classification (Desired: LOW accuracy, ideally close to random chance)
    X_c_train_id, X_c_test_id, y_speaker_id_train_c, y_speaker_id_test_c = train_test_split(
        X_c_scaled, y_speaker_id, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_speaker_id
    )
    probe_c_speaker_id = MLPClassifier(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
    probe_c_speaker_id.fit(X_c_train_id, y_speaker_id_train_c)
    metrics['content_latent_speaker_id_accuracy'] = probe_c_speaker_id.score(X_c_test_id, y_speaker_id_test_c)

    # 2.2 Transcription Embedding Prediction (Desired: HIGH R2)
    X_c_train_transcription_reg, X_c_test_transcription_reg, y_transcription_emb_train_c, y_transcription_emb_test_c = train_test_split(
        X_c_scaled, y_transcription_emb, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE
    )
    probe_c_transcription_reg = MLPRegressor(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
    probe_c_transcription_reg.fit(X_c_train_transcription_reg, y_transcription_emb_train_c)
    metrics['content_latent_transcription_R2'] = probe_c_transcription_reg.score(X_c_test_transcription_reg, y_transcription_emb_test_c)

    # Re-enable warnings after probe training
    warnings.filterwarnings("default", category=ConvergenceWarning)

    self.model.train() # Set main model back to training mode
    return metrics

**------------------------------------**

**-------------- 13- Latent Plot ---------**

**-------------------------------------**

In [ ]:
# latent_plotter.py
import numpy as np
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import os

class LatentPlotter:
    def __init__(self, plot_dir="latent_plots", tsne_params=None, umap_params=None, plot_num_samples=1000):
        self.plot_dir = plot_dir
        os.makedirs(self.plot_dir, exist_ok=True)

        self.tsne_params = tsne_params if tsne_params else {
            'n_components': 2, 'random_state': 42, 'perplexity': 30, 'n_iter': 1000,
            'learning_rate': 'auto', 'init': 'pca'
        }
        self.umap_params = umap_params if umap_params else {
            'n_components': 2, 'random_state': 42, 'n_neighbors': 15, 'min_dist': 0.1,
            'metric': 'euclidean'
        }
        self.plot_num_samples = plot_num_samples

    def plot_latents(self,
                     current_iteration,
                     accumulated_speaker_latents_dict, # Dictionary of lists of numpy arrays
                     accumulated_speaker_ids_list,     # List of speaker IDs
                     accumulated_content_latents_list, # List of numpy arrays
                     accumulated_content_labels_list): # List of content labels
        """
        Processes and plots accumulated latents.

        Args:
            current_iteration (int): The current training iteration for filename/title.
            accumulated_speaker_latents_dict (dict): Dictionary where keys are level names (e.g., 'z1')
                                                     and values are lists of NumPy arrays of speaker latents.
            accumulated_speaker_ids_list (list): List of speaker IDs accumulated.
            accumulated_content_latents_list (list): List of NumPy arrays of content latents accumulated.
            accumulated_content_labels_list (list): List of content labels accumulated.
        """
        print(f"\n--- Plotting Latents at Iteration {current_iteration} ---")

        # --- Process Speaker Latents ---
        min_samples_speaker = len(accumulated_speaker_ids_list)
        if min_samples_speaker == 0:
            print("No valid speaker IDs accumulated for plotting. Skipping speaker latent plots.")
        else:
            processed_speaker_latents = {}
            for level_name, latent_arrays in accumulated_speaker_latents_dict.items():
                if len(latent_arrays) > 0:
                    processed_speaker_latents[level_name] = np.concatenate(latent_arrays, axis=0)[:min_samples_speaker]
                else:
                    processed_speaker_latents[level_name] = np.array([])

            processed_speaker_ids = np.array(accumulated_speaker_ids_list)[:min_samples_speaker]

            for level_name, latents_array in processed_speaker_latents.items():
                if latents_array.shape[0] > 0:
                    self._plot_scatter_reduction(
                        latents_array, processed_speaker_ids,
                        title_prefix=f'Iter {current_iteration} Speaker Encoder {level_name} Latents',
                        filename_suffix=f'iter_{current_iteration}_speaker_latent_{level_name}'
                    )
                else:
                    print(f"Skipping plot for {level_name} as no latents were collected.")

        # --- Process Content Latents ---
        min_samples_content = len(accumulated_content_labels_list)
        if min_samples_content == 0:
            print("No valid content labels accumulated for plotting. Skipping content latent plots.")
        else:
            processed_content_latents = np.array([])
            if len(accumulated_content_latents_list) > 0:
                processed_content_latents = np.concatenate(accumulated_content_latents_list, axis=0)[:min_samples_content]

            processed_content_labels = np.array(accumulated_content_labels_list)[:min_samples_content]

            if processed_content_latents.shape[0] > 0:
                self._plot_scatter_reduction(
                    processed_content_latents, processed_content_labels,
                    title_prefix=f'Iter {current_iteration} Content Encoder Latents',
                    filename_suffix=f'iter_{current_iteration}_content_latent'
                )
            else:
                print("Skipping plot for Content Encoder as no latents were collected.")

        print(f"--- Latent Plotting at Iteration {current_iteration} Finished ---")

    def _plot_scatter_reduction(self, latents, labels, title_prefix, filename_suffix):
        """Performs dimensionality reduction and plots the results."""
        if latents.shape[0] == 0:
            print(f"Skipping visualization for {title_prefix}: No latents provided.")
            return

        # Handle max_samples internally using self.plot_num_samples
        if self.plot_num_samples and latents.shape[0] > self.plot_num_samples:
            print(f"Subsampling {latents.shape[0]} to {self.plot_num_samples} for visualization.")
            indices = np.random.choice(latents.shape[0], self.plot_num_samples, replace=False)
            latents = latents[indices]
            labels = labels[indices]

        # --- t-SNE ---
        print(f"Running t-SNE for {title_prefix}...")
        try:
            tsne = TSNE(**self.tsne_params)
            tsne_results = tsne.fit_transform(latents)
            self._plot_scatter(tsne_results, labels, title_prefix, filename_suffix, "t-SNE")
        except Exception as e:
            print(f"Error running t-SNE for {title_prefix}: {e}")

        # --- UMAP ---
        print(f"Running UMAP for {title_prefix}...")
        try:
            reducer = umap.UMAP(**self.umap_params)
            umap_results = reducer.fit_transform(latents)
            self._plot_scatter(umap_results, labels, title_prefix, filename_suffix, "UMAP")
        except Exception as e:
            print(f"Error running UMAP for {title_prefix}: {e}")

    def _plot_scatter(self, reduced_latents, labels, title_prefix, filename_suffix, method_name):
        """Helper to create and save scatter plots."""
        unique_labels = np.unique(labels)
        num_unique_labels = len(unique_labels)
        palette = sns.color_palette("tab10", n_colors=min(num_unique_labels, 10))
        if num_unique_labels > 10:
            palette = sns.color_palette("tab20", n_colors=min(num_unique_labels, 20))
        if num_unique_labels > 20:
             palette = sns.color_palette(n_colors=num_unique_labels, desat=0.9)

        plt.figure(figsize=(10, 8))
        sns.scatterplot(
            x=reduced_latents[:, 0], y=reduced_latents[:, 1],
            hue=labels,
            palette=palette,
            legend="full" if num_unique_labels <= 20 else False,
            alpha=0.8, s=10
        )
        if num_unique_labels > 20:
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
            plt.subplots_adjust(right=0.7)

        plt.title(f'{title_prefix} {method_name} Plot')
        plt.xlabel(f'{method_name} Dimension 1')
        plt.ylabel(f'{method_name} Dimension 2')
        plt.grid(True)
        plot_path = os.path.join(self.plot_dir, f'{filename_suffix}_{method_name.lower()}.png')
        plt.savefig(plot_path)
        plt.close()

**-----------------------------------------------**

**14- HyperParameter & Configuration**

**-----------------------------------------------**

In [ ]:

import yaml

class ARGS2():
  def __init__(self):
    # Store the YAML content directly as a string
    self.config_content = """
SpeakerEncoder:
    c_in: 80
    c_h: 128
    c_out: 128
    kernel_size: 5
    bank_size: 8
    bank_scale: 1
    c_bank: 128
    n_conv_blocks: 6
    n_dense_blocks: 6
    subsample: [1, 2, 1, 2, 1, 2]
    act: 'relu'
    dropout_rate: 0
ContentEncoder:
    c_in: 80
    c_h: 128
    c_out: 128
    kernel_size: 5
    bank_size: 8
    bank_scale: 1
    c_bank: 128
    n_conv_blocks: 6
    subsample: [1, 2, 1, 2, 1, 2]
    act: 'relu'
    dropout_rate: 0
Decoder:
    c_in: 128
    c_cond: 128
    c_h: 128
    c_out: 80
    kernel_size: 5
    n_conv_blocks: 6
    upsample: [2, 1, 2, 1, 2, 1]
    act: 'relu'
    sn: False
    dropout_rate: 0
data_loader:
    segment_size: 128
    frame_size: 1
    batch_size: 128
    shuffle: True
optimizer:
    lr: 0.00001 # lr: 0.0005
    beta1: 0.9
    beta2: 0.999
    amsgrad: True
    weight_decay: 0.0001
    grad_norm: 5
lambda:
    lambda_rec: 10
    lambda_kl:  1 # MRH: changing here Start with a much smaller target, e.g., 0.0001
annealing_iters: 20000
    """
    self.data_dir='/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/' # MRH: You may Modify the part
    self.train_set='train'
    self.in_test='in_test'
    self.out_test='out_test'
    # self.train_index_file='train_samples_128.json'
    self.logdir='log/'
    self.load_model=False
    self.store_model_path='/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/checkpoint' # MRH: You may Modify the part
    self.load_model_path='/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/checkpoint' # MRH: You may Modify the part
    self.summary_steps=500
    self.save_steps=50
    self.tag='vctk_model'
    self.iters=500


    ############# for inference
    # These attributes directly correspond to your parser.add_argument defaults
    self.attr = "/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/attr.pkl"      # No default in argparse, so None
    self.model = "/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/checkpoint.ckpt"       # No default
    self.source = "/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin/p225_001.wav"      # No default
    self.target = "/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin/p226_001.wav"      # No default
    self.output = "/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin/_converted2.wav"      # No default
    self.sample_rate = 22050 # Default value from argparse



    #  Hierarchy

    self.use_content_hierarchy=True
    self.n_content_levels=1 # MRH: in the AE1,  if use_content_hierarchy= True then n_content_levels gets number else it is 1
    self.use_speaker_hierarchy=True
    self.n_speaker_levels=3 # MRH: in the AE1,  if use_speaker_hierarchy= True then n_content_levels gets number else it is 1

    # Parse the YAML content into a dictionary
    self.config = yaml.safe_load(self.config_content)


    ## for transcription
    self.train_transcripts_file = 'train_transcripts.pkl'
    self.speaker_info_file= 'speaker-info.txt'
    self.sampling_rate=22050

##
args2=ARGS2()
print(args2.train_set)
print(args2.data_dir)

# Now you can access your config parameters like this:
print("\nAccessing config parameters:")
print(args2.config['SpeakerEncoder']['c_in'])
print(args2.config['data_loader']['batch_size'])

train
/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin_unified_Experimental/

Accessing config parameters:
80
128


**-------------------------------------**

**---------- 15-Solver --------------**

**-------------------------------------**

In [ ]:
# adaptive_voice_conversion /solver5.py
import torch
import numpy as np
import sys
import os
import torch.nn as nn
import torch.nn.functional as F
import yaml
import pickle
# from model import AE
# from data_utils import get_data_loader
# from data_utils import PickleDataset
# from utils import *
from functools import reduce
from collections import defaultdict
from tqdm.auto import tqdm  ############### > MRH: change
##
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import accuracy_score, r2_score
from sklearn.preprocessing import StandardScaler # Good practice for probe inputs
from sklearn.exceptions import ConvergenceWarning
##


class Solver5(object):
    def __init__(self, config, args):
        # print('MRH1')
        # config store the value of hyperparameters, turn to attr by AttrDict
        self.config = config
        # print(config)
        # print('MRH2')

        # args store other information
        self.args = args
        # print(self.args)
        # print(' self.args.use_content_hierarchy = ',self.args.use_content_hierarchy)
        # print('MRH3')

        # logger to use tensorboard
        self.logger = Logger(self.args.logdir)
        # print('MRH4')

        # get dataloader
        self.get_data_loaders()
        # print('MRH5')

        # init the model with config
        self.build_model()
        # print('MRH6')
        self.save_config()
        # print('MRH7')

        if args.load_model:
            self.load_model()

        # print('MRH8')

                ####### MRH: This is the change
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        else:
            self.device = torch.device("cpu")
        ####

        ######################################################################### MRH: Loss
        ae2=AE2(self.args)   ### change AE1 to AE2
        self.ae_loss_calculator = AELoss2(n_speaker_levels=ae2.n_speaker_levels,
                                         n_content_levels=ae2.n_content_levels)

        # print('mrh 1 ae1.n_speaker_levels = ',ae1.n_speaker_levels)
        # print('mrh 2 args1.speaker_n_levels = ',args1.speaker_n_levels)
        self.s_disent_loss_calc = DisentanglementLossCalculator(args.n_speaker_levels, mi_weight=1, dcor_weight=1)
        self.c_disent_loss_calc = DisentanglementLossCalculator(args.n_content_levels, mi_weight=1, dcor_weight=1)
        self.speaker_loss_calculator=SpeakerLossCalculator(args.n_speaker_levels, alpha_latent_reg=0.001, latent_reg_type='l1')
        # speaker_loss_calculator=SpeakerLossCalculator(args1.speaker_n_levels, alpha_latent_reg=0.001, latent_reg_type='l2')

        self.cross_enc_mi_loss = CrossEncoderMutualInformationLoss(args.n_speaker_levels, args.n_content_levels, mi_weight=4.0)

        #####################
        self.mrh_AttributeData_prepare()
        # 1. Speaker Identity Information Loss
        self.speaker_id_loss_calc = SpeakerIdentityInformationLoss(
            num_unique_speakers=self.num_unique_speakers,
            latent_dim=args.config['SpeakerEncoder']['c_out'], # Dimension of your speaker encoder's latent output
            mi_weight=1.0,  # Hyperparameter: Adjust as needed
            temperature=0.05, # Hyperparameter: Adjust as needed (e.g., 0.07 to 0.2)
            target_speaker_level=0,
        ).to(self.device) # Which hierarchical level of speaker_latents to target)

        # 2. Transcription Information Loss
        #    Requires a text encoder setup, as it expects embeddings, not raw strings.
        self.transcription_loss_calc = TranscriptionInformationLoss(
            transcription_embedding_dim=args.config['ContentEncoder']['c_out'], # This MUST match your text encoder's output dim
            mi_weight=1.0,  # Hyperparameter: Adjust as needed
            temperature=0.5, # Hyperparameter: Adjust as needed
            target_content_level=0 # Which hierarchical level of content_latents to target
        ).to(self.device)

        # Text Encoder Initialization (Crucial for TranscriptionInformationLoss)
        # This will be `self.text_encoder` and `self.text_encoder_projection`
        # You MUST replace this with your actual text embedding model.
        # Example using SentenceTransformers (ensure 'pip install sentence-transformers' if using):
        try:
            from sentence_transformers import SentenceTransformer
            # Initialize the SentenceTransformer model
            # Choose a model. 'all-MiniLM-L6-v2' outputs 384-dim embeddings.
            # You might need to try different models or add a projection.
            raw_text_encoder = SentenceTransformer('all-MiniLM-L6-v2')
            raw_text_encoder_output_dim = raw_text_encoder.get_sentence_embedding_dimension()

            # Create a projection layer if the text encoder's output dim doesn't match args.config['ContentEncoder']['c_out']
            # In Solver's __init__
            if raw_text_encoder_output_dim != args.config['ContentEncoder']['c_out']:
                # Use an MLP with ReLU activation, maybe a dropout layer
                self.text_encoder_projection = nn.Sequential(
                    nn.Linear(raw_text_encoder_output_dim, raw_text_encoder_output_dim // 2), # Hidden layer
                    nn.ReLU(),
                    # nn.Dropout(0.1), # Optional
                    nn.Linear(raw_text_encoder_output_dim // 2, args.config['ContentEncoder']['c_out']),
                    # nn.LayerNorm(args.config['ContentEncoder']['c_out']) # Consider if still needing normalization
                )
            else:
                self.text_encoder_projection = nn.Identity() # Or add LayerNorm/MLP here if transforming is always desired

            # Store the actual encoder and move to device
            self.text_encoder = raw_text_encoder
            self.text_encoder.to(self.device) # Assuming self.device is defined
            self.text_encoder_projection.to(self.device)

        except ImportError:
            print("Warning: sentence-transformers not found. Please install it (`pip install sentence-transformers`) or provide your own text encoder.")
            self.text_encoder = None
            self.text_encoder_projection = None
        except Exception as e:
            print(f"Error initializing text encoder: {e}")
            self.text_encoder = None
            self.text_encoder_projection = None


        # 3. Speaker Attribute Information Loss
        self.speaker_attribute_loss_calc = SpeakerAttributeInformationLoss(
            gender_types_count=self.gender_types_count,
            age_types_count=self.age_types_count,
            accent_types_count=self.accent_types_count,
            latent_dim=args.config['ContentEncoder']['c_out'], # Dimension of your speaker encoder's latent output
            mi_weight=1.0,  # Hyperparameter: Adjust as needed
            temperature=0.05, # Hyperparameter: Adjust as needed
            target_speaker_level=0 # Which hierarchical level of speaker_latents to target
        ).to(self.device)







        # self.speaker_verifier = SpeakerVerificationEvaluator(device=self.args.device if hasattr(self.args, 'device') else 'cpu')
        # self.speaker_verifier = SpeakerVerificationEvaluator(self.device)


        #################################### MRH: MCD
        # Initialize the MCD metric
        self.mcd_metric = MCDMetric(n_mfcc=20)
        # List to store MCD values for plotting
        self.mcd_values_history = [] # <-- ADD THIS LINE


        ###################################### MRH: To incorporate Inference
        # if isinstance(args2.config, dict):
        #     config = args2.config
        # else:
        #     # fallback if it's still a path string
        #     with open(args2.config) as f:
        #         config = yaml.load(f, Loader=yaml.SafeLoader)
        # inferencer = Inferencer(config=config, args=args2)
        # # inferencer.inference_from_path()


        # if isinstance(args.config, dict):
        #     config = args.config
        # else:
        #     # fallback if it's still a path string
        #     with open(args.config) as f:
        #         config = yaml.load(f, Loader=yaml.SafeLoader)
        # self.inferencer = Inferencer(config=config, args=args)
        # # inferencer.inference_from_path()

        ######################################### MRH: Audio Metrics
        # pesq_sample_rate=16000
        # self.VQA = VoiceQualityEvaluator(pesq_sample_rate=pesq_sample_rate)

        ######################################### MRH: ASR
        self.ASR_Whisper = ASRWhisperEvaluator2(model_name="openai/whisper-tiny")
        # self.ASR_wav2vec2 = ASRWav2Vec2Evaluator(model_name="facebook/wav2vec2-base-960h")

        ####
        self.history = {
            'iteration': [],
            'loss_rec': [],
            'speaker_loss_kl': [],
            'content_loss_kl': [],
            'lambda_kl': [],
            'mcd': [],
            'mcd_dtw': [],
            'batch_lsd': [],
            'pesq_score': [],
            'stoi_score': [],
            'whisper_source_mel_wer': [],
            'whisper_source_mel_cer': [],
            'whisper_output_mel_wer': [],
            'whisper_output_mel_cer': [],
            'whisper_source_wave_wer': [],
            'whisper_source_wave_cer': [],
            'whisper_output_wave_wer': [],
            'whisper_output_wave_cer': [],
            'wave2vec_source_wer': [],
            'wave2vec_source_cer': [],
            'wave2vec_target_wer': [],
            'wave2vec_target_cer': [],
            'batch_eer': [], # For batch-level EER
            'batch_roc_auc': [], # For batch-level ROC AUC
            # 'speaker_eval_eer': [], # If you have a separate full speaker eval
            # 'speaker_eval_roc;[_auc': [], # If you have a separate full speaker eval
            'inter_level_loss' :[],
            'intra_level_loss' :[],
            'c_inter_level_loss':[],
            'c_intra_level_loss':[],
            'cross_enc_loss':[],
            'loss_speaker_id_mi':[],
            'loss_transcription_mi':[],
            'loss_speaker_attribute_mi':[],
            'speaker_latent_speaker_id_accuracy':[],
            'speaker_latent_gender_accuracy':[],
            'speaker_latent_age_accuracy':[],
            'speaker_latent_accent_accuracy':[],
            'speaker_latent_content_R2':[],
            'content_latent_speaker_id_accuracy':[],
            'content_latent_transcription_R2':[]

        }



        #########################
        # Initialize the LatentPlotter for visualization (not accumulation)
        self.latent_plotter = LatentPlotter(
            plot_dir=self.config.get('plot_dir', 'latent_plots'),
            tsne_params=self.config.get('tsne_params'),
            umap_params=self.config.get('umap_params'),
            plot_num_samples=self.config.get('plot_num_samples', 1000)
        )

        # Storage for accumulating latents (remains in Solver)
        self.accumulated_speaker_latents = defaultdict(list)
        self.accumulated_speaker_ids = []
        self.accumulated_content_latents = []
        self.accumulated_content_labels = []

        self.speaker_id_map = {} # Will map original speaker identifier (e.g., string) to an integer ID
        self.next_speaker_id_index = 0 # Counter for assigning new integer IDs
###########################




    def save_model(self, iteration):
        # save model and discriminator and their optimizer
        torch.save(self.model.state_dict(), f'{self.args.store_model_path}.ckpt')
        torch.save(self.opt.state_dict(), f'{self.args.store_model_path}.opt')

    def save_config(self):
        with open(f'{self.args.store_model_path}.config.yaml', 'w') as f:
            yaml.dump(self.config, f)
        with open(f'{self.args.store_model_path}.args.yaml', 'w') as f:
            yaml.dump(vars(self.args), f)
        return

    def load_model(self):
        print(f'Load model from {self.args.load_model_path}')
        self.model.load_state_dict(torch.load(f'{self.args.load_model_path}.ckpt'))
        self.opt.load_state_dict(torch.load(f'{self.args.load_model_path}.opt'))
        return

    def get_data_loaders(self):
        data_dir = self.args.data_dir
        # self.train_dataset = PickleDataset(os.path.join(data_dir, f'{self.args.train_set}.pkl'),
        #         os.path.join(data_dir, self.args.train_index_file),
        #         segment_size=self.config['data_loader']['segment_size'])
        # self.train_loader = get_data_loader(self.train_dataset,
        #         frame_size=self.config['data_loader']['frame_size'],
        #         batch_size=self.config['data_loader']['batch_size'],
        #         shuffle=self.config['data_loader']['shuffle'],
        #         num_workers=4, drop_last=False)
        # self.train_iter = infinite_iter(self.train_loader)

        train_transcripts_path = os.path.join(data_dir, self.args.train_transcripts_file)
        speaker_info_path = os.path.join(data_dir, self.args.speaker_info_file)

################################################################################### --- >MRH: change PickleDataset to PickleDataset2
        # self.train_dataset = PickleDataset(
        #     pickle_path=os.path.join(data_dir, f'{self.args.train_set}.pkl'),
        #     sample_index_path=os.path.join(data_dir, self.args.train_index_file),
        #     segment_size=self.config['data_loader']['segment_size'],
        #     transcripts_path=train_transcripts_path, # NEW: Pass transcripts path
        #     speaker_info_path=speaker_info_path # NEW: Pass speaker info path
        # )


        # Inside solver.py initialization:
        # self.train_dataset = PickleDataset2(
        #     pickle_path=os.path.join(data_dir, f'{self.args.train_set}.pkl'),
        #     # sample_index_path=os.path.join(data_dir, self.args.train_index_file), # <-- REMOVE THIS LINE
        #     segment_size=self.config['data_loader']['segment_size'],
        #     transcripts_path=train_transcripts_path,
        #     speaker_info_path=speaker_info_path
        # )
        print(f"\n--- DEBUG: attr_path value BEFORE PickleDataset3 instantiation ---")
        print(f"Path to attr.pkl: {self.args.attr}")
        print(f"Type of attr.pkl path: {type(self.args.attr)}")
        print(f"Is attr.pkl path None? {self.args.attr is None}")
        print(f"------------------")

        self.train_dataset = PickleDataset3(
            pickle_path=os.path.join(data_dir, f'{self.args.train_set}.pkl'),
            segment_size=self.config['data_loader']['segment_size'],
            transcripts_path=train_transcripts_path,
            speaker_info_path=speaker_info_path,
            attr_path=self.args.attr # <-- Make sure you pass this new argument!
        )

        self.out_test_dataset = PickleDataset3(
            pickle_path=os.path.join(data_dir, f'{self.args.out_test}.pkl'),
            segment_size=self.config['data_loader']['segment_size'],
            transcripts_path=train_transcripts_path,
            speaker_info_path=speaker_info_path,
            attr_path=self.args.attr # <-- Make sure you pass this new argument!
        )

        self.in_test_dataset = PickleDataset3(
            pickle_path=os.path.join(data_dir, f'{self.args.in_test}.pkl'),
            segment_size=self.config['data_loader']['segment_size'],
            transcripts_path=train_transcripts_path,
            speaker_info_path=speaker_info_path,
            attr_path=self.args.attr # <-- Make sure you pass this new argument!
        )
###################################################################################

        # Your get_data_loader already handles CollateFn internally, so no change here!
        self.train_loader = get_data_loader(
            dataset=self.train_dataset,
            frame_size=self.config['data_loader']['frame_size'], # This gets passed to CollateFn
            batch_size=self.config['data_loader']['batch_size'],
            shuffle=self.config['data_loader']['shuffle'],
            num_workers=self.config['data_loader'].get('num_workers', 4),
            drop_last=self.config['data_loader'].get('drop_last', False)
        )
        self.train_iter = infinite_iter(self.train_loader)


        self.out_test_loader = get_data_loader(
            dataset=self.out_test_dataset,
            frame_size=self.config['data_loader']['frame_size'], # This gets passed to CollateFn
            batch_size=self.config['data_loader']['batch_size'],
            shuffle=self.config['data_loader']['shuffle'],
            num_workers=self.config['data_loader'].get('num_workers', 4),
            drop_last=self.config['data_loader'].get('drop_last', False)
        )
        self.out_test_iter = infinite_iter(self.out_test_loader)



        self.in_test_loader = get_data_loader(
            dataset=self.in_test_dataset,
            frame_size=self.config['data_loader']['frame_size'], # This gets passed to CollateFn
            batch_size=self.config['data_loader']['batch_size'],
            shuffle=self.config['data_loader']['shuffle'],
            num_workers=self.config['data_loader'].get('num_workers', 4),
            drop_last=self.config['data_loader'].get('drop_last', False)
        )
        self.in_test_iter = infinite_iter(self.in_test_loader)

        return

    def build_model(self):
        # create model, discriminator, optimizers
        # print('---')
        # self.model = cc(AE1(self.config))
        self.model = cc(AE2(self.args)) # MRH: change AE1 to AE2
        # print('***')
        # print(self.model)
        optimizer = self.config['optimizer']
        self.opt = torch.optim.Adam(self.model.parameters(),
                lr=optimizer['lr'], betas=(optimizer['beta1'], optimizer['beta2']),
                amsgrad=optimizer['amsgrad'], weight_decay=optimizer['weight_decay'])
        # print(self.opt)
        return
#################################################################
    def mrh_AttributeData_prepare(self):
        # --- Derived from your specific dataset metadata ---

        # 1. Speaker ID Mapping (Extract 'pXXX' from the 'ID' column)
        # All unique IDs from your data:
        # 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 236, 237, 238, 239, 240, 241,
        # 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258,
        # 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274,
        # 275, 276, 277, 278, 279, 281, 282, 283, 284, 285, 286, 287, 288, 292, 293, 294,
        # 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312,
        # 313, 314, 315, 316, 317, 318, 323, 326, 329, 330, 333, 334, 335, 336, 339, 340,
        # 341, 343, 345, 347, 351, 360, 361, 362, 363, 364, 374, 376

        all_unique_speaker_ids_extracted = sorted([f'p{id_}' for id_ in [
            225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 236, 237, 238, 239, 240, 241,
            243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258,
            259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274,
            275, 276, 277, 278, 279, 281, 282, 283, 284, 285, 286, 287, 288, 292, 293, 294,
            295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312,
            313, 314, 315, 316, 317, 318, 323, 326, 329, 330, 333, 334, 335, 336, 339, 340,
            341, 343, 345, 347, 351, 360, 361, 362, 363, 364, 374, 376
        ]])
        self.speaker_id_to_int = {speaker_id: i for i, speaker_id in enumerate(all_unique_speaker_ids_extracted)}
        self.num_unique_speakers = len(self.speaker_id_to_int) # Should be 104

        # 2. Gender Mapping
        self.all_unique_genders = sorted(['F', 'M'])
        self.gender_to_int = {g: i for i, g in enumerate(self.all_unique_genders)}
        self.gender_types_count = len(self.gender_to_int) # Should be 2

        # 3. Age Mapping (Map raw age integers to 0-indexed integers)
        # All unique ages from your data:
        # 23, 22, 38, 21, 20, 25, 24, 19, 26, 33, 32, 28, 27, 18, 29
        self.all_unique_age_values = sorted(list(set([
            23, 22, 38, 21, 20, 25, 24, 19, 26, 33, 32, 28, 27, 18, 29
        ])))
        self.age_to_int = {a: i for i, a in enumerate(self.all_unique_age_values)}
        self.age_types_count = len(self.age_to_int) # Should be 15

        # 4. Accent Mapping
        # All unique accents from your data:
        # 'English', 'Scottish', 'NorthernIrish', 'Irish', 'Indian', 'Welsh', 'American',
        # 'Canadian', 'SouthAfrican', 'NewZealand English', 'Australian English'
        self.all_unique_accents = sorted(list(set([
            'English', 'Scottish', 'NorthernIrish', 'Irish', 'Indian', 'Welsh', 'American',
            'Canadian', 'SouthAfrican', 'NewZealand', 'Australian', 'Indian'
        ])))
        self.accent_to_int = {a: i for i, a in enumerate(self.all_unique_accents)}
        self.accent_types_count = len(self.accent_to_int) # Should be 11

        # --- You can now use these variables in your initialization code: ---
        # (Example using these variable names)

        # speaker_id_to_int
        # gender_to_int
        # age_to_int
        # accent_to_int

        # num_unique_speakers
        # gender_types_count
        # age_types_count
        # accent_types_count

        # --- Example of how to print them to verify ---
        print("speaker_id_to_int:", self.speaker_id_to_int)
        print("num_unique_speakers:", self.num_unique_speakers)
        print("gender_to_int:", self.gender_to_int)
        print("gender_types_count:", self.gender_types_count)
        print("age_to_int:", self.age_to_int)
        print("age_types_count:", self.age_types_count)
        print("accent_to_int:", self.accent_to_int)
        print("accent_types_count:", self.accent_types_count)
        ##############################################################


# Assuming 'cc' function is available (e.g., self.cc(tensor))
# And self.device, self.model, self._to_int mappings, and self.text_encoder/projection are initialized.



    # Assuming 'cc' function is available (e.g., self.cc(tensor))
    # And self.device, self.model, self._to_int mappings, and self.text_encoder/projection are initialized.

    # Helper to get the latent tensor from the dictionary output of your model
    def _get_first_level_latent(latent_dict):
        # Assumes latent_dict['mus'] is a list of tensors, we take the first one
        return latent_dict['mus'][0]

    def evaluate_disentanglement_metrics(self, eval_dataloader, latent_dim=128):
        """
        Evaluates disentanglement metrics by training and testing probes on extracted latent spaces.
        The probes are trained on a subset of the collected latents and evaluated on another subset
        to provide a more realistic assessment of disentanglement.

        Args:
            self: The main model/trainer instance, expected to have attributes like
                  self.model, self.device, self.speaker_id_to_int, etc.
            eval_dataloader: DataLoader providing evaluation data (audio segments, metadata).
                            Crucially, this should ideally be a *separate* validation/test set
                            distinct from the training data.
            latent_dim: The dimensionality of the latent spaces (used for probe hidden layers).

        Returns:
            A dictionary of disentanglement metrics.
        """
        self.model.eval() # Set main model to evaluation mode

        all_speaker_latents = []
        all_content_latents = []

        all_speaker_ids_gt = [] # Ground Truth
        all_genders_gt = []
        all_ages_gt = []
        all_accents_gt = []
        all_transcription_embeddings_gt = []

        # Suppress ConvergenceWarning during probe training, as they might not fully converge
        # but still provide useful insight into separability.
        warnings.filterwarnings("ignore", category=ConvergenceWarning)

        # Collect latents and ground truth labels from a portion of the evaluation dataset
        print("Collecting latent representations for disentanglement evaluation...")
        with torch.no_grad(): # No gradient calculation for evaluation
            for batch_idx, batch_data in enumerate(eval_dataloader):
                # Unpack batch data (adjust according to your DataLoader's output structure)
                # Ensure the order matches your DataLoader's __getitem__
                segment, utt_id, transcription, gender, age, accent, t = batch_data

                # Move segment to device (assuming cc handles this correctly)
                segment = cc(segment) # Using self.cc instead of global cc

                # --- Preprocess auxiliary data (similar to ae_step) ---
                speaker_ids_for_batch = [uid.split('_')[0] for uid in utt_id]
                speaker_ids_int = torch.tensor(
                    [self.speaker_id_to_int[speaker_uid] for speaker_uid in speaker_ids_for_batch],
                    dtype=torch.long, device=self.device
                )
                gender_int = torch.tensor(
                    [self.gender_to_int[g] for g in gender],
                    dtype=torch.long, device=self.device
                )
                age_int = torch.tensor(
                    [self.age_to_int[a] for a in age],
                    dtype=torch.long, device=self.device
                )
                accent_int = torch.tensor(
                    [self.accent_to_int[a] for a in accent],
                    dtype=torch.long, device=self.device
                )

                # Ensure text_encoder and text_encoder_projection are available and correctly used
                raw_text_embeddings = self.text_encoder.encode(
                    transcription, convert_to_tensor=True, show_progress_bar=False
                )
                transcription_embeddings = self.text_encoder_projection(raw_text_embeddings).to(self.device)

                # Forward pass through AE model
                output = self.model(segment)
                speaker_latents_batch = _get_first_level_latent(output['speaker_latents'])
                content_latents_batch = _get_first_level_latent(output['content_latents'])

                # Store on CPU for scikit-learn processing (important for performance and compatibility)
                all_speaker_latents.append(speaker_latents_batch.cpu())
                all_content_latents.append(content_latents_batch.cpu())

                all_speaker_ids_gt.append(speaker_ids_int.cpu())
                all_genders_gt.append(gender_int.cpu())
                all_ages_gt.append(age_int.cpu())
                all_accents_gt.append(accent_int.cpu())
                all_transcription_embeddings_gt.append(transcription_embeddings.cpu())

                # Optional: Limit evaluation data size for faster checks during development.
                # Using too few batches can lead to unstable probe results. Aim for 100-200 batches if possible.
                if batch_idx >= 100: # Evaluate on 100 batches to get a reasonable sample size
                    break
        print(f"Collected data from {batch_idx + 1} batches.")

        # Concatenate all collected data and convert to numpy for scikit-learn
        X_s = torch.cat(all_speaker_latents).numpy()
        X_c = torch.cat(all_content_latents).numpy()

        y_speaker_id = torch.cat(all_speaker_ids_gt).numpy()
        y_gender = torch.cat(all_genders_gt).numpy()
        y_age = torch.cat(all_ages_gt).numpy()
        y_accent = torch.cat(all_accents_gt).numpy()
        y_transcription_emb = torch.cat(all_transcription_embeddings_gt).numpy()

        # Handle cases where collected data might be empty (e.g., if dataloader is exhausted too soon)
        if X_s.shape[0] == 0 or X_c.shape[0] == 0:
            print("Warning: No data collected for disentanglement metrics. Returning empty metrics.")
            self.model.train()
            return {}

        # Apply StandardScaler to the full collected data *before* splitting for probes
        scaler_s = StandardScaler()
        X_s_scaled = scaler_s.fit_transform(X_s)
        scaler_c = StandardScaler()
        X_c_scaled = scaler_c.fit_transform(X_c)

        metrics = {}
        PROBE_TEST_SIZE = 0.3 # Use 30% of collected data for probe evaluation
        PROBE_RANDOM_STATE = 42 # For reproducibility of probe training/testing splits

        # --- 1. Speaker Latent Space Evaluation (s_latent) ---
        # Expected: High accuracy for speaker-related attributes, low R2 for content.

        print("\n--- Evaluating Speaker Latent Space ---")

        # 1.1 Speaker ID Classification (Desired: HIGH accuracy)
        # Split the scaled speaker latents and ground truth speaker IDs for probe training/testing
        X_s_train_id, X_s_test_id, y_speaker_id_train, y_speaker_id_test = train_test_split(
            X_s_scaled, y_speaker_id, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE,
            stratify=y_speaker_id # Stratify to maintain class distribution in splits
        )
        probe_s_speaker_id = MLPClassifier(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
        probe_s_speaker_id.fit(X_s_train_id, y_speaker_id_train) # Train probe
        metrics['speaker_latent_speaker_id_accuracy'] = probe_s_speaker_id.score(X_s_test_id, y_speaker_id_test) # Evaluate probe

        # 1.2 Gender Classification (Desired: HIGH accuracy)
        X_s_train_gender, X_s_test_gender, y_gender_train, y_gender_test = train_test_split(
            X_s_scaled, y_gender, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_gender
        )
        probe_s_gender = MLPClassifier(hidden_layer_sizes=(latent_dim // 2,), max_iter=300, random_state=PROBE_RANDOM_STATE)
        probe_s_gender.fit(X_s_train_gender, y_gender_train)
        metrics['speaker_latent_gender_accuracy'] = probe_s_gender.score(X_s_test_gender, y_gender_test)

        # 1.3 Age Classification (Desired: HIGH accuracy)
        X_s_train_age, X_s_test_age, y_age_train, y_age_test = train_test_split(
            X_s_scaled, y_age, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_age
        )
        probe_s_age = MLPClassifier(hidden_layer_sizes=(latent_dim // 2,), max_iter=300, random_state=PROBE_RANDOM_STATE)
        probe_s_age.fit(X_s_train_age, y_age_train)
        metrics['speaker_latent_age_accuracy'] = probe_s_age.score(X_s_test_age, y_age_test)

        # 1.4 Accent Classification (Desired: HIGH accuracy)
        X_s_train_accent, X_s_test_accent, y_accent_train, y_accent_test = train_test_split(
            X_s_scaled, y_accent, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_accent
        )
        probe_s_accent = MLPClassifier(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
        probe_s_accent.fit(X_s_train_accent, y_accent_train)
        metrics['speaker_latent_accent_accuracy'] = probe_s_accent.score(X_s_test_accent, y_accent_test)

        # 1.5 Content Prediction (Transcription Embedding) (Desired: LOW/NEGATIVE R2)
        # R2 near 0 or negative indicates the latent space does not encode this information well.
        X_s_train_content_reg, X_s_test_content_reg, y_transcription_emb_train_s, y_transcription_emb_test_s = train_test_split(
            X_s_scaled, y_transcription_emb, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE
        )
        probe_s_content_reg = LinearRegression() # Simpler model to check for linear dependency
        probe_s_content_reg.fit(X_s_train_content_reg, y_transcription_emb_train_s)
        metrics['speaker_latent_content_R2'] = probe_s_content_reg.score(X_s_test_content_reg, y_transcription_emb_test_s)


        # --- 2. Content Latent Space Evaluation (c_latent) ---
        # Expected: Low accuracy for speaker-related attributes, high R2 for content.

        print("\n--- Evaluating Content Latent Space ---")

        # 2.1 Speaker ID Classification (Desired: LOW accuracy, ideally close to random chance)
        X_c_train_id, X_c_test_id, y_speaker_id_train_c, y_speaker_id_test_c = train_test_split(
            X_c_scaled, y_speaker_id, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE, stratify=y_speaker_id
        )
        probe_c_speaker_id = MLPClassifier(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
        probe_c_speaker_id.fit(X_c_train_id, y_speaker_id_train_c)
        metrics['content_latent_speaker_id_accuracy'] = probe_c_speaker_id.score(X_c_test_id, y_speaker_id_test_c)

        # 2.2 Transcription Embedding Prediction (Desired: HIGH R2)
        X_c_train_transcription_reg, X_c_test_transcription_reg, y_transcription_emb_train_c, y_transcription_emb_test_c = train_test_split(
            X_c_scaled, y_transcription_emb, test_size=PROBE_TEST_SIZE, random_state=PROBE_RANDOM_STATE
        )
        probe_c_transcription_reg = MLPRegressor(hidden_layer_sizes=(latent_dim,), max_iter=300, random_state=PROBE_RANDOM_STATE)
        probe_c_transcription_reg.fit(X_c_train_transcription_reg, y_transcription_emb_train_c)
        metrics['content_latent_transcription_R2'] = probe_c_transcription_reg.score(X_c_test_transcription_reg, y_transcription_emb_test_c)

        # Re-enable warnings after probe training
        warnings.filterwarnings("default", category=ConvergenceWarning)

        self.model.train() # Set main model back to training mode
        return metrics

    ########################################################
    # Assuming 'model' is an instance of AE and 'batch' contains input 'x'
    # This should be part of your training loop
    # def ae_step(model, x):
    # def ae_step(self,x,lambda_kl):
    #     x = cc(x)
    #     output = self.model(x)
    #     dec = output['dec']
    #     speaker_latents = output['speaker_latents']
    #     content_latents = output['content_latents']

    #     # print('speaker_latents shape = ',speaker_latents.shape)
    #     # print('content_latents shape = ',content_latents.shape)
    #     # raise Exception("MRH")


    #     #
    #     loss_components = self.ae_loss_calculator.loss_calculate(x, dec, speaker_latents, content_latents, lambda_kl)
    #     loss= loss_components['total_loss']

    #     ########################################################## MRH: speaker loss
    #             # Calculate speaker-related losses
    #     # total_speaker_loss, weighted_kld_loss, latent_reg_loss = \
    #     #     self.speaker_loss_calculator.calculate_total_speaker_loss(
    #     #         speaker_latents, lambda_kl
    #     #     )

    #     ######################################################### MRH: Calculate Disentanglement Losses ---
    #     s_inter_level_loss = self.s_disent_loss_calc.inter_level_disentanglement_loss(speaker_latents)
    #     s_intra_level_loss = self.s_disent_loss_calc.intra_level_disentanglement_loss(speaker_latents, target_level=0)

    #     c_inter_level_loss = self.c_disent_loss_calc.inter_level_disentanglement_loss(content_latents)
    #     c_intra_level_loss = self.c_disent_loss_calc.intra_level_disentanglement_loss(content_latents, target_level=0)


    #     #### cross-encoder information loss:
    #     # NEW: Cross-encoder MI loss (no need to specify levels here)
    #     cross_enc_loss = self.cross_enc_mi_loss(speaker_latents, content_latents)

    #     # print('inter_level_loss= ',inter_level_loss)
    #     # print('intra_level_loss= ',intra_level_loss)
    #     loss= loss  +4*c_inter_level_loss + 2*c_intra_level_loss + 4*s_inter_level_loss + 2*s_intra_level_loss + 4*cross_enc_loss



    #     ###
    #     self.opt.zero_grad()
    #     loss.backward()
    #     grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(),
    #             max_norm=self.config['optimizer']['grad_norm'])
    #     self.opt.step()

    #     ###

    #     return loss_components, output, s_inter_level_loss, s_intra_level_loss, c_inter_level_loss,  c_intra_level_loss , cross_enc_loss
################################# MRH: new ae_step:
    def ae_step2(self, segment: torch.Tensor,
                utt_id: list[str],
                transcription: list[str],
                gender: list[str],
                age: list[int],
                accent: list[str],
                lambda_kl: float):
        """
        Performs one autoencoder training step including all disentanglement and
        mutual information maximization losses.

        Args:
            segment (torch.Tensor): The input audio segment (e.g., mel spectrogram, waveform).
                                    Shape will depend on your data processing.
            utt_id (list[str]): List of utterance IDs for the batch (e.g., ['p329_012.wav', ...]).
            transcription (list[str]): List of raw text transcriptions for the batch.
            gender (list[str]): List of gender strings for the batch (e.g., ['F', 'M', ...]).
            age (list[int]): List of age integers for the batch (e.g., [23, 19, ...]).
            accent (list[str]): List of accent strings for the batch (e.g., ['English', 'American', ...]).
            lambda_kl (float): Weight for the KL divergence term.

        Returns:
            tuple: Contains various loss components and model outputs for logging/monitoring.
        """
        # print('transcription= ',transcription)

        # 1. Move segment to the appropriate device (e.g., GPU)
        segment = cc(segment)

    #         # --- ADD THIS DEBUG PRINT HERE ---
    #     print(f"\n--- DEBUG: Input 'segment' to ae_step2 ---")
    #     print(f"Shape: {segment.shape}")
    #     print(f"Min: {segment.min().item():.4f}, Max: {segment.max().item():.4f}")
    #     print(f"Mean: {segment.mean().item():.4f}, Std: {segment.std().item():.4f}")
    #     print(f"Contains NaN: {segment.isnan().any()}, Contains Inf: {segment.isinf().any()}")
    #     print(f"----------------------------------------")
    # # ---------------------------------

    # 2. Preprocess auxiliary data for MI Maximization Losses

        # 2.1 Speaker ID (from utt_id string to integer tensor)
        # The 'speaker_ids_int =' line is indented by 4 spaces
        speaker_ids_for_batch = [uid.split('_')[0] for uid in utt_id]
        speaker_ids_int = torch.tensor(
            # This line (the start of the list comprehension) is indented by 8 spaces
            [self.speaker_id_to_int[speaker_uid] for speaker_uid in speaker_ids_for_batch],
            # This line (the dtype/device) is also indented by 8 spaces
            dtype=torch.long, device=self.device
        )

        # 2.2 Gender (from string to integer tensor)
        # print('mrh= ',self.gender_to_int)
        # print('mrh gender = ',gender)
        gender_int = torch.tensor(
            [self.gender_to_int[g] for g in gender],
            dtype=torch.long, device=self.device
        )

        # print('\n\n------ MRH1')

        # 2.3 Age (from raw int to 0-indexed int tensor)
        age_int = torch.tensor(
            [self.age_to_int[a] for a in age],
            dtype=torch.long, device=self.device
        )

        # print('\n--- mrh1')
        # print('accent= ',accent, type(accent), np.shape(accent))

        # 2.4 Accent (from string to integer tensor)
        accent_int = torch.tensor(
            [self.accent_to_int[a] for a in accent],
            dtype=torch.long, device=self.device
        )
        # print('accent= ',accent, type(accent), np.shape(accent))
        # print('\n--- mrh2')

        # 2.5 Transcription (from string list to embedding tensor)
        if self.text_encoder is None or self.text_encoder_projection is None:
            raise RuntimeError("Text encoder not properly initialized. Cannot compute TranscriptionInformationLoss.")

        raw_text_embeddings = self.text_encoder.encode(
            transcription,
            convert_to_tensor=True,
            show_progress_bar=False
        )

        # # Inside ae_step2(), after raw_text_embeddings = self.text_encoder.encode(...)
        # print(f"DEBUG: raw_text_embeddings shape (before proj): {raw_text_embeddings.shape}")
        # print(f"DEBUG: raw_text_embeddings mean (before proj): {raw_text_embeddings.mean().item():.4f}")
        # print(f"DEBUG: raw_text_embeddings std (before proj): {raw_text_embeddings.std().item():.4f}")


        #########################
        raw_text_embeddings_for_projection = raw_text_embeddings.to(self.device).clone().detach()
        raw_text_embeddings_for_projection.requires_grad_(True)

        transcription_embeddings = self.text_encoder_projection(raw_text_embeddings_for_projection)


        # 3. Forward pass through your Autoencoder model
        output = self.model(segment)
        dec = output['dec']
        speaker_latents = output['speaker_latents'] # dict: {'mus': [z_s1, z_s2], 'log_sigmas': [...]}
        content_latents = output['content_latents'] # dict: {'mus': [z_c1, z_c2], 'log_sigmas': [...]}
        # print('content encoder log sigma = ',output['content_latents']['log_sigmas'])
        # print('speaker encoder log sigma = ',output['speaker_latents']['log_sigmas'])

        # print('mrh  dec= ', dec)
        # print('mrh  speaker_latents= ', speaker_latents)
        # print('mrh  content_latents= ', content_latents)


        # # Inside ae_step2, right before loss_components = self.ae_loss_calculator.loss_calculate(...)
        # print('\n--- Debugging Pre-Loss Tensors ---')
        # print(f"dec contains NaN: {dec.isnan().any()}")
        # print(f"dec contains Inf: {dec.isinf().any()}")
        # print(f"dec min: {dec.min().item():.4f}, max: {dec.max().item():.4f}, mean: {dec.mean().item():.4f}, std: {dec.std().item():.4f}")

        # for i, (mu, log_sigma) in enumerate(zip(speaker_latents['mus'], speaker_latents['log_sigmas'])):
        #     print(f"Speaker latent level {i} mu contains NaN: {mu.isnan().any()}")
        #     print(f"Speaker latent level {i} mu contains Inf: {mu.isinf().any()}")
        #     print(f"Speaker latent level {i} mu min: {mu.min().item():.4f}, max: {mu.max().item():.4f}, mean: {mu.mean().item():.4f}, std: {mu.std().item():.4f}")
        #     print(f"Speaker latent level {i} log_sigma contains NaN: {log_sigma.isnan().any()}")
        #     print(f"Speaker latent level {i} log_sigma contains Inf: {log_sigma.isinf().any()}")
        #     print(f"Speaker latent level {i} log_sigma min: {log_sigma.min().item():.4f}, max: {log_sigma.max().item():.4f}, mean: {log_sigma.mean().item():.4f}, std: {log_sigma.std().item():.4f}")
        #     # Also check priors if they are dynamic:
        #     if 'mu_priors' in speaker_latents and speaker_latents['mu_priors'] and i < len(speaker_latents['mu_priors']):
        #         mu_prior = speaker_latents['mu_priors'][i]
        #         log_sigma_prior = speaker_latents['log_sigma_priors'][i]
        #         print(f"Speaker latent level {i} mu_prior contains NaN: {mu_prior.isnan().any()}")
        #         print(f"Speaker latent level {i} log_sigma_prior contains NaN: {log_sigma_prior.isnan().any()}")


        # for i, (mu, log_sigma) in enumerate(zip(content_latents['mus'], content_latents['log_sigmas'])):
        #     print(f"Content latent level {i} mu contains NaN: {mu.isnan().any()}")
        #     print(f"Content latent level {i} mu contains Inf: {mu.isinf().any()}")
        #     print(f"Content latent level {i} mu min: {mu.min().item():.4f}, max: {mu.max().item():.4f}, mean: {mu.mean().item():.4f}, std: {mu.std().item():.4f}")
        #     print(f"Content latent level {i} log_sigma contains NaN: {log_sigma.isnan().any()}")
        #     print(f"Content latent level {i} log_sigma contains Inf: {log_sigma.isinf().any()}")
        #     print(f"Content latent level {i} log_sigma min: {log_sigma.min().item():.4f}, max: {log_sigma.max().item():.4f}, mean: {log_sigma.mean().item():.4f}, std: {log_sigma.std().item():.4f}")
        #     # Also check priors if they are dynamic:
        #     if 'mu_priors' in content_latents and content_latents['mu_priors'] and i < len(content_latents['mu_priors']):
        #         mu_prior = content_latents['mu_priors'][i]
        #         log_sigma_prior = content_latents['log_sigma_priors'][i]
        #         print(f"Content latent level {i} mu_prior contains NaN: {mu_prior.isnan().any()}")
        #         print(f"Content latent level {i} log_sigma_prior contains NaN: {log_sigma_prior.isnan().any()}")

        print('----------------------------')


        # 4. Calculate existing losses (Reconstruction, KL-Divergence, MI Minimization)
        loss_components = self.ae_loss_calculator.loss_calculate(segment, dec, speaker_latents, content_latents, lambda_kl)
        total_loss = loss_components['total_loss'] # Initialize total_loss with base AE loss

        # print('loss_components["recon_loss"]= ', loss_components['recon_loss'].item())
        # print('\n\n------------- MRH2')
#
        # raise Exception ('MRH')

        # Speaker Disentanglement (Inter/Intra-level MI Minimization)
        s_inter_level_loss = self.s_disent_loss_calc.inter_level_disentanglement_loss(speaker_latents)
        s_intra_level_loss = self.s_disent_loss_calc.intra_level_disentanglement_loss(speaker_latents, target_level=0)

        # Content Disentanglement (Inter/Intra-level MI Minimization)
        c_inter_level_loss = self.c_disent_loss_calc.inter_level_disentanglement_loss(content_latents)
        c_intra_level_loss = self.c_disent_loss_calc.intra_level_disentanglement_loss(content_latents, target_level=0)

        # Cross-Encoder Mutual Information Loss (Speaker vs. Content MI Minimization)
        cross_enc_loss = self.cross_enc_mi_loss(speaker_latents, content_latents)

        # Add these existing losses to the total
        total_loss += 2 * c_inter_level_loss + 1 * c_intra_level_loss \
                    + 2 * s_inter_level_loss + 1 * s_intra_level_loss \
                    + 8 * cross_enc_loss # Adjust weights (e.g., 4) as per your setup


        # 5. Calculate NEW Mutual Information Maximization Losses

        # 5.1 Speaker Identity Information Loss (maximize MI between speaker latent and speaker ID)
        loss_speaker_id_mi = self.speaker_id_loss_calc(speaker_latents, speaker_ids_int)
        total_loss += 2* loss_speaker_id_mi # Add to total loss

        # 5.2 Transcription Information Loss (maximize MI between content latent and transcription embedding)
        ##################to debug
# Inside ae_step2(), just before calculating loss_transcription_mi:

        # print('\n\n------------- MRH3')

        # 1. Check transcription_embeddings properties
        # print(f"transcription_embeddings shape: {transcription_embeddings.shape}")
        # print(f"transcription_embeddings mean: {transcription_embeddings.mean().item():.4f}")
        # print(f"transcription_embeddings std: {transcription_embeddings.std().item():.4f}")
        # Check if embeddings are collapsing or are all zeros/nans
        if transcription_embeddings.isnan().any() or transcription_embeddings.isinf().any():
            print("WARNING: transcription_embeddings contain NaN/Inf!")
        if transcription_embeddings.sum() == 0:
            print("WARNING: transcription_embeddings are all zeros!")

        # 2. Check content_latents properties (the ones used by transcription_loss_calc)
        content_latent_mu = content_latents['mus'][self.transcription_loss_calc.target_content_level]
        # print(f"content_latent_mu shape: {content_latent_mu.shape}")
        # print(f"content_latent_mu mean: {content_latent_mu.mean().item():.4f}")
        # print(f"content_latent_mu std: {content_latent_mu.std().item():.4f}")
        if content_latent_mu.isnan().any() or content_latent_mu.isinf().any():
            print("WARNING: content_latent_mu contain NaN/Inf!")

        # 3. Check effective KL regularization for content

        # print('\n\n------------- MRH4')


        loss_transcription_mi = self.transcription_loss_calc(content_latents, transcription_embeddings)
        total_loss += 16*loss_transcription_mi # Add to total loss




        #################################################### To Debug
        # 3. Check effective KL regularization for content
        effective_content_kl_loss = loss_components['content_kld_loss'] * lambda_kl
        # print(f"Effective content_loss_kl: {effective_content_kl_loss.item():.4f}")
        # print(f"Raw loss_transcription_mi (before weighting): {loss_transcription_mi.item():.4f}")

        # 4. What is the output dimension of your ContentEncoder?
        # You should know this from your config, but just to be sure:
        # print(f"ContentEncoder output dimension (c_out): {self.config['ContentEncoder']['c_out']}")
        ###########################################################

        # print('\n\n------------- MRH5')




        # 5.3 Speaker Attribute Information Loss (maximize MI between speaker latent and various attributes)
        loss_speaker_attribute_mi = self.speaker_attribute_loss_calc(speaker_latents, gender_int, age_int, accent_int)
        total_loss += 4*loss_speaker_attribute_mi # Add to total loss


        # print('\n\n------------- MRH6')

        # 6. Optimizer step (standard PyTorch training loop mechanics)
        self.opt.zero_grad()
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(),
                max_norm=self.config['optimizer']['grad_norm'])
        self.opt.step()


        # print('\n\n------------- MRH7')


        # 7. Return all relevant loss components for logging, monitoring, etc.
        return loss_components, output,s_inter_level_loss,s_intra_level_loss, c_inter_level_loss, c_intra_level_loss, cross_enc_loss, loss_speaker_id_mi, loss_transcription_mi, loss_speaker_attribute_mi # NEW: Speaker Attribute MI maximization




        # return {
        #     'recon_loss': recon_loss,
        #     'speaker_kld_loss': speaker_kld_loss,
        #     'content_kld_loss': content_kld_loss
        #     # You might add other relevant values here if they are directly calculated
        #     # and consistently available within ae_step (e.g., total_loss and grad_norm
        #     # if ae_step also handled optimization, but we decided against that earlier).
        # }
################################################################################
    def train(self, n_iterations):
                # Initialize lists for storing metrics
        batch_mcd = 0.0
        batch_mcd_dtw = 0.0
        batch_lsd = 0.0
        pesq_score = 0.0 # Initialize
        stoi_score = 0.0 # Initialize

        # Initialize ASR metrics to 0.0 before the loop
        source_wer_mel = 0.0
        source_cer_mel = 0.0
        output_wer_mel = 0.0
        output_cer_mel = 0.0
        source_wer_wave = 0.0
        source_cer_wave = 0.0
        output_wer_wave = 0.0
        output_cer_wave = 0.0
        source_wer_wav2vec2 = 0.0
        source_cer_wav2vec2 = 0.0
        target_wer_wav2vec2 = 0.0
        target_cer_wav2vec2 = 0.0

        # Speaker verification metrics
        batch_eer = 0.0
        batch_roc_auc = 0.0


        all_embeddings = []
        all_speaker_ids = []

        for iteration in range(n_iterations):


            # print('mrh 2')
            # print('iteration= ',iteration)
            if iteration >= self.config['annealing_iters']:
                lambda_kl = self.config['lambda']['lambda_kl']
            else:
                lambda_kl = self.config['lambda']['lambda_kl'] * (iteration + 1) / self.config['annealing_iters']
            # data = next(self.train_iter)
            segment, utt_id, transcription, gender, age, accent, t = next(self.train_iter)

            # print('segment Type & shape  = ',type(segment), segment.shape)
            # print('utt_id & shape & value= ',type(utt_id), np.shape(utt_id), utt_id)
            # print('transcription & shape & value = ',type(transcription), np.shape(transcription), transcription)
            # print('gender & shape & value = ',type(gender), np.shape(gender), gender)
            # print('age & shape & value= ',type(age), np.shape(age), age)
            # print('accent & shape & value = ',type(accent), np.shape(accent), accent)




            # raise Exception("MRH")


            ######################################################### MRH: change
            # meta = self.ae_step(data, lambda_kl)
            # meta , output, s_inter_level_loss, s_intra_level_loss, c_inter_level_loss,  c_intra_level_loss , cross_enc_loss= self.ae_step(segment,lambda_kl)
            # meta , output, s_inter_level_loss, s_intra_level_loss, c_inter_level_loss,  c_intra_level_loss , cross_enc_loss, loss_speaker_id_mi,loss_transcription_mi, loss_speaker_attribute_mi= self.ae_step2(segment,utt_id, transcription, gender, age, accent,lambda_kl)

            meta, output, s_inter_level_loss, \
            s_intra_level_loss, c_inter_level_loss, \
            c_intra_level_loss, cross_enc_loss, \
            loss_speaker_id_mi, loss_transcription_mi, \
            loss_speaker_attribute_mi = self.ae_step2(
                segment, utt_id, transcription, gender,
                age, accent, lambda_kl
            )




            ############################################################################## MRH: to check output of speaker and content
            #             # --- ADDED DEBUGGING PRINTS ---
            # print(f"\n--- Debugging Iteration {iteration+1} ---")
            # print(f"Output dictionary keys: {output.keys()}")

            # speaker_latents_dict = output.get('speaker_latents') # Use .get to avoid KeyError
            # content_latents_from_model = output.get('content_latents')

            # print(f"Type of speaker_latents_dict: {type(speaker_latents_dict)}")
            # if isinstance(speaker_latents_dict, dict):
            #     print(f"Keys in speaker_latents_dict: {speaker_latents_dict.keys()}")
            #     for key, val in speaker_latents_dict.items():
            #         print(f"  Speaker latent '{key}': Type={type(val)}, Value={val}")
            #         if isinstance(val, list):
            #             for i, item in enumerate(val):
            #                 print(f"    Item {i} in list: Type={type(item)}, Value={'Tensor' if isinstance(item, torch.Tensor) else item}")
            #                 if isinstance(item, torch.Tensor):
            #                     print(f"      Shape: {item.shape}, Device: {item.device}")
            #         elif isinstance(val, torch.Tensor):
            #             print(f"    Shape: {val.shape}, Device: {val.device}")
            # elif speaker_latents_dict is None:
            #     print("speaker_latents_dict is None.")
            # else:
            #     print(f"speaker_latents_dict is not a dictionary: {speaker_latents_dict}")

            # print(f"\nType of content_latents_from_model: {type(content_latents_from_model)}")
            # if isinstance(content_latents_from_model, torch.Tensor):
            #     print(f"Shape of content_latents_from_model: {content_latents_from_model.shape}")
            #     print(f"Device of content_latents_from_model: {content_latents_from_model.device}")
            # elif content_latents_from_model is None:
            #     print("content_latents_from_model is None.")
            # else:
            #     print(f"content_latents_from_model is not a Tensor: {content_latents_from_model}")
            # print(f"--- End Debugging Iteration {iteration+1} ---")
            # # --- END ADDED DEBUGGING PRINTS --
            # # raise Exception('MRH')



            ################################################################################


            # print('meta type = ',type(meta))
            # print('mrh 4')
            # add to logger
            if iteration % self.args.summary_steps == 0:
                # print('mrh 5')
                self.logger.scalars_summary(f'{self.args.tag}/ae_train', meta, iteration)
                # print('mrh 6')
            # loss_rec = meta['loss_rec']
            loss_rec = meta['recon_loss']
            # print('mrh 7')
            # loss_kl = meta['loss_kl']
            speaker_loss_kl = meta['speaker_kld_loss']
            content_loss_kl = meta['content_kld_loss']
            # print('mrh 8')

            # print(f'AE:[{iteration + 1}/{n_iterations}], loss_rec={loss_rec:.2f}, '
                    # f'loss_kl={loss_kl:.2f}, lambda={lambda_kl:.1e}     ', end='\r')


            if (iteration + 1) % self.args.save_steps == 0 or iteration + 1 == n_iterations:
                self.save_model(iteration=iteration)
                print()

            ############################################################################# MRH: change

            # if (iteration + 1) % 1 == 0 :
            #     self.model.eval() # Still good practice to put model in eval mode for this

            #     # print('1')
            #     audio_data_batch = segment # Use the current batch's audio data
            #     speaker_ids_batch = utt_id                 # Use the current batch's speaker IDs



            #     # print('2')
            #     # with torch.no_grad():
            #     audio_data_batch = cc(audio_data_batch)
            #     # print('audio_data_batch shape =',audio_data_batch.shape)
            #     model_output = self.model(audio_data_batch)
            #     # print('22')
            #     embeddings_batch = model_output['speaker_latents']['decoder_input']

            #     # Collect individual embeddings and speaker IDs from the current batch
            #     # print('3')
            #     for i in range(embeddings_batch.size(0)):
            #         all_embeddings.append(embeddings_batch[i].cpu())
            #         all_speaker_ids.append(speaker_ids_batch[i])

            #     # print('4')

            #     if (iteration + 1) % 5 == 0 :
            #         try:
            #             # Check if there are enough distinct speakers for a meaningful EER
            #             if len(set(all_speaker_ids)) < 2 or len(all_embeddings) < 2:
            #                 print("Skipping batch-level speaker verification: Not enough unique speakers or samples in batch.")
            #             else:
            #                 eval_results = self.speaker_verifier.evaluate(all_embeddings, all_speaker_ids)
            #                 # Log with a different tag to distinguish from full eval
            #                 self.logger.scalars_summary(f'{self.args.tag}/speaker_verification_batch', {
            #                     'eer': eval_results['eer'],
            #                     'roc_auc': eval_results['roc_auc'],
            #                     'eer_threshold': eval_results['eer_threshold']
            #                 }, iteration)
            #                 print(f"Batch Speaker Verification Results: EER={eval_results['eer']:.4f}, ROC AUC={eval_results['roc_auc']:.4f}")

            #                 all_embeddings = []
            #                 all_speaker_ids = []

            #         except ValueError as e:
            #             print(f"Skipping batch-level speaker verification due to data inadequacy: {e}")

            #     self.model.train()
            #     print("-----------------------------------------------------------------------------------\n")
            # --- End Speaker Verification on Current Batch ---



            # ###################################################### MRH: MCD
            #             # Add to logger and calculate MCD only when summary_steps condition is met
            # if iteration % 1 == 0:
            #     # self.logger.scalars_summary(f'{self.args.tag}/ae_train', meta, iteration)

            #     # --- Start: MCDMetric Calculation Moved Inside Summary Block ---

            #     # Convert PyTorch tensors to NumPy arrays for metric calculation
            #     # Ensure they are on CPU and detached from graph
            #     source_mels = segment.cpu().detach().numpy()
            #     converted_mels = output['dec'].cpu().detach().numpy()

            #     # Calculate MCD for the current batch using the initialized mcd_metric instance
            #     batch_mcd = self.mcd_metric.calculate_mcd(source_mels, converted_mels)


            #     # Calculate MCD-DTW
            #     batch_mcd_dtw = self.mcd_metric.calculate_mcd_dtw(source_mels, converted_mels)

            #     # Calculate lsd
            #     batch_lsd = self.mcd_metric.calculate_lsd(source_mels, converted_mels)

            #     # Log MCD
            #     # self.logger.scalars_summary(f'{self.args.tag}/metrics/mcd', {'mcd': batch_mcd}, iteration)

            #     # Store the calculated MCD value
            #     self.mcd_values_history.append(batch_mcd) # <-- STORE HERE

            #     # Print the average MCD immediately
            #     # print(f"MCD calculated at iteration {iteration}: {batch_mcd:.4f}") # <-- PRINT HERE

            #     # --- End: MCDMetric Calculation Moved Inside Summary Block ---


            # ####################################


            # if iteration % 1 == 0:
            #     ###############
            #     # MRH: remember we do not use the model here we just used the output of model= converted_mels = output['dec']
            #     source_mels = segment
            #     converted_mels = output['dec']
            #     sample_source_wav_data, sample_target_wav_data = self.inferencer.mrh_inference_from_mel_spectograms(source_mels,converted_mels)
            #     # print('type & shape  = ',type(sample_source_wav_data), np.shape(sample_source_wav_data))
            #     # print('type & shape  = ',type(sample_target_wav_data), np.shape(sample_target_wav_data))
            #     # raise Exception("Mohammad Reza Hasanabadi")
            #     # print(' Done ')
            #     pesq_score, stoi_score = self.VQA.calculate_all_quality_metrics(sample_source_wav_data,
            #                                                                sample_target_wav_data,
            #                                                                original_sr=22050
            #     )
            #     # print('pesq_score= ',pesq_score, 'stoi_score= ',stoi_score )

            # # if iteration % 10 == 0:
            # #     print('transcription = ',transcription)



            ################################################## ASR

            if iteration % 10 == 0:

                ###################################### Whisper
                # MRH: to check ASR on input
                # source_transcriptions = self.ASR_Whisper.transcribe(segment, input_type="mel")
                source_transcriptions = self.ASR_Whisper.transcribe(segment, input_type="mel", sampling_rate=22050,language="en")
                source_wer_mel, source_cer_mel = self.ASR_Whisper.evaluate(source_transcriptions, transcription)

                ### MRH: to check ASR on output Mel
                dec = output['dec']
                output_transcriptions = self.ASR_Whisper.transcribe(dec, input_type="mel", sampling_rate=22050, language="en")
                output_wer_mel, output_cer_mel = self.ASR_Whisper.evaluate(output_transcriptions, transcription)

                print('\nsource_transcriptions = ',source_transcriptions )
                print('output_transcriptions = ',output_transcriptions )
                print('transcription = ',transcription )




                #####  MRH: to check ASR on input wave

                # source_wave_transcription = self.ASR_Whisper.transcribe(sample_source_wav_data, input_type="wave", sampling_rate=args1.sampling_rate)
                # source_wer_wave, source_cer_wave = self.ASR_Whisper.evaluate(source_wave_transcription, transcription)
                source_wer_wave, source_cer_wave=0,0

                ######## MRH: to check ASR on output wave
                # output_wave_transcription = self.ASR_Whisper.transcribe(sample_target_wav_data, input_type="wave", sampling_rate=args1.sampling_rate)
                # output_wer_wave, output_cer_wave = self.ASR_Whisper.evaluate(output_wave_transcription, transcription)
                output_wer_wave, output_cer_wave=0,0

                # ############################################### MRH: Wave2Vec
                #     # Transcribe using Wav2Vec2
                # # wav2vec2_sample_source_wav_data = self.ASR_wav2vec2.transcribe(sample_source_wav_data, sampling_rate=args1.sampling_rate)
                # # source_wer_wav2vec2, source_cer_wav2vec2 = self.ASR_wav2vec2.evaluate(wav2vec2_sample_source_wav_data, transcription)
                # source_wer_wav2vec2, source_cer_wav2vec2=0,0

                # # wave2vec2_target_wave_transcription = self.ASR_wav2vec2.transcribe(sample_target_wav_data, sampling_rate=args1.sampling_rate)
                # # target_wer_wav2vec2, target_cer_wav2vec2 = self.ASR_wav2vec2.evaluate(wave2vec2_target_wave_transcription, transcription)
                # target_wer_wav2vec2, target_cer_wav2vec2=0,0

                # print(f"\nWav2Vec2 Raw Audio Results:")
                # for i in range(num_samples):
                #     print(f"  Reference: '{wave_ground_truths[i]}'")
                #     print(f"  Predicted: '{wav2vec2_predictions[i]}'\n")
                # print(f"  WER: {wer_wav2vec2:.4f}")
                # print(f"  CER: {cer_wav2vec2:.4f}")






            ################################################## MRH: Visualize
            # --- START Latent Accumulation and Visualization Block ---
            with torch.no_grad(): # Ensure no gradients are computed for this part
                speaker_latents_dict = output['speaker_latents']
                content_latents_from_model = output['content_latents']

                current_batch_speaker_numeric_ids = []
                for u_id_entry in utt_id:
                    speaker_identifier = u_id_entry.item() if isinstance(u_id_entry, torch.Tensor) else u_id_entry

                    if speaker_identifier not in self.speaker_id_map:
                        self.speaker_id_map[speaker_identifier] = self.next_speaker_id_index
                        self.next_speaker_id_index += 1

                    current_batch_speaker_numeric_ids.append(self.speaker_id_map[speaker_identifier])

                if len(current_batch_speaker_numeric_ids) != segment.size(0):
                    print(f"Warning: Mismatch in processed speaker IDs for iteration {iteration+1}. Expected {segment.size(0)}, Got {len(current_batch_speaker_numeric_ids)}. Skipping latent accumulation for this batch.")
                    continue

                # Accumulate Speaker Latents (if valid batch)
                for level_name, latent_data_for_level in speaker_latents_dict.items():
                    if latent_data_for_level is None:
                        print(f"Warning: Speaker latent data for level '{level_name}' is None. Skipping.")
                        continue

                    if isinstance(latent_data_for_level, list):
                        for i, z_latent in enumerate(latent_data_for_level):
                            if z_latent is not None and isinstance(z_latent, torch.Tensor): # <-- ADDED CHECK HERE
                                self.accumulated_speaker_latents[f'{level_name}_{i+1}'].append(z_latent.cpu().numpy())
                            else:
                                print(f"Warning: z_latent for '{level_name}_{i+1}' is not a valid Tensor. Skipping.")
                    elif isinstance(latent_data_for_level, torch.Tensor): # <-- ADDED CHECK HERE for non-list case
                        self.accumulated_speaker_latents[level_name].append(latent_data_for_level.cpu().numpy())
                    else:
                        print(f"Warning: Unexpected type for speaker latent data '{level_name}': {type(latent_data_for_level)}. Skipping.")

                self.accumulated_speaker_ids.extend(current_batch_speaker_numeric_ids)

                # Accumulate Content Latents
                if content_latents_from_model is not None and isinstance(content_latents_from_model, torch.Tensor): # <-- ADDED CHECK HERE
                    if content_latents_from_model.dim() == 3:
                        content_latents_for_acc = content_latents_from_model.mean(dim=1)
                    else:
                        content_latents_for_acc = content_latents_from_model
                    self.accumulated_content_latents.append(content_latents_for_acc.cpu().numpy())
                    self.accumulated_content_labels.extend([str(t_item) for t_item in transcription])
                else:
                    print(f"Warning: Content latent data is not a valid Tensor. Skipping content latent accumulation.")


            # Plot Latents if it's the plotting interval
            if (iteration + 1) % 10 == 0:
                self.model.eval() # Set model to eval mode before plotting
                self.latent_plotter.plot_latents(
                    iteration + 1,
                    self.accumulated_speaker_latents,
                    self.accumulated_speaker_ids,
                    self.accumulated_content_latents,
                    self.accumulated_content_labels
                )

                self.accumulated_speaker_latents = defaultdict(list)
                self.accumulated_speaker_ids = []
                self.accumulated_content_latents = []
                self.accumulated_content_labels = []

                ###

                self.model.train() # Set model back to train mode
            ##################################################################
            ##################################################
            if (iteration + 1) % 10 == 0:
                self.history['iteration'].append(iteration + 1)
                self.history['loss_rec'].append(loss_rec.item() if isinstance(loss_rec, torch.Tensor) else loss_rec)
                self.history['speaker_loss_kl'].append(speaker_loss_kl.item() if isinstance(speaker_loss_kl, torch.Tensor) else speaker_loss_kl)
                self.history['content_loss_kl'].append(content_loss_kl.item() if isinstance(content_loss_kl, torch.Tensor) else content_loss_kl)
                self.history['lambda_kl'].append(lambda_kl)
                self.history['mcd'].append(batch_mcd)
                self.history['mcd_dtw'].append(batch_mcd_dtw)
                self.history['batch_lsd'].append(batch_lsd)
                self.history['pesq_score'].append(pesq_score)
                self.history['stoi_score'].append(stoi_score)
                self.history['whisper_source_mel_wer'].append(source_wer_mel)
                self.history['whisper_source_mel_cer'].append(source_cer_mel)
                self.history['whisper_output_mel_wer'].append(output_wer_mel)
                self.history['whisper_output_mel_cer'].append(output_cer_mel)
                self.history['whisper_source_wave_wer'].append(source_wer_wave)
                self.history['whisper_source_wave_cer'].append(source_cer_wave)
                self.history['whisper_output_wave_wer'].append(output_wer_wave)
                self.history['whisper_output_wave_cer'].append(output_cer_wave)
                self.history['wave2vec_source_wer'].append(source_wer_wav2vec2)
                self.history['wave2vec_source_cer'].append(source_cer_wav2vec2)
                self.history['wave2vec_target_wer'].append(target_wer_wav2vec2)
                self.history['wave2vec_target_cer'].append(target_cer_wav2vec2)
                self.history['batch_eer'].append(batch_eer)
                self.history['batch_roc_auc'].append(batch_roc_auc)
                self.history['inter_level_loss'].append(s_inter_level_loss)
                self.history['intra_level_loss'].append(s_intra_level_loss)
                self.history['c_inter_level_loss'].append(c_inter_level_loss)
                self.history['c_intra_level_loss'].append(c_intra_level_loss)
                self.history['cross_enc_loss'].append(cross_enc_loss)
                self.history['loss_speaker_id_mi'].append(loss_speaker_id_mi)
                self.history['loss_transcription_mi'].append(loss_transcription_mi)
                self.history['loss_speaker_attribute_mi'].append(loss_speaker_attribute_mi)




            ##################################################
            print(f'AE:[{iteration + 1}/{n_iterations}], loss_rec={meta["recon_loss"]:.2f}, '
                    f'speaker_loss_kl={meta["speaker_kld_loss"]:.2f}, content_loss_kl={meta["content_kld_loss"]:.2f} ,'
                    f'lambda={lambda_kl:.1e}, MCD={batch_mcd:.4f}, MCD_DTW={batch_mcd_dtw:.4f}, batch_lsd={batch_lsd:.4f}'
                    f' pesq_score={pesq_score:.4f}, stoi_score={stoi_score:.4f} '
                    f' whisper source Mel WER: {source_wer_mel:.4f}'
                    f' whisper source Mel CER: {source_cer_mel:.4f}'
                    f' whisper output Mel WER: {output_wer_mel:.4f}'
                    f' whisper output Mel CER: {output_cer_mel:.4f}'
                    f' whisper source Wave WER: {source_wer_wave:.4f}'
                    f' whisper source Wave CER: {source_cer_wave:.4f}'
                    f' whisper output Wave WER: {output_cer_wave:.4f}'
                    f' whisper output Wave CER: {output_cer_wave:.4f}'
                    f' Wave2vec source WER: {source_wer_wav2vec2:.4f}'
                    f' Wave2vec source CER: {source_cer_wav2vec2:.4f}'
                    f' Wave2vec output WER: {target_wer_wav2vec2:.4f}'
                    f' Wave2vec output CER: {target_cer_wav2vec2:.4f}'
                    f' inter_level_loss: {s_inter_level_loss:.4f}'
                    f' intra_level_loss: {s_intra_level_loss:.4f}'
                    f' c_inter_level_loss: {c_inter_level_loss:.4f}'
                    f' c_intra_level_loss: {c_intra_level_loss:.4f}'
                    f' cross_enc_loss: {cross_enc_loss:.4f}'
                    f' loss_speaker_id_mi: {loss_speaker_id_mi:.4f}'
                    f' loss_transcription_mi: {loss_transcription_mi:.4f}'
                    f' loss_speaker_attribute_mi: {loss_speaker_attribute_mi:.4f}'
                    ) # This will print 0.0 if not calculated in this iter.

            # if iteration  % 9 == 0: # e.g., evaluate every 5 epochs
            #     print(f"--- Evaluating Disentanglement Metrics for Epoch {iteration} ---")
            #     print('\nself.in_test_iter= ',self.train_iter)
            #     disentanglement_metrics = self.evaluate_disentanglement_metrics(self.in_test_iter) # Assuming eval_dataloader is defined
            #     for metric_name, value in disentanglement_metrics.items():
            #         print(' mrh metric name= ', f" {metric_name}: {value:.4f}")

            #         # --- IMPORTANT CHANGE HERE ---
            #         # Ensure the key in self.history matches exactly the metric_name from disentanglement_metrics
            #         # Assuming your evaluate_disentanglement_metrics returns keys like 'speaker_latent_speaker_id_accuracy'
            #         if metric_name in self.history: # Check if the key exists to prevent errors
            #             self.history[metric_name].append(value)
            #         else:
            #             print(f"Warning: Metric '{metric_name}' not found in history dictionary. Skipping save.")


        # Log these metrics to TensorBoard/Weights & Biases if you're using them
        # self.writer.add_scalars('Disentanglement_Metrics', disentanglement_metrics, global_step=epoch)


        return

**------------------------------------**

**------------ 16-Run---------------**

**------------------------------------**

In [ ]:
# SC_combinations = [


#     (1, 1),
#     (1, 2),
#     (1, 3),
#     (1, 4),
#     (1, 5),
#     (1, 6),
#     (1, 7),
#     (1, 8),
#     (1, 9),
#     (1, 10),
# ]
    # (2, 2),
    # (3, 3),
    #     (4,4),
    #     (5,5),
    # # Add all your desired combinations here


# SC_combinations = [
#     (1, 1),
#     (2, 1),
#     (3, 1),
#     (4, 1),
#     (5, 1),
#     (6, 1),
#     (7, 1),
#     (8, 1),
#     (9, 1),
#     (10, 1),
# ]

########################## Symmetric Hierarchcial
SC_combinations = [
    (1, 1),
    (2, 2),
    (3, 3),
    (4, 4),
    (5, 5),
    (6, 6),
    (7, 7),
    (8, 8),
    (9, 9),
    (10, 10),
]
# SC_combinations = [
#     (5, 5),

# ]
########################### Non-Symmetric Hierarchcial
# SC_combinations = [
#     (2, 7),
#     (3, 9),
#     (4, 10),
#     (10, 7),
#     (6, 2),
#     (7, 9),
#     (8, 4),
#     (9, 5),
#     (3, 10),
# ]
history_save_base_dir = '/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin/MRHSavedData'
# --- Loop through each combination and run your training block ---
for i, j in SC_combinations :
    print(f'\n\n--------- Number of Levels = {i}{j}----')

    # 1. Create a fresh instance of ARGS1 for each run

    args2 = ARGS2()
    args2.n_speaker_levels=i
    args2.n_content_levels=j
    # print('args1.n_speaker_levels= ',args1.n_speaker_levels)
    # print('args1.n_content_levels= ',args1.n_content_levels)

    config = args2.config

    # Now, pass this 'config' dictionary to your Solver
    solver = Solver5(config=config, args=args2) ################### Solver to Solver2 to Solver3

    if args2.iters > 0:
        solver.train(n_iterations=args2.iters)


    filename = f'Disentanglement_Metrics_s{i}_c{j}.pkl' # e.g., 'solver_history_s1_c2.pkl'
    full_save_path = os.path.join(history_save_base_dir, filename)

    # 4. Save the solver.history
    try:
        with open(full_save_path, 'wb') as f:
            pickle.dump(solver.history, f)
        print(f"Disentanglement_Metrics saved to: {full_save_path}")
    except Exception as e:
        print(f"Error saving History_InformationLoss for s{sp_lvl}_c{con_lvl}: {e}")
    # --- YOUR ORIGINAL SAVE BLOCK ENDS HERE ---

print("\nAll experiments complete. Check your hHistory_InformationLoss for the saved .pkl files.")


**----------------------------------------------------**

**--------- 17- Plotting from Google Drive *.pkls-----**

**----------------------------------------------------**

In [ ]:
import matplotlib.pyplot as plt
import pickle # <--- Stick with pickle for loading
import os
import numpy as np
import torch # <--- STILL NEEDED for isinstance(item, torch.Tensor) and .cpu().item()

# Assuming your Google Drive is already mounted at /content/drive

# --- Configuration ---
# Directory where your .pkl files are saved
HISTORY_DIR = '/content/drive/MyDrive/AdaIN/mrh_AdaIN_Output_80bin/MRHSavedData'

# List of filenames for your history files
# HISTORY_FILENAMES = [
#     'History__InformationLoss_s1_c2.pkl',
#     'History__InformationLoss_s1_c3.pkl',
#     'History__InformationLoss_s1_c4.pkl',
#     'History__InformationLoss_s1_c5.pkl',
#     'History__InformationLoss_s1_c6.pkl',
#     'History__InformationLoss_s1_c7.pkl',
#     'History__InformationLoss_s1_c8.pkl',
#     'History__InformationLoss_s1_c9.pkl',
#     'History__InformationLoss_s1_c10.pkl',
# ]

# HISTORY_FILENAMES = [
#     'History_InformationLoss_s3_c1.pkl',
#     'History_InformationLoss_s4_c1.pkl',
#     'History_InformationLoss_s5_c1.pkl',
#     'History_InformationLoss_s6_c1.pkl',
#     'History_InformationLoss_s7_c1.pkl',
#     'History_InformationLoss_s8_c1.pkl',
#     'History_InformationLoss_s9_c1.pkl',
#     'History_InformationLoss_s10_c1.pkl',
# ]

# HISTORY_FILENAMES = [
#         'CrossEncoder__InformationLoss_s1_c1.pkl',
#         'CrossEncoder__InformationLoss_s1_c2.pkl',
#         'CrossEncoder__InformationLoss_s1_c3.pkl',
#         'CrossEncoder__InformationLoss_s1_c4.pkl',
#         'CrossEncoder__InformationLoss_s1_c5.pkl',
#         'CrossEncoder__InformationLoss_s1_c6.pkl',
#         'CrossEncoder__InformationLoss_s1_c7.pkl',
#         'CrossEncoder__InformationLoss_s1_c8.pkl',
#         'CrossEncoder__InformationLoss_s1_c9.pkl',
#         'CrossEncoder__InformationLoss_s1_c10.pkl',
# ]
SC_combinations = [
    (2, 7),
    (3, 9),
    (4, 10),
    (10, 7),
    (6, 2),
    (7, 9),
    (8, 4),
    (9, 5),
    (3, 10),
]
HISTORY_FILENAMES = [
                'Disentanglement_Metrics_s1_c1.pkl',
                'Disentanglement_Metrics_s2_c2.pkl',
                'Disentanglement_Metrics_s3_c3.pkl',
                'Disentanglement_Metrics_s4_c4.pkl',
                'Disentanglement_Metrics_s5_c5.pkl',
                'Disentanglement_Metrics_s6_c6.pkl',
                'Disentanglement_Metrics_s7_c7.pkl',
                'Disentanglement_Metrics_s8_c8.pkl',
                'Disentanglement_Metrics_s9_c9.pkl',
                'Disentanglement_Metrics_s10_c10.pkl',
]


# List of metrics you want to compare and plot
# METRICS_TO_COMPARE = [
#     'loss_speaker_id_mi',
#     'loss_transcription_mi',
#     'loss_speaker_attribute_mi',
# ]
METRICS_TO_COMPARE = [

            'speaker_latent_speaker_id_accuracy',
            'speaker_latent_gender_accuracy',
            'speaker_latent_age_accuracy',
            'speaker_latent_accent_accuracy',
            'speaker_latent_content_R2',
            'content_latent_speaker_id_accuracy',
            'content_latent_transcription_R2',
]


# Define a set of colors and linestyles for plotting multiple lines
COLORS = ['blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan', 'lime', 'gold']
LINESTYLES = ['-', '--', ':', '-.', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (3, 5, 1, 5)), (0, (3, 10, 1, 10))]


# --- Helper Functions ---

def load_history_from_pkl(filename):
    """
    Loads a solver history dictionary from a .pkl file using pickle.load().
    This is necessary because the files were saved with pickle.dump().
    """
    full_path = os.path.join(HISTORY_DIR, filename)
    loaded_data = None
    try:
        with open(full_path, 'rb') as f:
            # Revert to pickle.load() as the files were saved with pickle.dump()
            loaded_data = pickle.load(f)
        print(f"Successfully loaded history from: {full_path}")
    except FileNotFoundError:
        print(f"Error: File not found at '{full_path}'")
    except Exception as e:
        print(f"An error occurred while loading '{full_path}': {e}")
    return loaded_data

def plot_multi_history_comparison(
    histories_data,
    metric_key,
    min_iterations=None
):
    """
    Plots a single metric from multiple history dictionaries for comparison.
    This function still explicitly converts PyTorch tensors to Python numbers
    because pickle.load() might still leave them as tensor objects.
    """
    if not histories_data:
        print("No history data provided for plotting.")
        return

    plottable_histories = []
    min_overall_max_iter = float('inf')

    for name, history in histories_data:
        if not history:
            print(f"Skipping '{name}': History data is missing.")
            continue
        if metric_key not in history:
            print(f"Skipping '{name}': Metric '{metric_key}' not found in its history.")
            continue
        if 'iteration' not in history or not history['iteration']:
            print(f"Skipping '{name}': Iteration data not found or empty.")
            continue

        iters = history['iteration']
        raw_data = history[metric_key]

        # CRITICAL: Convert any PyTorch tensors in the data list to Python numbers
        # This is essential because pickle.load() might return tensor objects
        # that matplotlib cannot plot directly.
        data = []
        for item in raw_data:
            if isinstance(item, torch.Tensor):
                # Ensure it's on CPU and then convert to a standard Python number
                # The .cpu() might not be strictly necessary if pickle already moved it,
                # but it's harmless and robust.
                data.append(item.cpu().item())
            else:
                # If it's already a Python number (int, float), just append it
                data.append(item)

        # Filter out NaN/Inf values
        valid_indices = [i for i, x in enumerate(data) if not (isinstance(x, (float)) and (np.isnan(x) or np.isinf(x)))]
        iters_filtered = [iters[i] for i in valid_indices]
        data_filtered = [data[i] for i in valid_indices]

        if not iters_filtered:
            print(f"Skipping '{name}': No valid data points after filtering NaNs/Infs for metric '{metric_key}'.")
            continue

        plottable_histories.append({
            'name': name,
            'iters': iters_filtered,
            'data': data_filtered,
            'max_iter': iters_filtered[-1]
        })
        min_overall_max_iter = min(min_overall_max_iter, iters_filtered[-1])


    if not plottable_histories:
        print(f"No valid histories to plot for metric '{metric_key}'.")
        return

    actual_max_plot_iter = min_iterations
    if min_iterations is None:
        actual_max_plot_iter = min_overall_max_iter
        print(f"Plotting {metric_key} up to minimum common iteration: {actual_max_plot_iter}")
    else:
        print(f"Plotting {metric_key} up to specified iteration: {actual_max_plot_iter}")
        if actual_max_plot_iter > min_overall_max_iter:
             print(f"Warning: Specified min_iterations ({actual_max_plot_iter}) exceeds "
                   f"the shortest model's max iterations ({min_overall_max_iter}). "
                   f"Plotting will be limited to {min_overall_max_iter}.")
             actual_max_plot_iter = min_overall_max_iter


    plt.figure(figsize=(12, 6))
    for i, model_data in enumerate(plottable_histories):
        name = model_data['name']
        iters = model_data['iters']
        data = model_data['data']

        idx = next((j for j, x in enumerate(iters) if x > actual_max_plot_iter), len(iters))
        plot_iters = iters[:idx]
        plot_data = data[:idx]

        if not plot_data:
            print(f"Not enough valid data for '{name}' to plot {metric_key} for the specified iteration range. Skipping this model.")
            continue

        color = COLORS[i % len(COLORS)]
        linestyle = LINESTYLES[i % len(LINESTYLES)]

        display_label = name.replace('History__InformationLoss_s1_', '')

        plt.plot(plot_iters, plot_data,
                 label=f'{display_label}',
                 color=color,
                 linestyle=linestyle)

    plt.xlabel('Iteration')
    plt.ylabel(metric_key.replace('_', ' ').title())
    plt.title(f'Comparison of {metric_key.replace("_", " ").title()} Across Models (First {actual_max_plot_iter} Iterations)')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- Main Execution Block ---

if __name__ == "__main__":
    loaded_histories_with_names = []

    for filename in HISTORY_FILENAMES:
        history_data = load_history_from_pkl(filename)
        if history_data:
            model_full_name = os.path.splitext(filename)[0]
            loaded_histories_with_names.append((model_full_name, history_data))

    if loaded_histories_with_names:
        for metric in METRICS_TO_COMPARE:
            print(f"\n--- Plotting Metric: {metric} ---")
            plot_multi_history_comparison(
                loaded_histories_with_names,
                metric,
                min_iterations=500
            )
    else:
        print("No history files were successfully loaded. Cannot perform comparison plots.")